# Macaron dish drawings
Draws every dish in the chosen cute style with Z-Image Turbo, on both GPUs at once, cuts out the background and saves small webp pictures plus contact sheets for checking. Everything is generated, so no one else's photos are used.

In [ ]:
!pip -q install -U --no-deps diffusers 2>&1 | tail -1
!pip -q install -U bitsandbytes 2>&1 | tail -1
!pip -q install --no-deps rembg pymatting onnxruntime flatbuffers coloredlogs humanfriendly 2>&1 | tail -2
import torch; print("GPUs:", torch.cuda.device_count())

In [ ]:
# which dishes to draw in this run, and the style
ONLY = [683, 5201, 5202, 5203, 5204, 5205, 5206, 343, 5208, 5209, 260, 427, 430, 431, 432, 433, 434, 435, 441, 825, 926, 1123, 5222, 262, 527, 530, 641, 594, 276, 285, 286, 356, 357, 360, 361, 367, 5236, 371, 372, 365, 5240, 301, 459, 5243, 5244, 5245, 5246, 5247, 169, 463, 462, 5251, 1069, 868, 1055, 7301, 7302, 7303, 7304, 7305, 7306, 7307, 7308, 7309, 7310, 7311, 7312, 7313, 7314, 7315, 7316, 7317, 7318, 7319, 7320, 7321, 7322, 7323, 7324, 7325, 7326, 7327, 7328, 7329, 7330, 7331, 7332, 7333, 7334, 7335, 7336, 7337, 7338, 7339, 7340, 7341]
STYLE_KEY = "Sticker"
SIZE, STEPS = 512, 8


In [ ]:
PROMPTS = [{"id": 0, "name": "Nasi lemak biasa", "desc": "Malaysian nasi lemak on a piece of banana leaf. A dome of white coconut rice, a spoon of dark red sambal, a small heap of crispy fried anchovies, a small heap of roasted peanuts, four cucumber slices and half a hard-boiled egg showing the yellow yolk", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 9}, {"id": 76, "name": "Chicken rice, steamed", "desc": "a plate of Malaysian Hainanese steamed chicken rice. Six neat slices of boneless poached chicken with smooth pale white skin, laid in a row on cucumber slices, drizzled with light soy sauce and sprinkled with spring onion and coriander. A smooth dome of pale yellow rice beside it and a tiny saucer of red chilli sauce", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 82, "name": "Chicken rice, roasted", "desc": "a plate of Malaysian roasted chicken rice. Six neat boneless slices of roasted chicken, each slice with a thin layer of glossy dark golden-brown roasted skin on top and juicy white meat underneath, laid in a row on cucumber slices with a little soy sauce. A smooth dome of pale yellow chicken rice beside it and a tiny saucer of red chilli sauce", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 5, "name": "Roti bawang", "desc": "a plate of Malaysian roti bawang. One round flat golden-brown pan-fried flatbread with crisp darker spots, its top generously covered with lots of chopped purple-red onion pieces pressed into the bread and lightly browned, clearly showing many onion bits. Beside it, a small bowl of orange-yellow dhal curry", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 10}, {"id": 141, "name": "Curry laksa", "desc": "a bowl of Malaysian curry laksa. Orange coconut curry soup with yellow noodles, three tofu puffs, four prawns, half a boiled egg, bean sprouts and fresh mint leaves", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 135, "name": "Char kway teow", "desc": "a plate of Penang char kuey teow. A loose pile of wide flat ribbon rice noodles, each strand a broad flat white band about as wide as a finger like wide fettuccine ribbons, clearly flat and wide rather than thin round strands, stir-fried and stained glossy brown with dark soy sauce, with slightly charred edges. Mixed in are three pink prawns, small cockles, crunchy bean sprouts, short green chive pieces and small yellow bits of scrambled egg (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 51, "name": "Wantan mee, dry", "desc": "a plate of Malaysian dry wantan mee. A heap of thin springy egg noodles tossed in glossy dark soy sauce, topped with five thin slices of red-edged char siu barbecued pork and a few stalks of green choy sum. Beside the plate, a small bowl of clear soup with three silky boiled wontons", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 119, "name": "Mee goreng mamak", "desc": "a plate of Malaysian mee goreng mamak. A heap of yellow noodles stir-fried until light brown and dry, with no sauce or gravy, mixed with cubes of fried tofu, small potato pieces, bean sprouts and green leafy vegetables, with no meat, topped with one fried egg with a runny yellow yolk, and a lime half on the side", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 10}, {"id": 103, "name": "Nasi goreng kampung", "desc": "Malaysian village fried rice with anchovies and water spinach (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 133, "name": "Satay ayam", "desc": "Malaysian chicken satay. Six bamboo skewers of grilled chicken pieces with caramelised charred edges, a small bowl of thick peanut sauce, cubes of cucumber and red onion, and a few rice cake cubes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 13, "name": "Kaya toast set", "desc": "Malaysian kaya toast set. Two thin slices of crisp toasted bread cut into rectangles, sandwiched with a layer of green kaya jam and a thick cold slice of yellow butter. Beside them, a small white bowl holding two runny soft-boiled eggs, cracked open, with dark soy sauce. A cup of black kopi coffee", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 610, "name": "Kopitiam or cafe drink", "desc": "a tall clear glass of frothy light-brown teh tarik milk tea with foam on top, beside a white kopitiam cup of dark coffee on a saucer, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1091, "name": "Fried wantan", "desc": "a small plate of five golden deep-fried wontons. Each wonton is a round puffy pouch with crinkly, ruffled, frilly crispy edges like a flower. A small dish of red sweet chilli sauce beside them", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 935, "name": "Curry puff", "desc": "Malaysian curry puffs: golden half-moon pastries with crimped twisted edges (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 788, "name": "PAUL croissant", "desc": "croissant (1 croissant)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 9}, {"id": 775, "name": "Cendol", "desc": "a bowl of Malaysian cendol. A mound of white shaved ice piled high in the middle of a bowl, drizzled with dark brown palm sugar syrup and white coconut milk running down the sides, with a heap of bright green worm-shaped pandan jelly strands on one side and a heap of glossy dark red kidney beans on the other side", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 857, "name": "Ice cream", "desc": "a single scoop of vanilla ice cream in a small paper cup. One round scoop of creamy white ice cream with a rough scooped texture, clear curved ridges and ripples left by the ice cream scoop, a little curled lip of ice cream around its base and a soft melting drip, sitting in a short pastel paper cup with a tiny wooden spoon stuck in it", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 14}, {"id": 148, "name": "Prawn mee", "desc": "Penang prawn mee: red-orange prawn broth with yellow noodles and vermicelli, prawns, pork slices, half a boiled egg, water spinach and fried shallots (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 56, "name": "Pan mee, soup", "desc": "a bowl of Malaysian pan mee soup. Hand-torn flat irregular pieces of pale noodle in clear light brown broth, topped with a generous heap of dark brown fried minced pork, sliced brown shiitake mushrooms, a pile of small crispy golden-brown fried anchovies (ikan bilis) clearly visible on top, and a bunch of very dark deep green leafy vegetables", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 95, "name": "Economy rice: 1 meat, 2 veg", "desc": "Malaysian economy rice: white rice with a piece of meat and two heaps of stir-fried vegetables, with : 1 meat, 2 veg (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 153, "name": "Bak kut teh with rice", "desc": "a claypot of Malaysian herbal bak kut teh soup with a small bowl of rice beside it. A round dark brown clay pot filled with dark brown herbal soup, with four chunky pork ribs on the bone, a whole bulb of garlic, brown shiitake mushrooms and fried tofu puffs, with Chinese herbs floating in the soup: bright red goji berries, a brown star anise, thin pale slices of dried angelica root and a stick of cinnamon bark. Beside it, a small bowl of plain white rice and a small dish of chopped red chilli in soy sauce", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 139, "name": "Hokkien mee (KL)", "desc": "a plate of KL Hokkien mee. A heap of thick round noodles braised in thick glossy very dark brown soy sauce gravy, the colour of dark chocolate brown, not black and not inky. Everything is the same dark brown colour together: the noodles, the pieces of cabbage, the prawns, the pork slices and the small crispy cubes of fried pork lard on top are all coated and stained dark brown by the sauce", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 152, "name": "Yong tau foo, soup", "desc": "a bowl of Malaysian yong tau foo soup. Clear pale broth with white round fish balls, golden fried tofu puffs stuffed with white fish paste, a chunk of purple eggplant stuffed with fish paste, green okra lady fingers stuffed with fish paste, and a long red chilli stuffed with fish paste, sprinkled with chopped spring onion. Beside it, a small dish of red chilli sauce", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 20, "name": "Chicken porridge", "desc": "a bowl of Chinese chicken porridge. Smooth thick white rice congee, topped with neat slices of pale poached chicken laid in a fan like Hainanese chicken rice, sprinkled with chopped green spring onion, golden fried shallots and thin ginger strips, with a drizzle of soy sauce", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 833, "name": "Pho", "desc": "a bowl of Vietnamese beef pho. Clear light brown broth with flat white rice noodles, covered with many very thin wide slices of cooked dark brown beef, a big handful of white bean sprouts, fresh Thai basil leaves with purple stems, long green sawtooth coriander leaves, sliced white onion and a lime wedge on the rim", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 597, "name": "Latte, iced", "desc": "a tall glass of iced latte. A clear tall glass filled with ice cubes and cold white milk, with dark brown espresso swirling down from the top into the milk making soft brown and white layers, and a paper straw, a plain glass with no decorations", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 35}, {"id": 609, "name": "Kopi C", "desc": "a kopitiam cup of strong Malaysian coffee (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 9}, {"id": 617, "name": "Ticco milk tea", "desc": "a plastic cup of milk tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 14}, {"id": 947, "name": "Milo ais", "desc": "a glass of chocolate malt drink (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 396, "name": "Nasi Lemak McD", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 9, "name": "Roti telur", "desc": "a plate of Malaysian roti telur. One thin flat square folded flatbread, pan-fried golden brown with darker crisp spots, slightly puffed, with thin yellow egg visible at a torn edge, lying flat on a round steel plate. Beside it, a small bowl of orange-yellow dhal curry", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 10, "name": "Thosai with dhal and chutney", "desc": "a plate of South Indian thosai. A thin large golden-brown crispy crepe folded in half on a round steel plate. Beside it, a small bowl of yellow dhal curry with chunks of potato and a few dark green curry leaves floating in it, and a small bowl of white coconut chutney", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 115, "name": "Nasi briyani ayam", "desc": "Malaysian nasi briyani: long-grain yellow and orange spiced rice with a chicken curry, papadam and acar, with ayam (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 522, "name": "Banana leaf rice with chicken", "desc": "a big green banana leaf with a mound of white rice in the middle, a clear golden fried chicken drumstick on the side, three different curry gravies poured in small pools around the rice, orange chicken curry, yellow dhal and dark fish curry, plus three small vegetable sides, cabbage, beans and pickle, crispy papadum on top", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 484, "name": "Char siu rice", "desc": "a plate of Malaysian char siu rice. A mound of white rice with six thick slices of char siu barbecued pork fanned on top, each slice with a dark red caramelised charred edge, a glossy sticky glaze and pink-brown meat with a thin strip of fat, drizzled with dark brown sweet char siu sauce that runs onto the rice, with three cucumber slices on the side", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 991, "name": "Roast duck rice", "desc": "chopped roast duck pieces with dark glossy skin in a little sauce, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 158, "name": "Steamed fish with rice and greens", "desc": "a plate of Chinese steamed fish with rice. A whole steamed white fish with pale silvery grey skin and soft white flesh, lying in a shallow pool of thin light soy sauce, topped with thin strips of ginger, spring onion and red chilli. Beside it, one small bowl of white rice", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 287, "name": "Tom yum soup with rice", "desc": "a bowl of Thai tom yum soup with a separate bowl of rice. A bowl of bright orange-red spicy clear tom yum soup with whole pink prawns, straw mushrooms, tomato wedges, lemongrass stalks, lime leaves and fresh coriander. Beside it, a small separate bowl of plain white steamed rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 297, "name": "Pad thai", "desc": "Thai pad thai: stir-fried rice noodles with prawns, egg, bean sprouts, crushed peanuts, chives and a lime wedge (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 525, "name": "Bankara ramen", "desc": "a bowl of Japanese tonkotsu ramen with a pair of chopsticks lifting a big bundle of long thin wavy yellow ramen noodles up out of the creamy light brown pork broth, so the noodles are the main thing you see, with two round slices of chashu pork, a soft-boiled egg cut in half showing a jammy orange yolk, a small pile of thin black wood ear fungus strips, a sheet of dark nori and chopped green spring onion", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 237, "name": "Sushi plates", "desc": "Japanese Sushi plates (6 pieces nigiri and maki, edamame)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 266, "name": "Chicken katsu curry rice", "desc": "a plate of Japanese chicken katsu curry rice. A mound of white rice on one side, a golden breaded fried chicken cutlet sliced into strips laid across the middle, and thick glossy brown Japanese curry sauce on the other side with big chunks of yellow potato and bright orange carrot in the sauce, with a little red pickled ginger", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 217, "name": "Cheeseburger", "desc": "a cheeseburger. A soft round golden sesame bun, a brown beef patty, a slice of melted yellow cheese, green lettuce, a red tomato slice and pickles, stacked neatly", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 317, "name": "Pizza", "desc": "a whole round pizza cut into eight slices. A thick puffy golden-brown baked crust with darker toasted spots, red tomato sauce, a generous layer of melted bubbly mozzarella cheese with browned patches, fully covered with plenty of toppings", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 16}, {"id": 397, "name": "French fries", "desc": "French fries", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 410, "name": "KFC Original Recipe, thigh", "desc": "a single piece of crispy fried chicken thigh on a small plate. A chunky irregular piece of chicken with a thick craggy, crunchy, deep golden-brown breaded coating full of crispy ridges and flakes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 822, "name": "Egg tart", "desc": "Hong Kong egg tarts with smooth yellow custard in flaky pastry (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 10}, {"id": 572, "name": "Aseana nasi lemak", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg (1 plate with chicken)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 23}, {"id": 351, "name": "Mala mini hotpot", "desc": "Chinese hotpot (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 15}, {"id": 814, "name": "Christine's bun", "desc": "bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 13}, {"id": 219, "name": "Acme burger", "desc": "burger (With fries)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 11}, {"id": 779, "name": "MET fruit tea", "desc": "fruit tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 11}, {"id": 803, "name": "White coffee", "desc": "a cup of Ipoh white coffee with light brown froth (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 10}, {"id": 169, "name": "Fish and chips", "desc": "fish and chips: two long fish fillets in puffy golden crisp batter on a plate, a pile of thick-cut golden fries, a yellow lemon wedge, one small bowl of white tartar sauce and a small heap of coleslaw, everything inside the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 624, "name": "MET matcha latte", "desc": "matcha latte", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 9}, {"id": 17, "name": "Dolly Dim Sum har gow", "desc": "three translucent har gow shrimp dumplings in a bamboo steamer; dim sum in bamboo steamers: har gow, siew mai and a steamed bun (4 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 417, "name": "Popcorn chicken", "desc": "Popcorn chicken", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 692, "name": "Cadbury Dairy Milk", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 802, "name": "Mango Magic smoothie", "desc": "Mango Magic smoothie", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 837, "name": "TWG macarons", "desc": "macarons (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 8}, {"id": 50, "name": "Ali Muthu roti bakar set", "desc": "toasted bread with butter and kaya (Toast, 2 eggs and kopi), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 79, "name": "Yee3 chicken rice", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 276, "name": "Bibimbap", "desc": "bibimbap: a black stone bowl of white rice topped with separate neat wedges of colour arranged in a circle, orange carrot strips, green spinach, pale bean sprouts, brown mushrooms, yellow egg strips and brown beef, a fried egg with a bright yolk in the centre and a spoonful of red chilli paste, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 325, "name": "ABC One fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 353, "name": "Da On galbi set", "desc": "Korean galbi set", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 811, "name": "Har gow", "desc": "three translucent har gow shrimp dumplings in a bamboo steamer (3 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 7}, {"id": 167, "name": "Dave's chicken chop", "desc": "Malaysian chicken chop: a breaded fried chicken cutlet with brown or black pepper sauce, fries and coleslaw (With fries and coleslaw)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 184, "name": "Marco carbonara", "desc": "carbonara (1 plate), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 343, "name": "Pumpkin soup", "desc": "pumpkin soup: one bowl of thick smooth bright orange pumpkin soup on a saucer, a white swirl of cream on the surface, a few green pumpkin seeds and a pinch of green parsley in the centre, one slice of toasted bread resting on the saucer, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 358, "name": "Palsaik kimchi stew", "desc": "Korean kimchi stew (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 361, "name": "Tteokbokki", "desc": "tteokbokki: a shallow bowl of thick white cylinder rice cakes coated in thick glossy bright red-orange sauce, with triangular pale fish cake slices, half a boiled egg and green spring onion, white sesame seeds on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 602, "name": "Mocha, iced", "desc": "Mocha, iced", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 611, "name": "Wong Kok milk tea, iced", "desc": "a plastic cup of milk tea, with iced (1 glass), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 6}, {"id": 780, "name": "Fong Lye beef noodles", "desc": "braised beef noodle soup: dark brown broth, thick noodles, braised beef chunks and bok choy (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 853, "name": "Doughnut", "desc": "Doughnut (1 glazed)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 854, "name": "Butter cake", "desc": "Butter cake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 6}, {"id": 84, "name": "Fong Lye pork chop rice", "desc": "pork chop rice: a mound of white rice with one big golden-brown fried breaded pork chop sliced into strips on top, with pickled mustard greens and half a braised egg, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 5}, {"id": 91, "name": "Yuan Room roast duck", "desc": "chopped roast duck pieces with dark glossy skin in a little sauce (Quarter duck), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 98, "name": "Bumbu Desa nasi campur", "desc": "Malaysian mixed rice: white rice with several small heaps of dishes such as curry chicken, vegetables and fried fish (1 plate, 3 dishes), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 144, "name": "KAKI 5 curry mee", "desc": "Malaysian curry mee: orange coconut curry soup with yellow noodles, tofu puffs, long beans and cockles (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 166, "name": "Dandy grilled chicken plate", "desc": "grilled chicken plate (With rice and salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 211, "name": "Nando's chicken burger", "desc": "chicken burger (Breast fillet, peri-peri), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 232, "name": "DayOne salmon don", "desc": "Japanese salmon don (1 bowl), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 238, "name": "Salmon nigiri", "desc": "Japanese Salmon nigiri (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 302, "name": "Pineapple fried rice", "desc": "pineapple fried rice: half a pineapple cut lengthways and hollowed out, golden-yellow diamond-patterned skin with a crown of fresh bright green spiky leaves at the top, the leaves clearly vivid green, filled with yellow turmeric fried rice mixed with big yellow pineapple chunks, pink prawns, cashew nuts, raisins and green peas, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 318, "name": "Pizza, BBQ chicken", "desc": "Pizza, BBQ chicken", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 333, "name": "Nyonya chap chye with rice", "desc": "Nyonya chap chye with rice: a plate with a mound of white rice and a serving of braised mixed vegetables in brown sauce: cabbage, glass noodles, black fungus, beancurd skin sticks, shiitake mushrooms and carrot, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 5}, {"id": 427, "name": "Tonkatsu set", "desc": "tonkatsu set: a tray with one breaded pork cutlet fried golden-brown and cut into six strips showing pale meat, on a plate with a tall mound of finely shredded pale green cabbage and a lemon wedge, a bowl of white rice, a small bowl of miso soup and a small dish of dark brown sauce, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 573, "name": "Ayam gepuk with rice", "desc": "Malaysian with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 605, "name": "Lindt hot chocolate", "desc": "hot chocolate (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 706, "name": "Oreo Mini", "desc": "chocolate sandwich cookies with white cream", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 798, "name": "Chocolate cake", "desc": "Chocolate cake (1 slice), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 804, "name": "Dave's ribeye steak", "desc": "ribeye steak", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 5}, {"id": 841, "name": "Egg and cheese bagel", "desc": "Egg and cheese bagel (1 bagel)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 5}, {"id": 868, "name": "Croffle", "desc": "croffle: one croissant pressed in a waffle iron on a small plate, golden-brown with a deep square waffle grid pattern and flaky croissant layers showing at the curved edges, one scoop of vanilla ice cream on top and a thin drizzle of golden syrup, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 63, "name": "Mee hoon soup with fish balls", "desc": "fish ball mee hoon soup: a bowl of thin white rice vermicelli in clear light broth with five round white fish balls, slices of fish cake, green lettuce and chopped spring onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 4}, {"id": 65, "name": "Imbi pork noodles, soup", "desc": "pork noodle soup: a bowl of noodles in cloudy pork broth topped with minced pork, slices of pork, pork liver, a pork ball, a poached egg, green lettuce and fried lard cubes, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 4}, {"id": 107, "name": "ParaThai pad kra pao chicken", "desc": "Thai minced meat stir-fried with holy basil over rice with a fried egg, with chicken (With rice and egg)", "neg": "whole roast chicken", "n": 4}, {"id": 122, "name": "Food court char kuey teow", "desc": "a plate of Penang char kuey teow. A loose pile of wide flat ribbon rice noodles, each strand a broad flat white band about as wide as a finger like wide fettuccine ribbons, clearly flat and wide rather than thin round strands, stir-fried and stained glossy brown with dark soy sauce, with slightly charred edges. Mixed in are three pink prawns, small cockles, crunchy bean sprouts, short green chive pieces and small yellow bits of scrambled egg, with food court (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 162, "name": "Johnny's chicken chop", "desc": "Malaysian chicken chop: a breaded fried chicken cutlet with brown or black pepper sauce, fries and coleslaw (With fries), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 164, "name": "ABC One chicken chop rice", "desc": "Malaysian chicken chop: a breaded fried chicken cutlet with brown or black pepper sauce, fries and coleslaw, with rice (1 plate), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 165, "name": "Baked pork chop rice", "desc": "Hong Kong baked pork chop rice: rice covered with fried pork chop and tomato sauce, baked with cheese in an oval dish (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 195, "name": "Quinoa chicken bowl", "desc": "Quinoa chicken bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 196, "name": "LAC protein bowl", "desc": "protein bowl (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 220, "name": "Dandy beef burger", "desc": "beef burger (With fries)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 240, "name": "Kinori salmon nori taco", "desc": "Japanese salmon nori taco (1 taco)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 262, "name": "Ramen, shoyu", "desc": "ramen, shoyu: a deep bowl of clear dark brown soy broth with thin wavy yellow noodles, two round slices of pork chashu, half a soft-boiled egg with an orange yolk, pale bamboo shoots, a sheet of dark seaweed standing at the edge and green spring onion, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 267, "name": "Sukiya curry rice", "desc": "Japanese curry rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 279, "name": "bhc Bburinkle chicken", "desc": "Korean Bburinkle chicken (Half chicken)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 280, "name": "Kyochon Soy Garlic wings", "desc": "Korean Soy Garlic wings (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 307, "name": "Chicken shawarma wrap", "desc": "Middle Eastern Chicken shawarma wrap (1 wrap), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 332, "name": "Kangkung belacan with tofu and rice", "desc": "a plate with a mound of white rice, a big heap of stir-fried dark green kangkung water spinach thickly coated in lots of dark red-brown belacan chilli sambal with sliced red chillies and dried shrimp, and golden fried tofu cubes, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 349, "name": "Cucur udang", "desc": "cucur udang prawn fritters: five round golden-brown crispy deep-fried batter fritters with a bumpy surface, green chives in the batter and one small whole prawn pressed on top of each, with a small dish of red chilli sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 364, "name": "K Fry kimchi fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with kimchi (1 plate), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 411, "name": "A&W chicken, 2 pieces", "desc": "chicken, 2 pieces (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 424, "name": "Hot dog", "desc": "Hot dog (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 506, "name": "Ayam bakar with rice", "desc": "ayam bakar with rice: a grilled chicken leg quarter with dark brown charred glaze beside a mound of white rice, with red sambal, cucumber slices and raw green herbs, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 4}, {"id": 604, "name": "Tim Hortons Iced Capp", "desc": "Iced Capp", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 686, "name": "KitKat Chunky", "desc": "chocolate-covered wafer fingers", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 714, "name": "Chocolate chip cookie", "desc": "chocolate chip cookies", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 743, "name": "Ritz crackers", "desc": "round crackers", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 796, "name": "Fruit tart", "desc": "Fruit tart", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 830, "name": "Matcha tiramisu", "desc": "Japanese Matcha tiramisu (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 851, "name": "adough donut", "desc": "donut (1 donut)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 871, "name": "Pretzel, original", "desc": "Pretzel, original (1 pretzel)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 878, "name": "Eureka popcorn", "desc": "Eureka popcorn", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 4}, {"id": 897, "name": "Black bean soup with chicken", "desc": "black bean soup with chicken: a bowl of very dark brown-black broth with chicken pieces on the bone, whole black beans and red dates, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 4}, {"id": 7, "name": "Mantra butter chicken with naan", "desc": "creamy orange butter chicken curry with a piece of naan; a soft blistered naan flatbread (1 serving), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 22, "name": "Hon Kee fish porridge", "desc": "fish porridge: a bowl of smooth white rice congee topped with several thin slices of white fish fillet with a grey skin edge, ginger strips, chopped spring onion and a drizzle of sesame oil, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 27, "name": "Oats bowl", "desc": "an oats bowl: a bowl of creamy beige oat porridge topped with banana slices, blueberries, a spoon of peanut butter, chia seeds and a drizzle of honey, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 36, "name": "Pokok big breakfast", "desc": "big breakfast (Eggs, sausage, toast and salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 53, "name": "On Lee wonton noodles", "desc": "dry wonton noodles: thin yellow egg noodles tossed in dark soy sauce on a plate, topped with slices of char siu barbecued pork with a red glazed edge and pale pink-brown meat, green choy sum, and a small bowl of clear soup with three wontons, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 55, "name": "PappaRich wantan mee", "desc": "Malaysian wantan mee: thin springy egg noodles tossed in dark soy sauce with slices of red-edged char siu, choy sum and small boiled wontons (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 3}, {"id": 60, "name": "KAKI 5 pan mee", "desc": "pan mee soup: a bowl of flat hand-torn noodles in clear broth, generously topped with browned minced pork, a big handful of very dark forest-green sayur manis leaves, sliced black wood ear fungus, sliced shiitake mushrooms, crispy fried anchovies and a poached egg, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 64, "name": "Hot and sour soup", "desc": "hot and sour soup: a bowl of thick dark reddish-brown soup with ribbons of egg, strips of tofu, black wood ear fungus, bamboo shoot strips and chopped spring onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 3}, {"id": 89, "name": "Village roast duck rice", "desc": "Chinese rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 99, "name": "Food hall mixed rice", "desc": "Malaysian mixed rice: white rice with a piece of meat and two heaps of stir-fried vegetables, with food hall (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 145, "name": "Grandmama's asam laksa", "desc": "Penang asam laksa: thick white rice noodles in brown sour fish broth topped with shredded cucumber, onion, pineapple, mint and a spoon of black prawn paste (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 146, "name": "Madam Kwan's assam laksa", "desc": "Penang asam laksa: thick white rice noodles in brown sour fish broth topped with shredded cucumber, onion, pineapple, mint and a spoon of black prawn paste (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 154, "name": "Klang bak kut teh", "desc": "Malaysian bak kut teh: a clay pot of dark herbal pork rib soup with pork ribs, tofu puffs, mushrooms and lettuce, with klang, bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 156, "name": "Claypot chicken rice", "desc": "claypot chicken rice: a black clay pot of dark soy sauce rice covered with many big chunks of braised chicken, slices of red Chinese sausage and salted fish, dark sweet soy sauce drizzled over the chicken, chopped spring onion on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 160, "name": "Crispy popiah", "desc": "small crispy fried popiah spring rolls (4 rolls), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 168, "name": "Nando's macho peas", "desc": "macho peas", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 173, "name": "Sekinchan grilled fish rice", "desc": "grilled fish rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 189, "name": "Come True cafe brunch pasta", "desc": "cafe brunch pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 193, "name": "Spaghetti carbonara", "desc": "Spaghetti carbonara (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 231, "name": "Rakuzen chirashi don", "desc": "Japanese chirashi don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 247, "name": "Chicken teriyaki don", "desc": "Japanese Chicken teriyaki don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 249, "name": "Teppanyaki salmon set", "desc": "Japanese salmon set (With rice and vegetables)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 252, "name": "DayOne teriyaki chicken rice", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce, with teriyaki (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 263, "name": "Ippudo chicken ramen", "desc": "Japanese chicken ramen (1 bowl, pork-free)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 270, "name": "Itsu chicken katsu don", "desc": "Japanese chicken katsu don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 303, "name": "Johnny's Thai fried rice", "desc": "Thai fried rice: a mound of light fried rice with egg, chicken pieces and green spring onion, served Thai style with a fried egg on top, a lime wedge, cucumber slices, sprigs of coriander and a small dish of fish sauce with sliced red and green chillies, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 3}, {"id": 327, "name": "Wok Hey egg fried rice with chicken", "desc": "golden egg fried rice with spring onion and diced vegetables, with egg with chicken (1 box), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 337, "name": "Putien stir-fried yam", "desc": "stir-fried yam: golden-brown cubes of deep-fried yam with a crisp sugary coating, light purple inside where one is broken, piled on a plate with chopped spring onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 3}, {"id": 342, "name": "Watercress soup with lean pork", "desc": "watercress soup: a bowl of clear light brown broth full of dark green watercress stems and leaves, with slices of lean pork, red dates and orange goji berries, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 3}, {"id": 370, "name": "Kyochon Red Original wings", "desc": "Korean Red Original wings (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 382, "name": "Al Halabi mixed grill", "desc": "Middle Eastern mixed grill (With rice and salad), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 443, "name": "Thai Thai green curry chicken", "desc": "Malaysian chicken curry with potatoes in orange gravy, with thai thai green (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 477, "name": "Chili's chicken fajitas", "desc": "chicken fajitas (1 skillet with tortillas)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 478, "name": "Pho Vietz fresh spring rolls", "desc": "golden crispy fried spring rolls, with fresh s (2 rolls)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 507, "name": "Ayam pongteh with rice", "desc": "ayam pongteh with rice: a bowl of chicken pieces and potato chunks with shiitake mushrooms in dark brown fermented bean gravy, beside a mound of white rice, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 3}, {"id": 509, "name": "Chicken biryani", "desc": "chicken biryani: a deep blue ceramic bowl heaped with long-grain basmati rice in mixed white, yellow and orange grains, a whole chicken leg coated in brown masala half buried in the rice, dark crispy fried onions scattered on top, fresh green mint leaves and one long green chilli, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 518, "name": "Roti bom", "desc": "a small thick round flaky roti bom with sugar and butter (1 piece), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 520, "name": "Rava thosai", "desc": "a thin crispy golden dosa folded on a plate with coconut chutney and sambar, with rava (1 piece with chutney), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 526, "name": "Bankara tsukemen", "desc": "Japanese Bankara tsukemen (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 576, "name": "Warung Prima nasi kunyit with chicken", "desc": "nasi kunyit with chicken: a mound of bright yellow turmeric glutinous rice beside chicken curry with potatoes in orange-brown gravy, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 3}, {"id": 592, "name": "Salmon sashimi", "desc": "Japanese Salmon sashimi (5 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 613, "name": "Taimal herbal tea", "desc": "Chinese herbal tea (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 620, "name": "HEYTEA cheese grape", "desc": "Chinese cheese grape", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 621, "name": "Leekor Leekor drink", "desc": "drink", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 691, "name": "KitKat White", "desc": "chocolate-covered wafer fingers", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 694, "name": "Tim Tam Original", "desc": "Tim Tam Original", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 695, "name": "Chocolate bar", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 696, "name": "Twix", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 725, "name": "Cheezels", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 792, "name": "Cinnabon MiniBon", "desc": "MiniBon (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 793, "name": "Dome chicken pie", "desc": "chicken pie (1 pie with salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 801, "name": "Boost Juice Berry Crush", "desc": "Juice Berry Crush (Original size)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 3}, {"id": 818, "name": "Pancakes with syrup", "desc": "Pancakes with syrup (3 pancakes), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 847, "name": "Godiva chocolixir", "desc": "chocolixir", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 862, "name": "Banana pudding", "desc": "Banana pudding (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 870, "name": "Soft drink", "desc": "one bright red aluminium soda can, beside a glass of dark brown fizzy cola with ice cubes and bubbles, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 3}, {"id": 872, "name": "Cheesecake", "desc": "Cheesecake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 3}, {"id": 891, "name": "Chicken soup noodles", "desc": "chicken noodle soup: a bowl of yellow egg noodles in clear golden chicken broth topped with shredded chicken, green choy sum, fried shallots and spring onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 3}, {"id": 895, "name": "ABC soup with pork ribs", "desc": "ABC soup with pork ribs: a bowl of clear light broth with pork ribs, chunks of potato, orange carrot, red tomato wedges and onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 3}, {"id": 1, "name": "Bungkus Kaw Kaw nasi lemak sotong", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg, with sotong (1 packet), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 2, "name": "Lele penyet with rice", "desc": "lele penyet: a whole small deep-fried catfish, dark golden-brown and crispy, flattened on a plate beside a mound of white rice, with a spoon of red sambal, raw cabbage, cucumber slices, a piece of fried tofu and a piece of fried tempeh, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 2}, {"id": 21, "name": "Century egg and pork porridge", "desc": "century egg and pork congee: a bowl of thick smooth white rice porridge with no noodles, topped with wedges of century egg with dark translucent brown-black white and grey-green yolk, shreds of lean pork, ginger strips, chopped spring onion and a piece of fried you tiao, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 26, "name": "Egg McMuffin", "desc": "Egg McMuffin (1 muffin)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 28, "name": "Yogurt bowl", "desc": "a yogurt bowl: a white bowl of thick white Greek yogurt topped with rows of sliced strawberries, blueberries, banana slices, golden granola clusters and a drizzle of honey, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 31, "name": "Egg sandwich", "desc": "Egg sandwich (2 slices wholemeal, 2 eggs)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 32, "name": "LOVNS banh mi", "desc": "banh mi: a crusty golden Vietnamese baguette split open and filled with slices of pork, pink ham, pickled orange carrot and white daikon shreds, cucumber strips, green coriander and red chilli slices, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 2}, {"id": 35, "name": "PAUL croque monsieur", "desc": "croque monsieur (1 sandwich with salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 38, "name": "6-inch Subway Melt", "desc": "6-inch Subway Melt (Italian bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 43, "name": "O'Briens chicken sandwich", "desc": "chicken sandwich (1 sandwich)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 45, "name": "Egg mayo sandwich", "desc": "Egg mayo sandwich (1 pack)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 57, "name": "AK Noodles pan mee, dry", "desc": "dry pan mee: a bowl of flat hand-torn noodles coated in dark soy sauce, no soup at all, topped with browned minced pork, crispy fried anchovies, sliced mushrooms, green vegetables and a poached egg, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 58, "name": "AK Noodles chili pan mee", "desc": "dry pan mee noodles with minced pork, crispy fried anchovies, a poached egg and dark chilli flakes on the side, with noodles (1 bowl, with poached egg), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 59, "name": "Dragon-i dan dan noodles", "desc": "Sichuan dan dan noodles with minced pork in red chilli oil and sesame sauce, with noodles (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 62, "name": "Lor mee", "desc": "lor mee: thick yellow noodles in very thick dark brown glossy starchy gravy with wisps of egg, topped with slices of braised pork, half a braised brown egg, a piece of fried fish and a dab of minced garlic, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 2}, {"id": 78, "name": "MyIPOH bean sprout chicken", "desc": "Ipoh bean sprout chicken: a plate of sliced poached chicken with pale yellow skin in light soy sauce, beside a separate plate of short fat white bean sprouts with spring onion and soy sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 2}, {"id": 80, "name": "Yun Nans steam pot chicken soup", "desc": "Yunnan steam pot chicken soup: a round brown clay steam pot filled with clear golden chicken broth, chicken pieces on the bone, red goji berries and red dates floating in the soup, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 83, "name": "Canton-i char siew rice", "desc": "rice with slices of red-edged glossy char siu barbecued pork and cucumber (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 123, "name": "Jjajangmyeon", "desc": "Korean jajangmyeon: thick noodles covered in glossy black bean sauce with diced pork and onion, with cucumber strips (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 142, "name": "Simple Life vegetarian laksa", "desc": "Malaysian laksa noodle soup, with vegetarian (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 150, "name": "Pu Tien lor mee", "desc": "lor mee: yellow noodles in thick dark starchy gravy with braised pork, egg and fried fish (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 175, "name": "Teochew steamed pomfret, per person", "desc": "Teochew steamed fish: thick slices of steamed white fish with silver-grey skin in a shallow dish of clear light broth, with red tomato wedges, white tofu slices, green salted mustard vegetable, ginger strips and one sour plum, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 214, "name": "Texas Chicken sandwich", "desc": "Chicken sandwich (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 222, "name": "Double Whopper with cheese", "desc": "Double Whopper with cheese (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 235, "name": "Rakuzen bento set", "desc": "Japanese bento set (Sashimi, tempura and rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 242, "name": "NakNak bulgogi kimbap", "desc": "Korean bulgogi kimbap (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 248, "name": "Sushi Zanmai unagi don", "desc": "Japanese Sushi Zanmai unagi don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 250, "name": "Teppanyaki chicken set", "desc": "Japanese chicken set (With rice, bean sprouts and egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 253, "name": "Cloud 9 rice bowl", "desc": "Chinese rice bowl (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 257, "name": "Eatomo chicken teriyaki bento", "desc": "Japanese chicken teriyaki bento (1 bento)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 261, "name": "Takeno udon set", "desc": "Japanese udon set (With tempura)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 278, "name": "K Fry soy garlic chicken", "desc": "Korean soy garlic chicken (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 282, "name": "Taiwanese XXL fried chicken", "desc": "Taiwanese XXL fried chicken: one huge flat golden-brown crispy fried chicken cutlet, bigger than a hand, dusted with red chilli powder, sticking halfway out of a brown paper bag, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 288, "name": "Thai Thai tom yum goong", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with thai thai goong (1 bowl with rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 293, "name": "Andaman tom yum seafood soup", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with seafood soup (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 305, "name": "Boat noodles", "desc": "Thai boat noodles: a small bowl of thin rice noodles in very dark brown rich broth, with slices of beef, brown meatballs, green morning glory, bean sprouts and crispy pork rind, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 2}, {"id": 321, "name": "Gordon Ramsay Street Pizza pepperoni slice", "desc": "Pizza pepperoni slice (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 324, "name": "Steamed chicken, rice and broccoli", "desc": "Chinese Steamed chicken, rice and broccoli (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 2}, {"id": 334, "name": "Kita vegetarian rice bowl", "desc": "vegetarian rice bowl (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 335, "name": "Cili Kampung sayur campur", "desc": "sayur campur: a plate of stir-fried mixed vegetables, green broccoli, white cauliflower, orange carrot slices, baby corn, cabbage and snow peas in a light clear sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 2}, {"id": 344, "name": "Green papaya fish soup", "desc": "green papaya fish soup: a bowl of milky white broth with chunks of pale green papaya, thick white fish slices with grey skin, and ginger slices, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 2}, {"id": 345, "name": "Seaweed and tofu soup", "desc": "Japanese Seaweed and tofu soup, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 354, "name": "Palsaik 8-flavour pork belly set, per person", "desc": "Korean 8-flavour pork belly set, per person (Pork belly, sides and rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 368, "name": "bhc Hot Fried chicken", "desc": "Korean Hot Fried chicken (Half chicken)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 375, "name": "BBQ chicken wings", "desc": "BBQ chicken wings (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 376, "name": "Black Tap buffalo wings", "desc": "buffalo wings (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 384, "name": "India Gate tandoori chicken", "desc": "red tandoori chicken pieces with onion and lemon, with chicken (Half chicken), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 388, "name": "Big Mac", "desc": "Big Mac (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 398, "name": "Shake Shack cheese fries", "desc": "cheese fries", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 2}, {"id": 399, "name": "Happy Potato tornado potato", "desc": "Korean tornado potato (1 stick)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 413, "name": "KFC Original Recipe, wing", "desc": "Original Recipe, wing (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 426, "name": "Ramly burger, biasa", "desc": "Malaysian Ramly burger, biasa (1 burger), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 442, "name": "Green curry chicken with rice", "desc": "Thai green curry with rice: a bowl of pale green creamy coconut curry with chicken pieces, round green Thai eggplants, basil leaves and red chilli slices, beside a mound of white rice, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 2}, {"id": 445, "name": "India Gate palak paneer", "desc": "palak paneer: a bowl of thick dark green spinach curry with white cubes of paneer cheese and a swirl of cream, with a piece of naan bread, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 2}, {"id": 451, "name": "Tom yum noodles", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with noodles (1 bowl), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 453, "name": "Jjamppong", "desc": "Korean jjamppong: spicy red seafood noodle soup with mussels, squid and vegetables (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 455, "name": "Para Thai tom yum noodles", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with thai noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 459, "name": "Thai milk tea", "desc": "thai milk tea: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with opaque bright vivid orange milk tea over crushed ice, a white layer of evaporated milk on top slowly swirling down into the orange tea in soft streaks, and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 466, "name": "Kim Gary macaroni soup with ham", "desc": "Chinese Kim Gary macaroni soup with ham (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 476, "name": "Grilled chicken wrap", "desc": "Grilled chicken wrap (1 wrap)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 497, "name": "Las Meninas tapas, per person", "desc": "tapas, per person", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 505, "name": "Ahh Yum nasi bunga telang", "desc": "nasi bunga telang: a dome of bright blue butterfly pea flower rice in the centre of the plate, with a piece of crispy fried chicken, half a salted egg, a spoon of red sambal, cucumber slices and green herbs, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 535, "name": "Salted egg", "desc": "salted egg: one salted duck egg cut in half lengthways, both halves facing up on a small dish, the egg white a pale cool grey-white with a slight grey tint, firm and smooth, inside a thin pale blue-green shell, each half with a round deep orange-red oily yolk in the centre, clearly a boiled salted egg, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 584, "name": "Plain naan", "desc": "a soft blistered naan flatbread, with plain (1 piece), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 598, "name": "Caramel macchiato, iced", "desc": "Caramel macchiato, iced", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 608, "name": "Kopi O", "desc": "kopi o: one white ceramic kopitiam coffee cup with a small green flower pattern, sitting on a matching saucer, seen from a three-quarter angle, filled with very dark brown almost black coffee with no milk at all, a thin tan ring of froth around the edge, two soft curls of steam above, a small metal spoon resting on the saucer, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 612, "name": "Mr Bean soya milk", "desc": "Chinese soya milk", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 614, "name": "Teh O", "desc": "teh o: one white ceramic kopitiam coffee cup with a small green flower pattern, sitting on a matching saucer, seen from a three-quarter angle, filled with clear see-through reddish-amber hot black tea with no milk at all, the bottom of the cup faintly visible through the tea, two soft curls of steam above, a small metal spoon on the saucer, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 630, "name": "Pho, chicken", "desc": "Vietnamese beef pho: clear broth, flat rice noodles, thin slices of beef, basil, bean sprouts and lime, with chicken (1 bowl), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 631, "name": "MyIPOH Ipoh chicken hor fun", "desc": "Ipoh chicken hor fun: a bowl of flat white kway teow rice noodles in clear orange-tinted prawn broth with red prawn oil floating on top, slices of poached chicken laid on top, two peeled prawns and green chives, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 632, "name": "Ipoh hor fun", "desc": "Ipoh hor fun: smooth wide flat ribbon rice noodles, each strand a broad flat white band about as wide as a finger like wide fettuccine ribbons, clearly flat and wide rather than thin round strands in clear chicken broth with shredded chicken and prawns (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 633, "name": "Bun bo Hue", "desc": "Vietnamese bun bo hue: spicy red lemongrass beef noodle soup with thick round rice noodles and herbs (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 657, "name": "Omelette, plain", "desc": "Omelette, plain (2 eggs)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 679, "name": "White rice", "desc": "White rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 685, "name": "Garlic bread", "desc": "Garlic bread (2 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 687, "name": "Mars", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 700, "name": "Pocky Strawberry", "desc": "thin biscuit sticks dipped in coating", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 711, "name": "Munchy's Oat Krunch Chocolate", "desc": "oat cookies", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 716, "name": "Dubai chewy cookie", "desc": "Dubai chewy cookie", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 720, "name": "Kinder Joy", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 726, "name": "Mamee Monster", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 733, "name": "Hello Panda Strawberry", "desc": "small biscuits with chocolate dip", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 736, "name": "Twisties Tomato", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 751, "name": "Tora corn snack", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 768, "name": "Tic Tac", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 771, "name": "Mango lassi", "desc": "a glass of thick yellow mango lassi, with mango (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 782, "name": "Fong Lye beef noodle soup", "desc": "braised beef noodle soup: dark brown broth, thick noodles, braised beef chunks and bok choy, with soup (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 783, "name": "Rén la mian with minced pork", "desc": "Chinese hand-pulled noodles, with minced pork (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 784, "name": "Mama Yap Hakka noodles", "desc": "Hakka noodles: a bowl of thin flat yellow noodles tossed in oil with no soup, topped with a big heap of browned minced pork and chopped spring onion, with a small bowl of soup beside it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 2}, {"id": 795, "name": "Almond croissant", "desc": "Almond croissant (1 piece), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 797, "name": "Chateraise strawberry shortcake", "desc": "Japanese strawberry shortcake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 805, "name": "Shook! grilled wagyu", "desc": "grilled wagyu", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 815, "name": "Ippudo pork bun", "desc": "Japanese pork bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 817, "name": "Char siu bao", "desc": "fluffy white steamed char siu bao buns split open at the top showing red barbecued pork (1 bun), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 821, "name": "HWC chicken pie", "desc": "chicken pie (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 824, "name": "Komugi cheese tart", "desc": "Japanese cheese tart (1 tart)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 825, "name": "Gyoza", "desc": "gyoza: six pan-fried dumplings in a row on a rectangular plate, crescent-shaped with pleated edges, pale see-through wrappers and deep golden-brown crisp bottoms facing up, one small dipping bowl of dark soy sauce on the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 828, "name": "Injeolmi bingsu", "desc": "Korean Injeolmi bingsu", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 829, "name": "Marco tiramisu", "desc": "tiramisu (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 831, "name": "Mee tarik beef soup", "desc": "beef noodle soup: a bowl of clear brown beef broth with slices of beef, white radish chunks, thick noodles, green coriander and spring onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 2}, {"id": 832, "name": "Pho Vietz beef pho", "desc": "Vietnamese beef pho: clear broth, flat rice noodles, thin slices of beef, basil, bean sprouts and lime, with beef (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 836, "name": "Soft serve cone", "desc": "Japanese Soft serve cone (1 cone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 842, "name": "Park's Bagels smoked salmon bagel", "desc": "smoked salmon bagel (1 bagel)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 845, "name": "Bagel, plain", "desc": "Bagel, plain (1 bagel)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 849, "name": "Chocolate milkshake", "desc": "Chocolate milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 2}, {"id": 850, "name": "Krispy Kreme chocolate iced glazed", "desc": "chocolate iced glazed (1 doughnut)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 860, "name": "Inside Scoop, double scoop in a waffle cone", "desc": "double scoop in a waffle cone (2 scoops, 1 cone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 865, "name": "Burnt cheesecake", "desc": "Burnt cheesecake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 866, "name": "Herbal Farmer herbal jelly", "desc": "Chinese herbal jelly (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 879, "name": "Scone with jam and cream", "desc": "Scone with jam and cream (1 scone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 880, "name": "Takoyaki", "desc": "Japanese Takoyaki (6 balls)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 885, "name": "Pisang goreng", "desc": "pisang goreng: five banana fritters, whole small bananas coated in a thick golden-brown crunchy craggy fried batter, piled on a sheet of brown paper, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 886, "name": "Ayam masak merah with rice", "desc": "chicken pieces in thick red tomato chilli sauce, with rice, bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 889, "name": "Pan-seared salmon, rice and greens", "desc": "Pan-seared salmon, rice and greens (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 2}, {"id": 900, "name": "Omelette with rice and cucumber", "desc": "a simple plate with exactly one folded golden-yellow omelette, one mound of white rice and four cucumber slices, nothing else on the plate, no drink, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 3, "name": "Teh Tarik Nation nasi lemak", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg; a glass mug of Malaysian teh tarik milk tea, light brown with a thick frothy foam top, with nation (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 4, "name": "Nasi lemak telur rebus, kurang nasi", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg, with telur rebus, kurang nasi (Half rice, boiled egg, cucumber), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 6, "name": "Roti cheese", "desc": "roti cheese: a square folded golden-brown flaky roti canai flatbread cut into four pieces, melted yellow cheese oozing from the edges, with a small bowl of orange dhal curry, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 8, "name": "Teh Tarik Club roti canai with curry", "desc": "Malaysian roti canai: round flaky golden flatbread folded loosely on a plate with a small bowl of yellow dhal curry; a glass mug of Malaysian teh tarik milk tea, light brown with a thick frothy foam top, with club with curry (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 11, "name": "Chapati with dhal", "desc": "two soft round chapati flatbreads with dhal; small bowl of yellow lentil dhal curry (2 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 12, "name": "Idli with sambar", "desc": "five round white fluffy steamed idli rice cakes arranged on a round steel plate, with a steel bowl filled to the top with thick orange-red sambar lentil stew with vegetable pieces and one dried red chilli, and a smaller steel bowl filled with white coconut chutney with a green curry leaf on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 14, "name": "Half-boiled eggs with wholemeal toast", "desc": "two soft half-boiled eggs cracked open into a small saucer, runny translucent egg white and two bright orange liquid yolks with a splash of dark soy sauce, beside two slices of brown wholemeal toast, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 15, "name": "Chee cheong fun, plain", "desc": "plain rolled white rice noodle rolls drizzled with sweet brown sauce and sesame seeds, with plain (1 plate with sweet sauce), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 16, "name": "Dim sum breakfast", "desc": "dim sum in bamboo steamers: har gow, siew mai and a steamed bun, with breakfast (3 har gow, 3 siew mai, 1 bao), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 18, "name": "Dolly Dim Sum chee cheong fun with prawns", "desc": "plain rolled white rice noodle rolls drizzled with sweet brown sauce and sesame seeds; dim sum in bamboo steamers: har gow, siew mai and a steamed bun, with prawns (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 19, "name": "Lor mai kai", "desc": "lor mai kai: a tall dome of medium caramel-brown glutinous sticky rice, warm brown like milk tea and not dark, every sticky grain visible, unmoulded on a small white plate, the top covered with chunks of braised chicken in glossy brown sauce, one dark shiitake mushroom and three slices of red Chinese sausage, nothing else on the plate, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 23, "name": "Souper Tang double-boiled chicken soup", "desc": "double-boiled chicken soup: a small white ceramic lidded soup pot with the lid resting beside it, filled with clear amber broth, chicken pieces, red dates, red goji berries and slices of herbs, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 24, "name": "Teochew porridge with sides", "desc": "smooth Cantonese rice porridge, with teochew with sides (1 bowl and 3 sides), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 25, "name": "Bubur Ayam McD", "desc": "chicken porridge: a white bowl of thick smooth white rice porridge topped with shredded chicken strips, chopped green spring onion, sliced ginger, crispy golden fried shallots and sliced red chilli, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 29, "name": "Scrambled eggs on wholemeal toast", "desc": "Scrambled eggs on wholemeal toast (2 eggs, 2 slices)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 30, "name": "Peanut butter banana toast", "desc": "Peanut butter banana toast (2 slices wholemeal)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 33, "name": "Onigirazu, chicken katsu", "desc": "Japanese Onigirazu, chicken katsu (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 34, "name": "Onigirazu, salmon and egg", "desc": "Japanese Onigirazu, salmon and egg (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 37, "name": "Whisk eggs Benedict", "desc": "eggs Benedict (2 eggs on muffin)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 39, "name": "6-inch Italian B.M.T. (chicken ham, pepperoni, salami)", "desc": "6-inch Italian B.M.T. (Italian bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 40, "name": "6-inch Chicken Slice sub", "desc": "6-inch Chicken Slice sub (Italian bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 41, "name": "6-inch Steak & Cheese sub", "desc": "6-inch Steak & Cheese sub (Italian bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 42, "name": "6-inch seafood sensation sub", "desc": "6-inch seafood sensation sub (Italian bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 44, "name": "Onigiri and boiled egg", "desc": "Japanese Onigiri and boiled egg (1 tuna onigiri, 1 egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 46, "name": "Paris Baguette ham and cheese sandwich", "desc": "ham and cheese sandwich (1 sandwich)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 47, "name": "6-inch egg mayo sub", "desc": "6-inch egg mayo sub (Italian bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 48, "name": "Hainan toast with kaya, and eggs", "desc": "Hainan kaya toast set: two toasted bread sandwiches cut in half showing a thick slab of yellow butter and a layer of green pandan kaya jam in the middle, and next to them a small bowl of two soft runny half-boiled eggs with wobbly translucent whites, liquid orange yolks and a splash of dark soy sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 49, "name": "Steamed Hainan bread with kaya", "desc": "steamed Hainan bread: two thick fluffy soft white steamed bread slices stacked on a small plate, a slab of yellow butter and a thick layer of green pandan kaya jam between them, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 52, "name": "AK Noodles wantan mee", "desc": "Malaysian wantan mee: thin springy egg noodles tossed in dark soy sauce with slices of red-edged char siu, choy sum and small boiled wontons, with noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 54, "name": "Noodle Shack dry wantan noodles", "desc": "small boiled wontons with silky wrappers, with noodle shack dry noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 61, "name": "Mee tarik dry with minced beef", "desc": "dry hand-pulled noodles: a bowl of thick long hand-pulled wheat noodles with no soup, topped with a big heap of dark brown minced beef in sauce, chopped green spring onion and red chilli oil, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 66, "name": "Se'Indonesia bakso", "desc": "bakso: a bowl of clear beef broth with five big round grey-brown beef meatballs, thin glass noodles, green bok choy, fried shallots and chopped celery leaves, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 67, "name": "Wanhoi prawn wantan soup", "desc": "prawn wonton soup: a bowl of clear golden broth with six plump prawn wontons with silky wrappers floating in the soup, yellow chives and chopped spring onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 68, "name": "Yunnan crossing-the-bridge noodles", "desc": "Yunnan crossing-the-bridge rice noodles in a big bowl of clear soup with toppings, with yunnan noodles (1 set), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 69, "name": "Yun Nans rice noodles with minced pork", "desc": "Yunnan rice noodles in a black clay pot: thick round white rice noodles in red chilli-oil broth, topped with a heap of browned minced pork, dark green pickled mustard greens, chopped green chives and red chilli flakes, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 70, "name": "Bone soup rice noodles", "desc": "bone soup rice noodles: a big bowl of milky white pork bone broth with white rice noodles, slices of pork, half a soft-boiled egg, corn kernels, black fungus and spring onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 71, "name": "Hot Crush pork noodles", "desc": "Chinese pork noodles (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 72, "name": "Shihlin oyster mee sua", "desc": "oyster mee sua: a bowl of thin brown wheat vermicelli in thick brown starchy soup, topped with small plump grey oysters, green coriander leaves and a dab of minced garlic, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 73, "name": "Lontong", "desc": "lontong: a bowl of bright yellow coconut vegetable curry with white compressed rice cake cubes, cabbage, long beans, carrot, fried tofu pieces, half a hard-boiled egg and a dollop of dark red sambal on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 74, "name": "Nasi dagang", "desc": "nasi dagang: a mound of steamed rice speckled with brown fenugreek seeds, a thick piece of tuna fish in orange-brown gulai curry gravy beside it, a small pile of pickled cucumber and carrot strips, half a hard-boiled egg, one red chilli, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 75, "name": "Mee siam", "desc": "mee siam: a plate of thin orange-red fried rice vermicelli, topped with bean sprouts, green chives, strips of omelette, half a hard-boiled egg, fried tofu pieces and a halved green calamansi lime, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 77, "name": "Bar B Q Plaza jasmine rice", "desc": "Thai jasmine rice (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 81, "name": "Thai chicken rice (khao man gai)", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce, with thai (1 plate), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 85, "name": "Fong Lye Three-cup chicken set", "desc": "three-cup chicken in a small clay pot: chicken pieces coated in a dark brown glossy soy glaze with sauce drizzled over, whole garlic cloves, green basil leaves and red chillies scattered around, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 86, "name": "Long Jing Dongpo pork with rice", "desc": "a wooden bucket bowl filled with white rice, on top one big square of braised Dongpo pork belly with red-brown glossy skin sliced into four pieces, dark brown sauce drizzled over the pork and rice, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 87, "name": "Oversea char siew", "desc": "slices of red-edged glossy char siu barbecued pork (1 portion), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 88, "name": "Lu rou fan", "desc": "lu rou fan: a bowl filled with fluffy white rice, the rice clearly visible with separate grains around the edge of the bowl, the centre topped with dark brown braised diced pork belly and brown soy sauce soaking down into the rice, half a brown braised egg and yellow pickled radish slices at the side, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 90, "name": "Yat Yat Seng roast pork rice", "desc": "roast pork rice: a mound of white rice on a plate with sliced crispy siew yuk roast pork belly with golden crackling laid beside it, cucumber slices, a little dark sauce on the rice, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 92, "name": "Grand Imperial roast pork and duck rice", "desc": "roast pork and roast duck rice: a mound of white rice on a plate with two meats side by side, sliced crispy roast pork belly with golden crackling and sliced roast duck with dark red-brown glossy skin, cucumber slices, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 93, "name": "Peony roast meat rice", "desc": "Chinese roast meat rice (1 plate), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 94, "name": "Steamed chicken rice set", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce, with steamed set (1 set with soup and bean sprouts)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 96, "name": "Economy rice: steamed fish, tofu, greens", "desc": "economy rice plate: white rice with a piece of steamed white fish fillet with silver-grey skin and white flesh topped with ginger strips and soy sauce, soft white tofu cubes in gravy, stir-fried green vegetables, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 97, "name": "Esquire Kitchen sweet and sour fish with rice", "desc": "pieces of battered meat in glossy red sweet and sour sauce with pineapple, peppers and onion, with fish with rice (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 100, "name": "Nasi campur: ayam bakar, sayur, ulam", "desc": "Malaysian mixed rice: white rice with several small heaps of dishes such as curry chicken, vegetables and fried fish; grilled marinated chicken with charred edges; fresh Malay herbs and raw vegetables with sambal belacan, with : , sayur (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 101, "name": "Ikan bakar with rice and ulam", "desc": "ikan bakar: a plate with a mound of white rice, a grilled fish steak with charred dark skin coated in red sambal, raw ulam greens of fresh leaves, cucumber and long beans, and a small dish of sambal belacan, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 102, "name": "Nasi ayam penyet", "desc": "Indonesian ayam penyet: smashed fried chicken with red sambal, rice, tofu, tempeh and cucumber, with nasi (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 104, "name": "Grandmama's nyonya fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with nyonya (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 105, "name": "Las Meninas seafood paella, per person", "desc": "seafood paella, per person (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 106, "name": "Nasi goreng with fried egg", "desc": "Malaysian fried rice, reddish-brown, with egg and vegetables, with fried egg (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken", "n": 1}, {"id": 108, "name": "Black Canyon pad kra pao chicken rice", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce; Thai minced meat stir-fried with holy basil over rice with a fried egg (1 plate), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken", "n": 1}, {"id": 109, "name": "Jatujak pad kra pao with egg", "desc": "Thai minced meat stir-fried with holy basil over rice with a fried egg (1 plate)", "neg": "whole roast chicken", "n": 1}, {"id": 110, "name": "Thai Hou Sek basil pork rice", "desc": "Thai Hou Sek basil pork rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 111, "name": "Chakri basil minced pork rice", "desc": "Thai basil minced pork rice (With egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 112, "name": "Pad kra pao with rice and egg", "desc": "Thai minced meat stir-fried with holy basil over rice with a fried egg, with rice and egg (1 plate)", "neg": "whole roast chicken", "n": 1}, {"id": 113, "name": "Pad kra pao chicken, home-style", "desc": "Thai minced meat stir-fried with holy basil over rice with a fried egg, with chicken, home-style", "neg": "whole roast chicken", "n": 1}, {"id": 114, "name": "Nasi kandar", "desc": "nasi kandar: yellow turmeric rice flooded with mixed brown and orange curries, a piece of fried chicken, a green okra and half a boiled egg, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 116, "name": "Banana leaf rice, vegetarian", "desc": "South Indian banana leaf rice: white rice on a banana leaf with three small heaps of vegetable curries, a papadam and curry gravy, with rice, vegetarian (Rice, 3 vegetables, dhal, rasam), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 117, "name": "Tandoori chicken with naan", "desc": "red tandoori chicken pieces with onion and lemon; a soft blistered naan flatbread, with chicken (1 piece chicken, 1 plain naan), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 118, "name": "Barbeque Nation buffet, per person", "desc": "Indian buffet, per person (Grills, curries, rice and dessert), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 120, "name": "Vegetarian fried bee hoon", "desc": "vegetarian fried bee hoon: a plate of thin light-brown fried rice vermicelli noodles with separate strands clearly visible, shredded cabbage, carrot strips, sliced mushrooms and fried tofu pieces, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 121, "name": "Esquire Kitchen golden pumpkin noodles", "desc": "golden pumpkin noodles: a bowl of thick noodles in bright orange creamy pumpkin soup, topped with prawns, clams and green vegetables, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 124, "name": "Mee tarik fried with chicken", "desc": "fried hand-pulled noodles: a plate of thick stir-fried wheat noodles, brown with soy sauce, tossed with chicken slices, cabbage, carrot strips and spring onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 125, "name": "Pu Tien Heng Hwa fried bee hoon", "desc": "thin rice vermicelli, with heng hwa fried (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 126, "name": "Wok Hey fried udon", "desc": "fried udon: a plate of thick white udon noodles stir-fried brown with soy sauce, with cabbage, carrot, onion, slices of meat and a sprinkle of bonito flakes, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 127, "name": "Maggi goreng", "desc": "Maggi goreng: a plate of curly yellow instant noodles stir-fried orange-brown, with bean sprouts, green choy sum, tofu cubes, a fried egg on top and a lime half, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 128, "name": "Mee rebus", "desc": "mee rebus: yellow noodles in thick brown sweet potato gravy, topped with half a boiled egg, fried tofu cubes, sliced red and green chillies, green coriander leaves, fried shallots and a halved green calamansi lime, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 129, "name": "Mee bandung", "desc": "mee bandung: yellow noodles in thick orange-red prawn gravy, topped with a poached egg, prawns, sliced red and green chillies, green coriander leaves and a halved green calamansi lime, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 130, "name": "Soto ayam", "desc": "soto ayam: a bowl of clear yellow turmeric chicken broth with rice cake cubes, shredded chicken, bean sprouts, half a boiled egg, sliced red chilli, fried shallots, green celery leaves and a halved lime, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 131, "name": "Sup ayam with rice", "desc": "Malay chicken soup: clear spiced broth with chicken pieces, potato and carrot, with a bowl of rice, with rice (1 bowl soup, 1 cup rice), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 132, "name": "Laksam", "desc": "laksam: thick flat rice noodle sheets rolled into white spiral rolls and cut into rounds, sitting in a thick pale cream-grey fish and coconut gravy, topped with shredded cucumber, bean sprouts, chopped long beans, a red sambal dollop and a lime wedge, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 134, "name": "Robataya grilled skewers, per person", "desc": "Japanese grilled skewers, per person (6 skewers)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 136, "name": "Putien Heng Hwa fried bee hoon", "desc": "Heng Hwa fried bee hoon: a plate of very fine thin pale rice vermicelli noodles with separate strands clearly visible, tossed with clams, prawns, pork slices and green vegetables, topped with roasted peanuts and dark seaweed strips, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 137, "name": "Kim Lian Kee Hokkien mee", "desc": "a plate of KL Hokkien mee. A heap of thick round noodles braised in thick glossy very dark brown soy sauce gravy, the colour of dark chocolate brown, not black and not inky. Everything is the same dark brown colour together: the noodles, the pieces of cabbage, the prawns, the pork slices and the small crispy cubes of fried pork lard on top are all coated and stained dark brown by the sauce", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 138, "name": "Duck egg char kuey teow", "desc": "a plate of Penang char kuey teow. A loose pile of wide flat ribbon rice noodles, each strand a broad flat white band about as wide as a finger like wide fettuccine ribbons, clearly flat and wide rather than thin round strands, stir-fried and stained glossy brown with dark soy sauce, with slightly charred edges. Mixed in are three pink prawns, small cockles, crunchy bean sprouts, short green chive pieces and small yellow bits of scrambled egg, with duck egg (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 140, "name": "Wat tan hor", "desc": "wat tan hor: wide flat rice noodles lightly charred brown, covered in thick pale-yellow silky egg gravy with white egg ribbons, prawns, white squid rings, pork slices and green choy sum, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 143, "name": "Vegetarian curry mee", "desc": "Malaysian curry mee: orange coconut curry soup with yellow noodles, tofu puffs, long beans and cockles, with vegetarian (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 147, "name": "Asam laksa", "desc": "asam laksa: thick white round rice noodles in tangy brown-orange fish broth, topped with shredded cucumber, pineapple, red onion, mint leaves, sliced red chilli, a halved lime and a spoon of dark prawn paste, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 149, "name": "Mr Fish seafood noodles, milky broth", "desc": "seafood noodles in milky broth: a bowl full of thick white rice vermicelli in creamy milky white soup, topped with prawns, white fish slices, clams, squid rings, a tomato wedge and green vegetables, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 151, "name": "Fish head mee hoon, clear soup", "desc": "fish head noodle soup: rice vermicelli in clear soup with fried fish pieces, tomato and salted vegetables, with clear soup (1 bowl, no milk), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 155, "name": "Pao Xiang dry bak kut teh", "desc": "a claypot of Malaysian dry bak kut teh. A round dark brown clay pot with no soup, filled with chunks of pork ribs and pork belly coated in a thick, sticky, very dark almost black soy sauce glaze, mixed with whole dried red chillies, sliced green okra lady fingers and strips of dried cuttlefish, glossy and caramelised", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 157, "name": "Kwan Kee claypot rice", "desc": "Chinese claypot rice (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 159, "name": "Sweet and sour chicken rice", "desc": "pieces of battered meat in glossy red sweet and sour sauce with pineapple, peppers and onion; sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 161, "name": "Siwawa Guizhou wraps", "desc": "Guizhou siwawa: a round platter of many small piles of colourful shredded vegetables, carrot, cucumber, purple cabbage, bean sprouts, radish, with a stack of thin round white wrappers in the middle and a bowl of red dipping sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 163, "name": "Kim Gary baked cheese rice with chicken chop", "desc": "baked cheese rice: an oval white baking dish of rice with a chicken chop on top, everything covered with a blanket of melted golden-brown bubbly baked cheese with a little red tomato sauce showing at the edge, no fries, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 170, "name": "Fish & Co Fish in a Pan", "desc": "Fish in a Pan (Fish fillets with rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 171, "name": "Fish & Co seafood platter for one", "desc": "seafood platter for one (Fish, prawns, calamari and fries)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 172, "name": "Delirium Belgian mussels with fries", "desc": "Belgian mussels with fries (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 174, "name": "Laem Charoen deep-fried seabass with fish sauce", "desc": "deep-fried fish: thick pieces of golden-brown crispy deep-fried fish fillet on a plate, topped with green mango shreds, red onion slices and coriander, with a small bowl of fish sauce dip, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 176, "name": "Peng Chu steamed fish with chili", "desc": "steamed fish with chilli: thick slices of steamed white fish fillet with grey skin on an oval dish, covered with lots of chopped red and green chillies and garlic, in light soy sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 177, "name": "Xu's Cuisine Hunan chili fish", "desc": "Hunan chilli fish: thick slices of white fish fillet with grey skin on an oval dish, completely covered with bright red chopped pickled chillies, with green spring onion, in a red oily sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 178, "name": "De.Wan ikan bakar set", "desc": "grilled whole fish wrapped in charred banana leaf with red sambal (With rice and sambal), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 179, "name": "Robataya grilled saba set", "desc": "Japanese grilled saba set (With rice and miso soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 180, "name": "Steamed siakap with ginger and soy, rice and greens", "desc": "a plate with a mound of white rice, one thick slice of steamed white fish fillet with grey skin topped with ginger strips and spring onion in light soy sauce, and stir-fried green vegetables, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 181, "name": "Ikan kembung bakar with ulam and rice", "desc": "fresh Malay herbs and raw vegetables with sambal belacan, with ikan kembung bakar with and rice, bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 182, "name": "Grilled fish with salad and sweet potato", "desc": "Grilled fish with salad and sweet potato (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 183, "name": "Steamed fish with tofu and spring onion, no rice", "desc": "steamed fish with tofu: one big thick slice of steamed white fish fillet with grey skin in the centre of an oval dish, small soft white tofu cubes around it, shredded spring onion and ginger on top, in a pool of light soy sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 185, "name": "Secret Recipe chicken lasagna", "desc": "chicken lasagna (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 186, "name": "Pizza Hut cheesy baked spaghetti, bolognese", "desc": "Pizza Hut cheesy baked spaghetti, bolognese (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 187, "name": "Spaghetti aglio olio with chicken", "desc": "Spaghetti aglio olio with chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 188, "name": "ANTE pork belly aglio olio", "desc": "pork belly aglio olio (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 190, "name": "Ocean Twe12ve seafood pasta", "desc": "seafood pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 191, "name": "Aglio olio with prawns", "desc": "Aglio olio with prawns", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 192, "name": "Zucchini noodles with prawns and tomato", "desc": "Zucchini noodles with prawns and tomato (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 194, "name": "Kenny Rogers mac and cheese", "desc": "mac and cheese", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 197, "name": "Se'Indonesia gado-gado", "desc": "gado-gado: a plate of blanched vegetables, green long beans, cabbage, bean sprouts, potato slices, with fried tofu cubes, tempeh, half a hard-boiled egg and rice cake cubes, thick brown peanut sauce poured over, prawn crackers on the side, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 198, "name": "Daily Greens salmon salad", "desc": "Daily Greens salmon salad (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 199, "name": "Subway salad, roasted chicken", "desc": "salad, roasted chicken (No bread, standard veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 200, "name": "Cauli & Rice grain bowl", "desc": "Rice grain bowl (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 201, "name": "Lettuce wraps with minced chicken", "desc": "Chinese Lettuce wraps with minced chicken (1 serving), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 202, "name": "Quarter chicken with 2 sides", "desc": "Quarter chicken with 2 sides (Chicken, garden salad, corn)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 203, "name": "Chicken pita", "desc": "Chicken pita (1 pita)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 204, "name": "Quarter chicken with 3 sides", "desc": "Quarter chicken with 3 sides (Chicken, coleslaw, vegetables, mash)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 205, "name": "Sandwich, sub or wrap", "desc": "a long sub sandwich cut in half and stacked, crusty golden bread filled with grilled chicken slices, green lettuce, red tomato slices, cucumber and yellow cheese, with a tortilla wrap cut in half beside it showing the same filling, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 206, "name": "6-inch chicken teriyaki sub", "desc": "6-inch chicken teriyaki sub (Wholemeal, all the veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 207, "name": "6-inch tuna sub", "desc": "6-inch tuna sub (Wholemeal, all the veg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 208, "name": "6-inch Veggie Delite with egg", "desc": "6-inch Veggie Delite with egg (Wholemeal, all the veg, egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 209, "name": "Filet-O-Fish", "desc": "Filet-O-Fish (1 burger), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 210, "name": "McChicken", "desc": "McChicken (1 burger), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 212, "name": "Shake Shack Chick'n Shack", "desc": "Chick'n Shack (1 sandwich)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 213, "name": "Burger King fish burger", "desc": "Burger King fish burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 215, "name": "Marrybrown MB burger", "desc": "MB burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 216, "name": "Ayam Goreng McD", "desc": "Malaysian spiced fried chicken piece, crispy and dark golden", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 218, "name": "ANTE pork burger", "desc": "pork burger (1 burger with fries)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 221, "name": "Whopper Jr", "desc": "Whopper Jr (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 223, "name": "Original Recipe chicken, 2 pieces", "desc": "Original Recipe chicken, 2 pieces (With coleslaw)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 224, "name": "Zinger burger", "desc": "Zinger burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 225, "name": "Crispy chicken with rice", "desc": "Crispy chicken with rice (1 piece, rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 226, "name": "Chicken porridge (Marrybrown)", "desc": "smooth Cantonese rice porridge, with chicken (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 227, "name": "Original chicken, 2 pieces", "desc": "Original chicken, 2 pieces (With mashed potato)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 228, "name": "Salmon poke bowl", "desc": "Japanese Salmon poke bowl, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 229, "name": "Salad bowl", "desc": "a big salad bowl: green lettuce leaves, grilled chicken slices, half a boiled egg, red cherry tomatoes, cucumber slices, yellow corn, orange carrot shreds and avocado slices arranged in neat sections, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 230, "name": "Salmon sashimi and rice", "desc": "Japanese Salmon sashimi and rice (6 slices, 1 bowl rice, miso soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 233, "name": "Fish Bowl poke bowl, salmon", "desc": "Japanese Fish Bowl poke bowl, salmon (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 234, "name": "Fish Bowl poke bowl, tuna", "desc": "Japanese Fish Bowl poke bowl, tuna (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 236, "name": "Ichi Zen sashimi set", "desc": "Japanese sashimi set (With rice and miso soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 239, "name": "Empire Sushi California roll", "desc": "Japanese Sushi California roll (6 pieces), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 241, "name": "Makii Makii salmon hand roll", "desc": "Japanese salmon hand roll (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 243, "name": "Sushiro grilled eel nigiri", "desc": "Japanese grilled eel nigiri (1 plate of 2)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 244, "name": "Sushiro tuna nigiri", "desc": "Japanese tuna nigiri (1 plate of 2)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 245, "name": "Kinori spicy tuna nori taco", "desc": "Japanese spicy tuna nori taco (1 taco)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 246, "name": "Mama Kimbap tuna mayo kimbap", "desc": "Korean tuna mayo kimbap (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 251, "name": "Itsu salmon teriyaki bento", "desc": "Japanese salmon teriyaki bento (1 bento)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 254, "name": "Sukiya gyudon, regular", "desc": "Japanese gyudon, regular (1 bowl), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 255, "name": "Sukiya cheese gyudon", "desc": "Japanese cheese gyudon (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 256, "name": "Kampachi gyu teriyaki set", "desc": "Japanese gyu teriyaki set (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 258, "name": "Tenkai tendon", "desc": "Japanese tendon (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 259, "name": "Salmon teriyaki bento", "desc": "Japanese Salmon teriyaki bento (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 260, "name": "Udon in soup", "desc": "udon in soup: a deep bowl of clear light brown dashi broth with thick round white udon noodles, every thick strand clearly drawn, topped with green spring onion rings, two pink-and-white fish cake slices and one square of dark green seaweed, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 264, "name": "Menya Musashi tsukemen", "desc": "Japanese tsukemen (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 265, "name": "MyeongDong ramyeon", "desc": "Korean ramyeon (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 268, "name": "Monster Curry pork katsu curry", "desc": "Japanese Curry pork katsu curry (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 269, "name": "Monster Curry chicken katsu curry", "desc": "Malaysian chicken curry with potatoes in orange gravy, with katsu curry (1 plate), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 271, "name": "Ma Maison katsu curry", "desc": "Japanese Ma Maison katsu curry (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 272, "name": "Donkai katsu don", "desc": "Japanese katsu don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 273, "name": "DayOne katsu curry pork chop omelette rice", "desc": "Japanese katsu curry pork chop omelette rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 274, "name": "Gyudon", "desc": "Japanese Gyudon", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 275, "name": "Gyudon, mini with salad", "desc": "Japanese Gyudon, mini with salad (1 mini bowl, side salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 277, "name": "Miga beef bulgogi set", "desc": "Korean beef bulgogi set (With rice and sides)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 281, "name": "Kyochon Honey Original drumsticks", "desc": "Korean Honey Original drumsticks (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 283, "name": "MyeongDong Korean fried chicken", "desc": "Korean fried chicken (6 pieces), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 284, "name": "Soy garlic chicken rice box", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce, with soy garlic box (1 box)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 285, "name": "Kimbap", "desc": "kimbap: eight round slices of seaweed rice roll lined up on a rectangular plate, each slice a ring of dark green-black seaweed around white rice with a centre of yellow pickled radish, orange carrot, green spinach, yellow egg and pink ham, sesame seeds on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 286, "name": "Japchae", "desc": "japchae: a plate of glossy see-through brown sweet potato glass noodles tangled with orange carrot strips, green spinach, brown mushroom slices, red capsicum strips and thin beef strips, white sesame seeds on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 289, "name": "Cili Kampung asam pedas ikan with rice", "desc": "fish in a red sour spicy tamarind gravy with okra, with ikan with rice (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 290, "name": "Laem Charoen tom yum seafood", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with seafood (1 bowl), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 291, "name": "Melaka Nyonya asam pedas fish with rice", "desc": "fish in a red sour spicy tamarind gravy with okra, with nyonya fish with rice (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 292, "name": "Tai Er sauerkraut fish, per person", "desc": "sauerkraut fish: a big bowl of golden-yellow sour broth filled with many thin white fish slices, green pickled mustard greens, dried red chillies, white sesame seeds and green coriander, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 294, "name": "Amarin tom yum goong", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with goong (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 295, "name": "Celadon tom yum goong", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with goong (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 296, "name": "Absolute Thai tom yum seafood", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with thai seafood (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 298, "name": "Absolute Thai pad thai with prawns", "desc": "Thai pad thai: stir-fried rice noodles with prawns, egg, bean sprouts, crushed peanuts, chives and a lime wedge, with thai with prawns (1 plate), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 299, "name": "Bun cha", "desc": "bun cha: a bowl of warm clear amber dipping broth with four small round grilled pork patties and slices of grilled pork belly floating in it with thin carrot and green papaya slices, beside it a separate plate holding a neat bundle of thin white rice vermicelli noodles with every thin strand clearly drawn as long threads, and a small pile of fresh green lettuce and mint leaves, the noodles clearly thin strands and not rice grains, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 300, "name": "Boat Noodle pad thai", "desc": "Thai pad thai: stir-fried rice noodles with prawns, egg, bean sprouts, crushed peanuts, chives and a lime wedge, with noodle (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 301, "name": "Thai basil chicken rice with egg", "desc": "thai basil chicken rice with egg: a plate with a dome of white rice, beside it brown minced chicken stir-fried with green holy basil leaves and red chilli slices in dark glossy sauce, one crispy-edged fried egg with a runny yolk resting on the rice, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 304, "name": "Laem Charoen crab fried rice", "desc": "crab fried rice: a mound of pale golden egg fried rice with many large lumps and flakes of white crab meat mixed through and piled on top, chopped spring onion, with a lime wedge and cucumber slices, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 306, "name": "Boat Noodle beef noodles", "desc": "braised beef noodle soup: dark brown broth, thick noodles, braised beef chunks and bok choy, with noodle s", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 308, "name": "Al Halabi hummus with bread", "desc": "hummus with bread: a shallow plate of smooth beige hummus swirled with a pool of golden olive oil in the centre, a few whole chickpeas and red paprika dust, with triangles of pita bread, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 309, "name": "Al-Amar chicken shawarma plate", "desc": "chicken shawarma plate: a pile of thin shaved slices of browned spiced chicken on yellow rice, with white garlic sauce, pink pickled turnip, tomato and cucumber salad and a folded pita, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 310, "name": "Nasi mandi ayam", "desc": "Arabic nasi mandi: long-grain golden rice with a roasted chicken quarter on top and tomato salsa, with ayam (1 plate), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 311, "name": "Nasi mandi ayam, half rice", "desc": "Arabic nasi mandi: long-grain golden rice with a roasted chicken quarter on top and tomato salsa, with ayam, half rice (Quarter chicken, half rice, salad)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 312, "name": "Chicken kebab plate with salad", "desc": "Middle Eastern Chicken kebab plate with salad (2 skewers, salad, 1 pita), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 313, "name": "Hummus and falafel plate", "desc": "Middle Eastern Hummus and falafel plate (Hummus, 4 falafel, 1 pita, salad), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 314, "name": "Lentil soup with pita", "desc": "Middle Eastern Lentil soup with pita (1 bowl, 1 pita), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 315, "name": "Kofta plate with rice", "desc": "kofta plate: three long sausage-shaped grilled minced lamb kebabs with dark char marks on a mound of yellow rice, with a grilled tomato, raw onion slices and parsley, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 316, "name": "Mi xian in soup with meatballs", "desc": "Yunnan rice noodle soup with meatballs and greens, with in soup with meatballs (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 319, "name": "Pizza, seafood (tuna or prawn)", "desc": "Pizza, seafood", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 320, "name": "Pizza, cheese stuffed crust", "desc": "Pizza, cheese stuffed crust (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 322, "name": "Murtabak ayam", "desc": "Malaysian murtabak: thick square folded pan-fried bread stuffed with egg, onion and minced meat, with curry and pickled onion, with ayam (1 serving), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 323, "name": "Roti john", "desc": "Malaysian roti john: a long bread roll fried with egg and minced meat, topped with chilli sauce and mayonnaise (1 roll), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 326, "name": "Esquire Kitchen salted fish fried rice", "desc": "salted fish fried rice: a mound of egg fried rice with spring onion, dotted with small golden-brown crispy pieces of dried salted fish, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 328, "name": "Wok Hey black pepper beef fried rice", "desc": "black pepper beef fried rice: a takeaway paper box of golden egg fried rice topped with a big serving of dark brown glossy stir-fried beef slices in black pepper sauce with onion strips and cracked black pepper, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 329, "name": "Nando's spicy rice", "desc": "spicy rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 330, "name": "DIN pork chop fried rice", "desc": "pork chop fried rice: a mound of golden egg fried rice with one big golden-brown crispy fried pork chop, sliced into thick strips, laid on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 331, "name": "Prawn and egg white fried cauliflower rice, light", "desc": "cauliflower rice stir-fry: a plate of white crumbly cauliflower rice grains with small white cauliflower florets visible, fluffy egg white pieces, pink prawns and spring onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 336, "name": "Long Jing tofu with minced pork", "desc": "a clay pot full of soft white tofu cubes in spicy red-brown chilli sauce with minced pork and chopped spring onion, red chilli oil on the surface, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 338, "name": "Peng Chu Hunan chili pork", "desc": "Hunan chilli pork: thin slices of stir-fried pork with lots of long green chillies and red chillies, garlic slices and black beans, glossy with oil, on a plate, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 339, "name": "Nong Geng Ji pepper fried pork", "desc": "Chinese pepper fried pork (With rice), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 340, "name": "Tofu and minced chicken with long beans", "desc": "Chinese Tofu and minced chicken with long beans, bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 341, "name": "Fish slice soup with tofu and tomato", "desc": "Chinese Fish slice soup with tofu and tomato (1 bowl, no rice), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 346, "name": "Mixed vegetable minestrone", "desc": "Mixed vegetable minestrone", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 347, "name": "Minced pork soup", "desc": "Chinese Minced pork soup (1 bowl, with gourd or greens), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 348, "name": "Fried fish cake", "desc": "fried fish cake: several slices of golden-brown fried fish cake, oval and flat, pale and springy inside, on a plate with a small dish of red chilli sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 350, "name": "Chicken quesadilla", "desc": "Chicken quesadilla (1 tortilla, 4 wedges)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 352, "name": "Shojikiya beef sukiyaki set", "desc": "Japanese beef sukiyaki set", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 355, "name": "Samgyeopsal set with rice", "desc": "Korean Samgyeopsal set with rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 356, "name": "Bulgogi with rice", "desc": "bulgogi with rice: a plate with thin slices of marinated beef grilled to a glossy caramel-brown with onion, green spring onion and sesame seeds, a dome of white rice beside it and a small dish of red kimchi on the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 357, "name": "Kimchi jjigae with rice", "desc": "kimchi jjigae with rice: a small black stone pot of bright red kimchi stew with red kimchi cabbage pieces, white tofu slices, pork slices and green spring onion, light steam rising, with a small metal bowl of white rice beside the pot, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 359, "name": "Kimchi tofu stew, light", "desc": "Korean Kimchi tofu stew, light (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 360, "name": "Sundubu jjigae with rice", "desc": "sundubu jjigae with rice: a small black stone pot of bright red-orange spicy broth filled with soft white silken tofu in big spoonfuls, one raw egg yolk in the centre, two open clams and green spring onion, with a small metal bowl of white rice beside the pot, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 362, "name": "MyeongDong topokki", "desc": "Korean topokki (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 363, "name": "Mr Dakgalbi cheese dakgalbi, per person", "desc": "Korean cheese dakgalbi, per person (With rice cakes and cabbage)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 365, "name": "Ramyeon with egg", "desc": "ramyeon with egg: a small gold-coloured metal pot with two handles filled with curly yellow instant noodles in red-orange spicy broth, one egg with a round yellow yolk in the centre and chopped green spring onion on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 366, "name": "Jajangmyeon", "desc": "Korean jajangmyeon: thick noodles covered in glossy black bean sauce with diced pork and onion, with cucumber strips (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 367, "name": "Samgyetang", "desc": "samgyetang: a black stone pot of cloudy pale cream-white broth with one small whole young chicken sitting in the centre with pale tender skin, one pale ginseng root, two red dates and chopped green spring onion floating, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 369, "name": "K Fry Yangnyeom chicken", "desc": "Korean Yangnyeom chicken (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 371, "name": "Seafood pajeon", "desc": "seafood pajeon: one large round thin golden-brown crisp pancake cut into eight wedges on a plate, long green spring onion stalks running through it with pink prawns and white squid rings showing, one small dipping bowl of soy sauce on the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 372, "name": "Army stew (budae jjigae)", "desc": "army stew: a wide shallow pot of red spicy broth with toppings arranged in sections in a circle, a block of yellow instant noodles in the centre with one slice of orange cheese on top, pink sausage slices, luncheon meat slices, red kimchi, white tofu and white rice cakes, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 373, "name": "Mookata (Thai BBQ), typical meal", "desc": "Thai Mookata, typical meal (1 person)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 374, "name": "Yakiniku, typical meal", "desc": "Japanese Yakiniku, typical meal (1 person with rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 377, "name": "Pizza Hut chicken wings", "desc": "Pizza Hut chicken wings (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 378, "name": "BBQ pork ribs", "desc": "BBQ pork ribs (half rack)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 379, "name": "Lamb skewers (BBQ)", "desc": "Chinese Lamb skewers (5 skewers), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 380, "name": "Bar B Q Plaza pork set, per person", "desc": "Thai pork set, per person (Grilled pork slices with vegetables)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 381, "name": "Rocku yakiniku beef set, per person", "desc": "Japanese beef set, per person (Grilled beef, rice and sides)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 383, "name": "Grilled chicken wings (BBQ)", "desc": "Malaysian Grilled chicken wings (2 wings), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 385, "name": "Otak-otak", "desc": "otak-otak: orange spiced fish paste grilled in banana leaf parcels (2 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 386, "name": "Ikan pari bakar with rice", "desc": "ikan pari bakar: one large flat triangular wing of grilled stingray, a wide fan shape with long straight parallel ribbed lines of flesh running from the thick edge to the thin edge like the ribs of a fan, charred dark at the edges, the top spread thickly with dark red sambal, on a green banana leaf with a mound of white rice, half a lime and sliced red onion, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 387, "name": "Lok lok", "desc": "lok lok skewers of fish balls, sausages and tofu with satay and chilli sauce (10 sticks with sauce), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 389, "name": "Double cheeseburger", "desc": "Double cheeseburger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 390, "name": "Double beef burger", "desc": "Double beef burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 391, "name": "Mushroom Swiss burger", "desc": "Mushroom Swiss burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 392, "name": "Spicy Chicken McDeluxe", "desc": "Spicy Chicken McDeluxe (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 393, "name": "Chicken burger, crispy", "desc": "Chicken burger, crispy (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 394, "name": "Fish burger", "desc": "Fish burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 395, "name": "Ramly burger, ayam special", "desc": "Malaysian Ramly burger, ayam special (1 burger with egg wrap), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 400, "name": "Burger King onion rings", "desc": "Burger King onion rings", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 401, "name": "Burger King fries", "desc": "Burger King fries", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 402, "name": "Nando's peri-peri chips", "desc": "peri-peri chips", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 403, "name": "Texas Chicken cajun fries", "desc": "Chicken cajun fries", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 404, "name": "Carl's Jr onion rings", "desc": "onion rings", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 405, "name": "Happy Potato cheese potato balls", "desc": "Korean cheese potato balls (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 406, "name": "Potato Corner flavoured fries, regular", "desc": "flavoured fries, regular (Cheese or BBQ)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 407, "name": "Chica Bonita nachos with cheese", "desc": "nachos with cheese", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 408, "name": "Hotcakes with syrup", "desc": "Hotcakes with syrup (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 409, "name": "Apple pie (McD)", "desc": "Apple pie (1 pie)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 412, "name": "The Barn fried chicken", "desc": "fried chicken (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 414, "name": "KFC Hot & Spicy, breast", "desc": "Hot & Spicy, breast (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 415, "name": "Cheezy wedges, large", "desc": "Cheezy wedges, large", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 416, "name": "KFC nuggets", "desc": "nuggets (6 pieces), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 418, "name": "Burger King nuggets", "desc": "Burger King nuggets (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 419, "name": "Whipped potato with gravy", "desc": "Whipped potato with gravy", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 420, "name": "Texas Chicken mashed potato", "desc": "Chicken mashed potato", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 421, "name": "Coleslaw", "desc": "Coleslaw", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 422, "name": "Texas Chicken, 2 pieces", "desc": "Chicken, 2 pieces (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 423, "name": "Honey butter biscuit", "desc": "Honey butter biscuit (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 425, "name": "Root beer float", "desc": "Root beer float (1 mug)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 428, "name": "Wang Donkas pork cutlet", "desc": "Korean pork cutlet (With rice and sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 429, "name": "Wang Donkas cheese donkas", "desc": "Korean cheese donkas (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 430, "name": "Salmon don", "desc": "salmon don: a bowl of white rice covered with eight slices of raw salmon fanned in a circle, bright orange with white fat lines, a small mound of shiny orange salmon roe in the centre, shredded dark seaweed and a dab of green wasabi, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 431, "name": "Unagi don", "desc": "unagi don: a black rectangular lacquer box of white rice topped with two long grilled eel fillets, glossy dark caramel-brown glaze with a fine ridged surface, a sprinkle of white sesame seeds and three yellow pickled radish slices in the corner, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 432, "name": "Tempura udon", "desc": "tempura udon: a deep bowl of clear light brown broth with thick white udon noodles, two large prawn tempura in light golden lacy crisp batter with red tails resting across the rim of the bowl, green spring onion rings floating, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 433, "name": "Chicken karaage", "desc": "chicken karaage: six craggy golden-brown bite-sized fried chicken chunks with a rough bumpy crisp coating, piled on a green lettuce leaf on a small plate, a yellow lemon wedge and one small dipping bowl of pale yellow mayonnaise on the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 434, "name": "Okonomiyaki", "desc": "okonomiyaki: one thick round savoury cabbage pancake on a plate, covered with criss-cross lines of dark brown sauce and white mayonnaise, a sprinkle of green seaweed powder and a pile of thin curly pale brown bonito flakes on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 435, "name": "Yakitori", "desc": "yakitori: three wooden skewers lying side by side on a rectangular plate, each threaded with four chunks of grilled chicken thigh alternating with pieces of green leek, glazed in glossy brown sauce with dark char marks, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 436, "name": "California roll", "desc": "Japanese California roll (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 437, "name": "Salmon maki", "desc": "Japanese Salmon maki (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 438, "name": "Kappa maki (cucumber)", "desc": "Japanese Kappa maki (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 439, "name": "Spicy salmon roll", "desc": "Japanese Spicy salmon roll (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 440, "name": "Fried ebi roll", "desc": "Japanese Fried ebi roll (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 441, "name": "Cold soba", "desc": "cold soba: a square bamboo tray holding a neat mound of thin light brown buckwheat noodles with every strand drawn, shredded dark seaweed on top, a small cup of dark dipping sauce and a tiny dish of green spring onion and wasabi beside it on the tray, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 444, "name": "Black Canyon green curry with rice", "desc": "Thai green curry with chicken and eggplant in a bowl beside rice, with rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 446, "name": "Chakri massaman chicken curry", "desc": "Malaysian chicken curry with potatoes in orange gravy; Thai massaman curry with potato and chicken beside rice (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 447, "name": "Amarin green curry chicken", "desc": "Malaysian chicken curry with potatoes in orange gravy, with green (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 448, "name": "Celadon green curry chicken", "desc": "Malaysian chicken curry with potatoes in orange gravy, with green (With rice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 449, "name": "Chicken curry with potatoes and rice", "desc": "Malaysian chicken curry with potatoes in orange gravy, with potatoes and rice, bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 450, "name": "Som tam", "desc": "som tam: a plate of pale green shredded green papaya salad with orange carrot shreds, red cherry tomato halves, green long beans, roasted peanuts, dried shrimp and a lime wedge, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 452, "name": "Dragon-i la mian in spicy soup", "desc": "Chinese hand-pulled noodles, with in spicy soup (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 454, "name": "NakNak ramyeon with egg", "desc": "Korean ramyeon with egg (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 456, "name": "Yunnan spicy rice noodles", "desc": "Yunnan spicy rice noodles: a bowl of thick round white rice noodles in bright red chilli-oil broth, with minced pork, peanuts, pickled greens, chives and red chilli flakes, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 457, "name": "Spicy bone soup rice noodles", "desc": "Chinese Spicy bone soup rice noodles (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 458, "name": "Thai fried rice with chicken", "desc": "golden egg fried rice with spring onion and diced vegetables, with thai with chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 460, "name": "ChaTraMue Thai tea", "desc": "ChaTraMue Thai tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 461, "name": "Moo ping with sticky rice", "desc": "moo ping with sticky rice: three wooden skewers of flat grilled marinated pork, caramel brown with charred edges, beside a small mound of white sticky rice in a banana leaf, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 462, "name": "Chicken chop with black pepper sauce", "desc": "chicken chop with black pepper sauce: one flat boneless fried chicken thigh, golden-brown, on a plate, covered in dark brown black pepper gravy with many cracked pepper specks, with golden fries and a small heap of creamy coleslaw beside it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 463, "name": "Lamb chop with fries", "desc": "lamb chop with fries: two grilled lamb chops with long curved rib bones on a plate, browned seared meat with grill marks and glossy brown sauce over the top, with golden fries and a small heap of mixed green peas and carrot, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 464, "name": "ANTE kurobuta pork chop", "desc": "kurobuta pork chop (1 plate with sides)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 465, "name": "Dave's lamb shank", "desc": "lamb shank (With mash)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 467, "name": "Fish & Co clam chowder", "desc": "clam chowder (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 468, "name": "Sushi Zanmai chawanmushi", "desc": "Japanese Sushi Zanmai chawanmushi (1 cup), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 469, "name": "Pizza Hut garlic bread", "desc": "Pizza Hut garlic bread (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 470, "name": "Pizza Hut mushroom soup", "desc": "Pizza Hut mushroom soup (1 bowl), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 471, "name": "Big breakfast", "desc": "Big breakfast (eggs, sausage, toast, beans, hash brown)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 472, "name": "Avocado toast with egg", "desc": "Avocado toast with egg (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 473, "name": "Beef burger with fries", "desc": "Beef burger with fries (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 474, "name": "Seafood marinara pasta", "desc": "Seafood marinara pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 475, "name": "Creamy mushroom chicken pasta", "desc": "Creamy mushroom chicken pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 479, "name": "Stuff'd burrito", "desc": "burrito (1 burrito)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 480, "name": "Tim Hortons Farmer's breakfast wrap", "desc": "breakfast wrap (1 wrap)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 481, "name": "La Boca beef tacos", "desc": "beef tacos (3 tacos)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 482, "name": "Chica Bonita beef burrito", "desc": "beef burrito (1 burrito)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 483, "name": "Smoothie bowl", "desc": "Smoothie bowl (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 485, "name": "Bee Cheng Hiang bak kwa, pork", "desc": "bak kwa: three square flat sheets of glossy dark red barbecued dried pork jerky with charred edges, stacked on a plate, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 486, "name": "Siew yuk rice", "desc": "siew yuk rice: a mound of white rice on a plate with sliced crispy roast pork belly with golden crackling beside it, cucumber slices and a small dish of mustard sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 487, "name": "Sweet and sour pork with rice", "desc": "pieces of battered meat in glossy red sweet and sour sauce with pineapple, peppers and onion, with pork with rice (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 488, "name": "Putien sweet and sour pork with lychee", "desc": "pieces of battered meat in glossy red sweet and sour sauce with pineapple, peppers and onion, with pork with lychee, bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 489, "name": "Sweet and sour chicken with rice", "desc": "pieces of battered meat in glossy red sweet and sour sauce with pineapple, peppers and onion, with chicken with rice, bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 490, "name": "Kung pao chicken with rice", "desc": "diced chicken stir-fried with dried red chillies and peanuts, with chicken with rice (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 491, "name": "Salted egg chicken rice", "desc": "fried chicken pieces coated in golden salted egg yolk sauce with curry leaves, with rice (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 492, "name": "Butter prawns", "desc": "creamy butter prawns: six large whole orange prawns with their shells and tails, soaking in a deep pool of thick pale yellow creamy butter milk sauce that covers the bottom of the plate and coats every prawn, with green curry leaves and small red bird's eye chilli slices in the sauce, glossy and saucy, on a white plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 493, "name": "Bar B Q Plaza seafood set, per person", "desc": "Thai seafood set, per person (Prawns, squid and fish)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 494, "name": "Tangsuyuk (sweet and sour pork)", "desc": "Korean Tangsuyuk", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 495, "name": "Andaman grilled seafood platter, per person", "desc": "Thai grilled seafood platter, per person (Prawns, squid, fish and rice)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 496, "name": "Fei Fei Crab spicy crab pot, per person", "desc": "spicy crab pot: one big whole orange-red crab as the main item in the centre of a wide pot, smothered in glossy red chilli sauce, with dried chillies, corn pieces and prawns around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 498, "name": "Butter garlic prawns with rice", "desc": "Butter garlic prawns with rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 499, "name": "Loh mee", "desc": "loh mee: yellow noodles in thick dark starchy gravy with braised pork and egg (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 500, "name": "Sarawak laksa", "desc": "Sarawak laksa: rice vermicelli in light orange sambal soup with prawns, shredded chicken and omelette strips (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 501, "name": "Kolo mee", "desc": "Sarawak kolo mee: springy curly egg noodles with slices of red char siu and minced pork (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 502, "name": "Chilli pan mee", "desc": "dry chilli pan mee in a black bowl: thick white noodles at the bottom with no soup, toppings in separate neat piles on top: one poached egg with runny orange yolk in the centre, a heap of browned minced pork, a heap of dark red dried chilli flakes, a pile of tiny crispy fried anchovies, golden fried shallots, chopped green spring onion and green leafy vegetables, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 503, "name": "Claypot yee mee", "desc": "claypot yee mee: dark brown crispy fried noodles softened in brown gravy in a clay pot, with an egg cracked on top, prawns, pork slices and green choy sum, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 504, "name": "Economy bee hoon", "desc": "economy fried bee hoon: a plate of thin light-brown fried rice vermicelli noodles with separate noodle strands clearly visible, bean sprouts and cabbage strips, a fried egg and one slice of fried luncheon meat on the side, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 508, "name": "Cili Kampung nasi ayam bakar", "desc": "grilled marinated chicken with charred edges, with nasi (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 510, "name": "Longjing Dongpo pork rice", "desc": "a glossy cube of dark braised pork belly, with rice (1 set), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 511, "name": "Nasi kandar with fried chicken and curry", "desc": "Penang nasi kandar: white rice flooded with mixed dark curry gravies, fried chicken, okra and a boiled egg, with fried chicken and curry (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 512, "name": "Hameed nasi kandar", "desc": "Penang nasi kandar: white rice flooded with mixed dark curry gravies, fried chicken, okra and a boiled egg (1 plate with fried chicken), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 513, "name": "Masala King mutton biryani", "desc": "mutton biryani: a mound of yellow and orange long-grain biryani rice with one whole lamb shank with the bone sticking out laid on top, coated in brown masala, green coriander leaves sprinkled over, a small bowl of raita, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 514, "name": "Ayam percik with rice", "desc": "grilled ayam percik chicken glazed with orange spiced coconut sauce, with rice (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 515, "name": "Nasi goreng pattaya", "desc": "nasi goreng pattaya: a square parcel of thin yellow omelette on a white plate, cut open in the centre with the four corners folded back to reveal orange-brown fried rice with small prawns and green peas inside, thin zigzag lines of red chilli sauce across the omelette, cucumber and tomato slices on the side, no sausage, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 516, "name": "Nasi goreng USA", "desc": "nasi goreng USA: brown-orange fried rice with a fried egg, a serving of beef strips in red sauce, prawns, cucumber slices and a lettuce leaf, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 517, "name": "Sambal sotong with rice", "desc": "squid rings in glossy red sambal, with rice (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 519, "name": "Cheese naan", "desc": "a soft blistered naan flatbread, with cheese (1 piece), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 521, "name": "Butter chicken with naan", "desc": "creamy orange butter chicken curry with a piece of naan; a soft blistered naan flatbread (1 bowl, 1 naan), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 523, "name": "Vadai", "desc": "vadai: three deep-fried lentil fritters shaped like rings with a hole in the middle, dark golden-brown with a rough crunchy bumpy fried crust, specks of green chilli and curry leaves, with a small bowl of white coconut chutney, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 524, "name": "Puri with potato masala", "desc": "puri with potato masala: three round golden puffed fried puri breads on a plate with a small bowl of yellow potato masala curry with green peas and curry leaves, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 527, "name": "Ramen, miso", "desc": "ramen, miso: a deep bowl of opaque creamy golden-tan miso broth with thick wavy yellow noodles, topped with a heap of yellow sweetcorn, one square of butter, pale bean sprouts, brown minced pork and green spring onion, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 528, "name": "Ramen, shio", "desc": "Japanese Ramen, shio (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 529, "name": "Ramen, spicy miso", "desc": "Japanese Ramen, spicy miso (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 530, "name": "Ramen, tantanmen", "desc": "ramen, tantanmen: a deep bowl of opaque creamy orange sesame broth with a layer of red chilli oil on top, thin noodles, a mound of brown spicy minced pork in the centre, two green bok choy halves and white sesame seeds, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 531, "name": "Ramen, chicken paitan", "desc": "Japanese Ramen, chicken paitan (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 532, "name": "Tsukemen (dipping ramen)", "desc": "Japanese Tsukemen (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 533, "name": "Extra chashu", "desc": "Japanese Extra chashu (2 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 534, "name": "Ajitama (ramen egg)", "desc": "Japanese Ajitama (1 egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 536, "name": "Shin Ramyun", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 537, "name": "Shin Ramyun with egg and cheese", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 538, "name": "Buldak hot chicken ramen", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 539, "name": "Buldak carbonara ramen", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 540, "name": "Jjapaguri (ram-don)", "desc": "Korean Jjapaguri (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 541, "name": "Kimchi ramyeon", "desc": "Korean Kimchi ramyeon (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 542, "name": "Seafood ramyeon (jjamppong style)", "desc": "Korean Seafood ramyeon (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 543, "name": "Cheese ramyeon", "desc": "Korean Cheese ramyeon (1 pot)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 544, "name": "Instant noodles, curry (Maggi)", "desc": "instant curry noodles: a bowl of curly yellow instant noodles in orange-red curry soup with a poached egg, green vegetables and a few fish balls, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 545, "name": "Indomie mi goreng", "desc": "Indomie mi goreng: a plate of curly yellow-brown fried instant noodles without soup, topped with a sunny-side-up fried egg, crispy fried shallots and a few cucumber slices, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 546, "name": "Cup noodles", "desc": "a bowl of instant noodles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 547, "name": "Spaghetti aglio olio", "desc": "Spaghetti aglio olio (1 plate, no meat), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 548, "name": "Spaghetti aglio olio with prawns", "desc": "Spaghetti aglio olio with prawns (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 549, "name": "Spaghetti pomodoro", "desc": "Spaghetti pomodoro (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 550, "name": "Spaghetti arrabbiata", "desc": "Spaghetti arrabbiata (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 551, "name": "Spaghetti pesto with chicken", "desc": "Spaghetti pesto with chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 552, "name": "Spaghetti alfredo", "desc": "Spaghetti alfredo (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 553, "name": "Spaghetti vongole", "desc": "Spaghetti vongole (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 554, "name": "Spaghetti meatballs", "desc": "Spaghetti meatballs (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 555, "name": "Truffle mushroom pasta", "desc": "Truffle mushroom pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 556, "name": "Salted egg pasta", "desc": "Salted egg pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 557, "name": "Tom yum pasta", "desc": "Thai tom yum soup: clear orange-red spicy broth with prawns, mushrooms, lemongrass and lime leaves, with pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 558, "name": "Penne arrabbiata with chicken", "desc": "Penne arrabbiata with chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 559, "name": "Baked cheese pasta", "desc": "Baked cheese pasta (1 dish)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 560, "name": "Mac and cheese with chicken", "desc": "Mac and cheese with chicken (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 561, "name": "Fettuccine carbonara with mushroom", "desc": "Fettuccine carbonara with mushroom (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 562, "name": "Ravioli in tomato sauce", "desc": "Ravioli in tomato sauce (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 563, "name": "Nasi goreng cina", "desc": "light golden Chinese-style egg fried rice with peas, carrot and spring onion (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 564, "name": "Nasi goreng ikan masin", "desc": "nasi goreng ikan masin: brown fried rice with a fried egg on top, small golden-brown crispy pieces of dried salted fish, long beans, cucumber slices on the side, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 565, "name": "Nasi goreng tom yum", "desc": "nasi goreng tom yum: a mound of orange-red fried rice by itself with small white squid rings and small pink prawns mixed in, lime leaf strips and a red chilli, no side dishes, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 566, "name": "Nasi goreng ayam", "desc": "Malaysian fried rice, reddish-brown, with egg and vegetables, with ayam (1 plate with fried chicken), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 567, "name": "Nasi goreng seafood", "desc": "Malaysian fried rice, reddish-brown, with egg and vegetables, with seafood (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 568, "name": "Nasi goreng belacan", "desc": "Malaysian fried rice, reddish-brown, with egg and vegetables, with belacan (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 569, "name": "Nasi goreng daging merah", "desc": "nasi goreng daging merah: red-orange fried rice with many strips of red glazed beef on top, a fried egg and cucumber slices, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 570, "name": "Yang chow fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with yang chow (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 571, "name": "Garlic fried rice", "desc": "golden egg fried rice with spring onion and diced vegetables, with garlic (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 574, "name": "Little Cravings nasi lemak with fried chicken", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg, with fried chicken (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 575, "name": "Se'Indonesia nasi ayam penyet", "desc": "Indonesian ayam penyet: smashed fried chicken with red sambal, rice, tofu, tempeh and cucumber, with nasi (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 577, "name": "Jollibee Chickenjoy, 1 piece with rice", "desc": "Chickenjoy, 1 piece with rice (1 piece, rice, gravy)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 578, "name": "Dirty duck with rice", "desc": "dirty duck with rice: half a deep-fried crispy duck with dark brown crackly skin beside a mound of white rice, with three small sambals, long beans and cucumber, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 579, "name": "Meriah platter", "desc": "Malaysian platter (Pandan rice, fried chicken, beef rendang), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 580, "name": "Nasi lemak sambal sotong", "desc": "nasi lemak with sambal sotong: a dome of white coconut rice, a big serving of squid rings coated in dark red glossy sambal sauce, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 581, "name": "Nasi lemak telur mata", "desc": "Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg, with telur mata (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken", "n": 1}, {"id": 582, "name": "Nasi lemak telur mata", "desc": "nasi lemak: a dome of white coconut rice with a sunny-side-up fried egg with a round yellow yolk on top, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts and cucumber slices, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 583, "name": "Roti boom Milo", "desc": "roti boom Milo: one small thick round golden-brown flaky flatbread coiled in a spiral on a plate, covered with a heavy dusting of brown Milo chocolate malt powder and a drizzle of white condensed milk, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 585, "name": "Veggie burger", "desc": "Veggie burger (1 burger)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 586, "name": "Tuna nigiri", "desc": "Japanese Tuna nigiri (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 587, "name": "Tamago nigiri", "desc": "Japanese Tamago nigiri (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 588, "name": "Unagi nigiri", "desc": "Japanese Unagi nigiri (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 589, "name": "Inari sushi", "desc": "Japanese Inari sushi (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 590, "name": "Makii Makii onigiri, tuna mayo", "desc": "Japanese onigiri, tuna mayo (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 591, "name": "Salmon mentai aburi", "desc": "Japanese Salmon mentai aburi (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 593, "name": "Shucked fresh oysters", "desc": "fresh oysters (6 oysters)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 594, "name": "Chirashi don", "desc": "chirashi don: a bowl of sushi rice covered with a colourful scatter of cubed raw fish, orange salmon, deep red tuna and pale white fish, with yellow egg cubes, green cucumber cubes, shiny orange roe and one green shiso leaf, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 595, "name": "Cappuccino", "desc": "Cappuccino", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 596, "name": "Flat white", "desc": "Flat white", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 599, "name": "Antipodean flat white", "desc": "flat white", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 600, "name": "illy cappuccino", "desc": "cappuccino", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 601, "name": "Latte with oat milk, iced", "desc": "Latte with oat milk, iced", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 603, "name": "Black Canyon iced cappuccino", "desc": "iced cappuccino (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 606, "name": "Gula melaka latte, iced", "desc": "Gula melaka latte, iced", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 607, "name": "Americano, iced", "desc": "Americano, iced", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 615, "name": "Milo ais kurang manis", "desc": "a glass of chocolate malt drink, with ais kurang manis (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 616, "name": "Milo dinosaur", "desc": "milo dinosaur: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with opaque chocolate-brown iced malt drink with ice cubes, topped with a tall heaped mound of dry dark brown chocolate malt powder piled high above the rim like a small hill, a little powder dusted on the rim, and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 618, "name": "Thai milk tea, iced", "desc": "a plastic cup of milk tea, with thai , iced (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 619, "name": "Mr Tuk Tuk Thai iced tea", "desc": "Thai iced tea (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 622, "name": "Chicha San Chen Dong Ding oolong milk tea", "desc": "a plastic cup of milk tea, with dong ding oolong, bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 623, "name": "Matcha milk tea with pearls", "desc": "a plastic cup of milk tea; with black tapioca pearls at the bottom and a thick straw, with matcha", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 625, "name": "ChaTraMue Thai green tea", "desc": "ChaTraMue Thai green tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 626, "name": "Chocolate milk tea with pearls", "desc": "a plastic cup of milk tea; with black tapioca pearls at the bottom and a thick straw, with chocolate", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 627, "name": "Fruit tea with pearls", "desc": "with black tapioca pearls at the bottom and a thick straw, with fruit tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 628, "name": "Cheese foam tea", "desc": "Cheese foam tea", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 629, "name": "Tea macchiato", "desc": "Tea macchiato", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 634, "name": "Banh mi", "desc": "Banh mi (1 sandwich)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 635, "name": "Vietnamese fresh spring rolls", "desc": "golden crispy fried spring rolls, with fresh s (2 rolls with dip)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 636, "name": "Beancurd skin roll, steamed", "desc": "dim sum beancurd skin rolls: three flat rectangular rolls wrapped in golden-brown wrinkled layered tofu skin, lying side by side on a small plate in a little light brown sauce, one roll cut open to show minced pork and prawn filling, chopped spring onion on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 637, "name": "Udon, kitsune", "desc": "Japanese Udon, kitsune (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 638, "name": "Udon, beef (niku)", "desc": "Japanese Udon, beef (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 639, "name": "Udon, curry", "desc": "Japanese Udon, curry (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 640, "name": "Yaki udon", "desc": "Japanese Yaki udon (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 641, "name": "Yakisoba", "desc": "yakisoba: a plate of brown stir-fried wheat noodles glossy with dark sauce, mixed with green cabbage, orange carrot strips and pork slices, topped with a small pile of red pickled ginger, green seaweed powder and thin white mayonnaise lines, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 642, "name": "Fish curry with rice", "desc": "fish curry with okra in red gravy, with rice (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 643, "name": "Mutton curry with rice", "desc": "dark mutton curry, with rice (1 plate), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 644, "name": "Vegetable curry with rice", "desc": "Indian Vegetable curry with rice (1 plate), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 645, "name": "Japanese curry rice, chicken", "desc": "Japanese curry rice, chicken (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 646, "name": "Japanese curry rice, beef", "desc": "Japanese curry rice, beef (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 647, "name": "Massaman curry with rice", "desc": "massaman curry with rice: a bowl of thick brown-orange curry with chunks of beef, whole potatoes, onion and peanuts, beside a mound of white rice, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 648, "name": "Red curry chicken with rice", "desc": "Thai red curry chicken with rice: a bowl of bright red-orange creamy coconut curry with clearly visible slices of chicken breast meat, bamboo shoot strips, green Thai basil leaves and red chilli slices, beside a mound of white rice, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 649, "name": "Kari kepala ikan", "desc": "fish head curry with okra and eggplant in a clay pot (1 person's share with rice), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 650, "name": "Fish porridge", "desc": "smooth Cantonese rice porridge, with fish (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 651, "name": "Pork porridge", "desc": "smooth Cantonese rice porridge, with pork (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 652, "name": "Plain porridge with side dishes (Teochew)", "desc": "Teochew porridge set: a bowl of plain watery white rice porridge with three small side dishes beside it: half a salted duck egg with orange yolk, braised peanuts with pickled mustard greens, and braised brown tofu, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 653, "name": "Bubur lambuk", "desc": "bubur lambuk: a bowl of thick savoury rice porridge, light yellow-brown with spices, with shreds of beef, bits of carrot, fried shallots and chopped celery leaves on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 654, "name": "Oat porridge, savoury", "desc": "smooth Cantonese rice porridge, with oat , savoury (1 bowl with egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 655, "name": "Scrambled eggs", "desc": "Scrambled eggs (2 eggs), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 656, "name": "Fried egg", "desc": "Fried egg (1 egg), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 658, "name": "Poached eggs", "desc": "Poached eggs (2 eggs)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 659, "name": "Half-boiled eggs", "desc": "a small bowl of two runny soft-boiled eggs with soy sauce and white pepper (2 eggs with soy sauce), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 660, "name": "Telur bungkus", "desc": "telur bungkus: a square parcel of thin yellow omelette folded with its four corners open in the middle, showing a filling of minced meat with green peas and diced carrots in red sauce, coriander on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 661, "name": "Tuna sandwich", "desc": "Tuna sandwich (2 slices bread)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 662, "name": "Chicken sandwich, grilled", "desc": "Chicken sandwich, grilled (2 slices bread)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 663, "name": "Ham and cheese sandwich", "desc": "Ham and cheese sandwich (2 slices bread)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 664, "name": "Grilled cheese sandwich", "desc": "Grilled cheese sandwich (2 slices bread)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 665, "name": "BLT sandwich", "desc": "BLT sandwich (2 slices bread)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 666, "name": "Caesar salad, no chicken", "desc": "Caesar salad, no chicken (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 667, "name": "Greek salad", "desc": "Greek salad (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 668, "name": "Grilled chicken salad", "desc": "Grilled chicken salad (1 bowl, dressing on the side)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 669, "name": "Tuna salad", "desc": "Tuna salad (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 670, "name": "Poke bowl, salmon", "desc": "Japanese Poke bowl, salmon (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 671, "name": "Ayam goreng berempah", "desc": "ayam goreng berempah: a fried chicken leg with a dark reddish-brown spiced crust, covered in crispy fried spice crumbs and curry leaves, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 672, "name": "Ayam goreng kunyit", "desc": "ayam goreng kunyit: small pieces of fried chicken stained bright yellow with turmeric, stir-fried with long beans, sliced onions and red chilli, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 673, "name": "Chicken tenders", "desc": "Chicken tenders (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 674, "name": "Chicken wings, fried", "desc": "Chicken wings, fried (3 wings)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 675, "name": "Roast chicken, quarter", "desc": "Roast chicken, quarter (1 leg quarter with skin)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 676, "name": "Nasi minyak", "desc": "golden ghee rice with raisins and cashew nuts (1 cup), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 677, "name": "Nasi tomato", "desc": "nasi tomato with ayam masak merah: a mound of orange-red tomato rice with raisins and cashews, beside a piece of chicken in thick red sauce and pickled cucumber and pineapple acar, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 678, "name": "Nasi briyani, rice only", "desc": "Malaysian nasi briyani: long-grain yellow and orange spiced rice with a chicken curry, papadam and acar, with rice only (1 cup), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 680, "name": "Japanese rice", "desc": "Japanese rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 681, "name": "Cauliflower rice", "desc": "Cauliflower rice (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 682, "name": "Butter toast", "desc": "butter toast: two slices of golden-brown toasted white bread cut into four triangles and stacked on a small plate, a thick pale yellow slab of cold butter melting between the slices and a glossy buttery shine on top, plain with no kaya, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 683, "name": "Peanut butter toast", "desc": "peanut butter toast: two slices of golden toasted bread on a small plate, one spread thickly from edge to edge with smooth light brown peanut butter showing knife swirl marks, the other slice leaning against it, plain with no banana and no jam, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 684, "name": "Cottage cheese toast with tomato", "desc": "Cottage cheese toast with tomato (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 688, "name": "Tim Tam Dark Chocolate", "desc": "Tim Tam Dark Chocolate", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 689, "name": "KitKat Matcha", "desc": "chocolate-covered wafer fingers", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 690, "name": "KitKat Strawberry", "desc": "chocolate-covered wafer fingers", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 693, "name": "Cadbury Dairy Milk Oreo", "desc": "chocolate sandwich cookies with white cream", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 697, "name": "Beryl's chocolate", "desc": "chocolate", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 698, "name": "Pocky Chocolate", "desc": "thin biscuit sticks dipped in coating", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 699, "name": "Choki Choki", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 701, "name": "Pocky Matcha", "desc": "thin biscuit sticks dipped in coating", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 702, "name": "Pocky Cookies & Cream", "desc": "thin biscuit sticks dipped in coating", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 703, "name": "Pocky Banana", "desc": "thin biscuit sticks dipped in coating", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 704, "name": "Pocky Almond Crush", "desc": "thin biscuit sticks dipped in coating", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 705, "name": "Yan Yan Chocolate", "desc": "small biscuits with chocolate dip", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 707, "name": "Oreo Strawberry Creme", "desc": "chocolate sandwich cookies with white cream", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 708, "name": "Oreo Chocolate Creme", "desc": "chocolate sandwich cookies with white cream", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 709, "name": "Chipsmore Original", "desc": "chocolate chip cookies", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 710, "name": "Chipsmore Double Chocolate", "desc": "chocolate chip cookies", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 712, "name": "Munchy's Oat Krunch Strawberry & Blackcurrant", "desc": "oat cookies", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 713, "name": "Subway chocolate chip cookie", "desc": "chocolate chip cookies", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 715, "name": "Tokyo Milk Cheese Factory cheese cookies", "desc": "Japanese cheese cookies (2 cookies)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 717, "name": "Cafe Kitsune fox shortbread", "desc": "Cafe Kitsune fox shortbread (1 cookie)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 718, "name": "Ferrero Rocher", "desc": "round chocolate truffles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 719, "name": "Le Bourne chocolate pralines", "desc": "pralines (4 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 721, "name": "Kinder Chocolate", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 722, "name": "Toblerone", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 723, "name": "Snickers", "desc": "a bar of milk chocolate broken into pieces", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 724, "name": "M&M's Milk Chocolate", "desc": "colourful sugar-coated chocolate buttons", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 727, "name": "Super Ring Cheese", "desc": "Super Ring Cheese", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 728, "name": "Hershey's Kisses", "desc": "round chocolate truffles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 729, "name": "Lindt Lindor", "desc": "round chocolate truffles", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 730, "name": "Hello Panda Chocolate", "desc": "small biscuits with chocolate dip", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 731, "name": "Twisties Chicken", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 732, "name": "Potato chips", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 734, "name": "Koala's March Chocolate", "desc": "March Chocolate", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 735, "name": "Mister Potato Hot & Spicy", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 737, "name": "Doritos Nacho Cheese", "desc": "Doritos Nacho Cheese", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 738, "name": "Mamee Monster Hot & Spicy", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 739, "name": "Popo Fish Muruku", "desc": "Popo Fish Muruku", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 740, "name": "Lotte Choco Pie", "desc": "a chocolate-coated marshmallow cake", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 741, "name": "Julie's Peanut Butter Sandwich", "desc": "Peanut Butter Sandwich", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 742, "name": "Julie's Cheese Sandwich", "desc": "Cheese Sandwich", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 744, "name": "Lexus Chocolate Cream Sandwich", "desc": "Lexus Chocolate Cream Sandwich", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 745, "name": "Wholemeal crackers with cheese", "desc": "Wholemeal crackers with cheese (4 crackers, 1 slice)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 746, "name": "Julie's Love Letters", "desc": "Love Letters", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 747, "name": "Apollo Chocolate Layer Cake", "desc": "small layer cake squares", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 748, "name": "Apollo Pandan Layer Cake", "desc": "small layer cake squares", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 749, "name": "Twisties Cheese", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 750, "name": "Mamee Snek Ku", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 752, "name": "Mister Potato Original", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 753, "name": "Mister Potato Barbecue", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 754, "name": "Mamee Monster Barbecue", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 755, "name": "Twisties Barbecue Curry", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 756, "name": "Pringles Original", "desc": "a small stack of curved potato crisps", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 757, "name": "Pringles Sour Cream & Onion", "desc": "a small stack of curved potato crisps", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 758, "name": "Pringles Barbecue", "desc": "a small stack of curved potato crisps", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 759, "name": "Pringles Cheesy Cheese", "desc": "a small stack of curved potato crisps", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 760, "name": "Jacker potato chips", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 761, "name": "Pringles Hot & Spicy", "desc": "a small stack of curved potato crisps", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 762, "name": "Lay's Sour Cream & Onion", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 763, "name": "Lay's Nori Seaweed", "desc": "potato crisps in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 764, "name": "Calbee Jagabee", "desc": "Calbee Jagabee", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 765, "name": "Ding Dang", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 766, "name": "Mentos", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 767, "name": "Hacks", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 769, "name": "Haw flakes", "desc": "colourful small sweets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 770, "name": "Absolute Thai mango sticky rice", "desc": "mango sticky rice: a mound of white sticky rice with white coconut cream poured over and toasted yellow mung beans on top, beside slices of ripe yellow mango, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 772, "name": "Jatujak mango sticky rice", "desc": "Thai mango sticky rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 773, "name": "Yomie's purple rice yogurt", "desc": "purple rice yogurt", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 774, "name": "Pai tee", "desc": "pai tee: small crispy golden top-hat cups filled with shredded vegetables and a prawn (4 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 776, "name": "Snowflake taro balls dessert", "desc": "taro balls dessert: a bowl of small chewy balls in purple, orange and yellow, on shaved ice with red beans, black grass jelly cubes and a pour of milk, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 777, "name": "bhc cheese balls", "desc": "Korean cheese balls (5 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 778, "name": "Pineapple tarts", "desc": "small golden pineapple tarts with jam (3 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 781, "name": "Zok bamboo noodles with char siu", "desc": "slices of red-edged glossy char siu barbecued pork, with bamboo noodles (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 785, "name": "Mak's Chee wonton noodle soup", "desc": "Hong Kong wonton noodle soup: a small bowl of thin yellow egg noodles in clear golden broth with four plump prawn wontons with pale silky wrappers on top and a few yellow chives, no char siu, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 786, "name": "Ki Ki dry noodles with scallion oil", "desc": "Chinese dry noodles with scallion oil (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 787, "name": "Soong Kee beef ball noodles, dry", "desc": "dry beef ball noodles: a plate of thin egg noodles tossed in dark soy sauce topped with minced beef, with a small bowl of clear soup holding four brown beef balls, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 789, "name": "Jipinhe scone, plain", "desc": "scone, plain (1 scone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 790, "name": "Jipinhe chocolate scone", "desc": "chocolate scone (1 scone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 791, "name": "Petite Maison pain au chocolat", "desc": "pain au chocolat (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 794, "name": "Elementary beef pie", "desc": "beef pie (1 pie with mash)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 799, "name": "Baskin Robbins ice cream cake", "desc": "ice cream cake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 800, "name": "Chili's Southwestern eggrolls", "desc": "Southwestern eggrolls (1 starter)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 806, "name": "Gordon Ramsay beef Wellington", "desc": "beef Wellington (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 807, "name": "Weissbrau pork knuckle", "desc": "pork knuckle (Half portion with sauerkraut)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 808, "name": "Corks Out steak with fries", "desc": "steak with fries (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 809, "name": "Grilled lean beef with asparagus and mash", "desc": "Grilled lean beef with asparagus and mash (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 810, "name": "Roti john", "desc": "roti john: a long soft bun split and filled with a layer of egg omelette and minced meat, toasted golden on top, cut into three pieces in a row, with zigzag lines of red chilli sauce and white mayonnaise, cucumber slices on the side, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 812, "name": "Din Tai Fung chicken xiao long bao", "desc": "xiao long bao soup dumplings with fine pleats in a bamboo steamer, with chicken (5 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 813, "name": "Dragon-i deep-fried dumplings", "desc": "Chinese pleated dumplings, with deep-fried (4 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 816, "name": "B.O.D steamed bun with meat", "desc": "steamed meat bun: a round white fluffy steamed bun with a pleated top, torn open to show juicy minced pork filling, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 819, "name": "Mr Bean soya pancake with peanut", "desc": "soya pancake with peanut: a thick round golden-brown pancake folded in half, filled with crushed peanuts and sugar, cut to show the filling, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 820, "name": "dipndip chocolate crepe", "desc": "chocolate crepe (1 crepe)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 823, "name": "Beard Papa chocolate cream puff", "desc": "Japanese chocolate cream puff (1 puff)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 826, "name": "Pan-fried pork dumplings", "desc": "Chinese pleated dumplings, with pan-fried pork (8 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 827, "name": "K Fry Berry-Matcha Mochi Bingsoo", "desc": "Korean Berry-Matcha Mochi Bingsoo (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 834, "name": "Ma Yi Lan Lanzhou beef noodles", "desc": "braised beef noodle soup: dark brown broth, thick noodles, braised beef chunks and bok choy, with lanzhou s (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 835, "name": "Nippori matcha soft serve", "desc": "Japanese matcha soft serve (1 cone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 838, "name": "Boat Noodle mango sticky rice", "desc": "Thai Noodle mango sticky rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 839, "name": "ParaThai mango sticky rice", "desc": "Thai mango sticky rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 840, "name": "Kuih, 2 pieces", "desc": "two pieces of Malay kuih on a banana leaf: one rectangular kuih lapis with many thin pink and white layers, and one diamond-shaped seri muka with a smooth green pandan custard layer on top of a white glutinous rice layer, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 843, "name": "Yin's sourdough loaf slice", "desc": "loaf slice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 844, "name": "Shae.co bagel sandwich", "desc": "bagel sandwich (1 bagel)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 846, "name": "Pho Vietz fried spring rolls", "desc": "golden crispy fried spring rolls, with fried s (3 rolls)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 848, "name": "Black Tap CrazyShake, cookies and cream", "desc": "CrazyShake, cookies and cream (1 shake)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 852, "name": "Dunkin' chocolate frosted donut", "desc": "chocolate frosted donut (1 donut)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 855, "name": "Pork and chive dumplings, boiled", "desc": "Chinese pleated dumplings, with pork and chive , boiled (10 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 856, "name": "Vegetable dumplings, steamed", "desc": "steamed vegetable dumplings: five pleated dumplings with translucent pale green wrappers showing green vegetable filling, in a bamboo steamer, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 858, "name": "Burger King sundae, chocolate", "desc": "Burger King sundae, chocolate (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 859, "name": "108 Matcha Saro matcha soft serve", "desc": "Japanese matcha soft serve (1 cone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 861, "name": "Gordon Ramsay sticky toffee pudding", "desc": "sticky toffee pudding (1 dessert)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 863, "name": "Mango pudding", "desc": "mango pudding: a small dome of smooth bright orange-yellow mango pudding on a plate with white evaporated milk poured around it and cubes of fresh mango on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 864, "name": "Din Tai Fung spicy wontons", "desc": "spicy wontons: six wontons with silky wrappers in a shallow white bowl, sitting in a pool of bright red chilli oil and dark vinegar sauce, red chilli oil drizzled over them, chopped spring onion and garlic on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 867, "name": "Longan and sea coconut", "desc": "longan and sea coconut dessert: a glass bowl of clear pale golden syrup with ice cubes, round translucent white longan fruits and oval translucent pale-yellow slices of sea coconut, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 869, "name": "dipndip waffle with chocolate", "desc": "waffle with chocolate (1 waffle)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 873, "name": "Mille crepe, original", "desc": "Mille crepe, original (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 874, "name": "Jollibee peach mango pie", "desc": "peach mango pie (1 pie)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 875, "name": "Apple pie", "desc": "Apple pie (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 876, "name": "Chicken pie", "desc": "Chicken pie (1 pie)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 877, "name": "Quiche", "desc": "Quiche (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 881, "name": "Gindaco takoyaki, cheese mentai", "desc": "Japanese takoyaki, cheese mentai (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 882, "name": "Okonomi by Tokyo Don okonomiyaki", "desc": "Japanese okonomiyaki (1 pancake)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 883, "name": "Putien Bian Rou soup (meat dumplings)", "desc": "Bian Rou soup: a bowl of clear broth with many small translucent thin-skinned meat dumplings, purple seaweed and chopped spring onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 884, "name": "Yuen Kee pork and chive dumplings, boiled", "desc": "Chinese pleated dumplings, with pork and chive , boiled (10 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 887, "name": "Sambal kangkung", "desc": "sambal kangkung: a plate of stir-fried dark green water spinach with long hollow stems, coated in red chilli belacan sambal with sliced red chillies, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 888, "name": "Stir-fried greens", "desc": "Stir-fried greens (1 plate (kangkung, sawi))", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 890, "name": "Baked salmon with sweet potato and greens", "desc": "Baked salmon with sweet potato and greens", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 892, "name": "Dumplings in soup with greens", "desc": "Chinese pleated dumplings, with in soup with greens, bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 893, "name": "Konjac noodle soup with prawns and greens", "desc": "Chinese Konjac noodle soup with prawns and greens (1 serving), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 894, "name": "Soba with chicken and edamame", "desc": "Japanese Soba with chicken and edamame (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 896, "name": "Lotus root soup, vegetarian", "desc": "lotus root soup: a bowl of clear light brown broth with round slices of lotus root showing their holes, peanuts, red dates and carrot chunks, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 898, "name": "Steamed egg with minced pork and rice", "desc": "Chinese Steamed egg with minced pork and rice, bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 899, "name": "Steamed egg with minced meat and rice", "desc": "two separate dishes side by side: a shallow round dish of smooth flat pale-yellow silky steamed egg custard topped with browned minced meat, soy sauce and chopped spring onion, and a separate small bowl of white rice, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 901, "name": "Chicken breast salad wrap", "desc": "Chicken breast salad wrap", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 902, "name": "Vietnamese prawn rice paper rolls", "desc": "Vietnamese prawn rice paper rolls (1 serving), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 903, "name": "Tuna and egg salad wrap", "desc": "Tuna and egg salad wrap (1 wrap)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 904, "name": "Miso salmon and tofu soup with rice", "desc": "Japanese Miso salmon and tofu soup with rice", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 905, "name": "Spinach and egg drop soup", "desc": "spinach and egg drop soup: one bowl of clear light golden broth filled with dark green spinach leaves and soft wispy yellow and white ribbons of egg swirled through the soup, a few small silver ikan bilis and red goji berries floating, only the soup in a bowl, no rice, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 906, "name": "Miso soup with tofu, rice and egg", "desc": "Japanese Miso soup with tofu, rice and egg (1 set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 907, "name": "Egg white omelette with spinach and mushrooms", "desc": "Egg white omelette with spinach and mushrooms (1 serving)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 908, "name": "Chicken and vegetable pasta", "desc": "Chicken and vegetable pasta (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 909, "name": "Baked chicken thigh with potatoes and salad", "desc": "Baked chicken thigh with potatoes and salad (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 910, "name": "Sardine sandwich", "desc": "Malaysian Sardine sandwich (2 slices wholemeal), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 911, "name": "Banana", "desc": "Banana", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 912, "name": "Papaya", "desc": "papaya: three long boat-shaped papaya slices lying side by side on a small plate, bright orange flesh with a thin green-yellow skin on the curved side, the seeds scooped out to leave a smooth hollow, plain fruit with no lime, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 913, "name": "Apple", "desc": "apple: one whole round red apple with a short brown stem and one green leaf, glossy red skin with a yellow blush and a bright white highlight, standing next to one apple wedge that shows cream-white flesh and two dark brown seeds, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 914, "name": "Guava", "desc": "guava: one whole round pale green guava with slightly bumpy skin, beside one guava half cut open to show white flesh and a centre packed with many small pale seeds, fresh light green colours, plain with no powder, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 915, "name": "Watermelon", "desc": "watermelon: two triangular watermelon slices leaning on each other on a small plate, bright red juicy flesh with a few black seeds, a thin white band and a dark green striped rind along the curved bottom edge, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 916, "name": "Orange", "desc": "orange: one whole round orange with dimpled bright orange skin and a small green stem end, beside one orange half cut across to show the wheel of juicy segments with thin white pith lines, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 917, "name": "Mixed cut fruit", "desc": "mixed cut fruit: a small plate of mixed cut fruit in five neat groups: three red watermelon cubes, three orange papaya cubes, three yellow pineapple chunks, three pale green honeydew cubes and three white guava slices, one wooden pick standing in a piece, everything inside the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 918, "name": "Boiled egg", "desc": "Boiled egg (1 egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 919, "name": "Boiled eggs", "desc": "Boiled eggs (2 eggs)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 920, "name": "Blanched greens with oyster sauce", "desc": "blanched greens with oyster sauce: a neat row of long blanched choy sum stalks on an oval plate, bright green leaves and pale green stems all lying in the same direction, dark glossy brown oyster sauce poured across the middle and golden fried shallots sprinkled on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 921, "name": "Ulam with sambal on the side", "desc": "ulam with sambal: a plate of mixed raw Malay herbs and vegetables arranged in neat piles: a bunch of feathery green ulam raja leaves, round cucumber slices, raw green long beans cut in finger lengths, sliced star-shaped green winged beans and a wedge of pale cabbage, with a small dish of dark red sambal belacan in the middle, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 922, "name": "Garden salad", "desc": "Garden salad (1 bowl, light dressing)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 923, "name": "Corn cup", "desc": "Corn cup (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 924, "name": "Edamame", "desc": "Japanese Edamame", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 925, "name": "Chawanmushi", "desc": "Japanese Chawanmushi (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 926, "name": "Miso soup", "desc": "miso soup: a small black lacquer bowl with a red inside, filled with cloudy light brown miso soup, small white tofu cubes, dark green wakame seaweed pieces and green spring onion rings floating on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 927, "name": "Unsweetened soy milk", "desc": "a glass of soy milk, with unsweetened", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 928, "name": "Plain yogurt", "desc": "Plain yogurt", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 929, "name": "Greek yogurt", "desc": "Greek yogurt", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 930, "name": "Low-fat milk", "desc": "one light blue and white milk carton with a gable top, and a tall glass of creamy warm white milk, nothing else, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 931, "name": "Mixed nuts", "desc": "Mixed nuts", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 932, "name": "Tau fu fah", "desc": "tau fu fah: a bowl of silky smooth white soy bean curd pudding scooped in thin soft layers, with dark brown syrup the colour of soy sauce, a deep brown gula melaka palm sugar syrup, pooled around and over the white curd, a porcelain spoon lifting one wobbly white slice, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 933, "name": "Steamed corn", "desc": "Steamed corn (1 cob)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 934, "name": "Popiah, 1 roll", "desc": "fresh popiah: one soft thin pale wrapper roll cut in half, showing a filling of shredded turnip, carrot, bean sprouts, egg strips and lettuce, with a streak of dark sweet sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 936, "name": "Apam balik", "desc": "apam balik: thick folded peanut pancake with crushed peanuts and sweet corn filling (1 thick slice), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 937, "name": "Ais kacang (ABC)", "desc": "ais kacang: a tall mound of shaved ice with red rose syrup and brown gula melaka, lots of red beans, black grass jelly cubes, green cendol jelly strands, sweet corn and a drizzle of evaporated milk, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 938, "name": "Oden", "desc": "Japanese Oden (4 pieces with soup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 939, "name": "Bun, kaya or red bean", "desc": "green kaya coconut jam, with bun, or red bean (1 bun), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 940, "name": "Chicken nuggets", "desc": "Chicken nuggets (6 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 941, "name": "Keropok lekor", "desc": "keropok lekor: a pile of long thin solid fried fish cracker sticks, finger length, uneven and bumpy, golden-brown to dark brown with a rough fried surface, no filling, with a small dish of red chilli sauce for dipping, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 942, "name": "Rojak buah", "desc": "rojak buah: a plate of cut fruit chunks, cucumber, pineapple, green mango, jicama and fried dough pieces, coated in thick dark brown prawn paste sauce and sprinkled with crushed peanuts, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 943, "name": "Teh O ais kurang manis", "desc": "a cup of black tea, with ais kurang manis (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 944, "name": "Teh C kosong", "desc": "a cup of Malaysian milk tea with evaporated milk, with kosong (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 945, "name": "Kopi (condensed milk)", "desc": "kopi: one white ceramic kopitiam coffee cup with a small green flower pattern, sitting on a matching saucer, seen from a three-quarter angle, filled with opaque medium caramel-brown milky coffee, the colour of milk chocolate, a lighter creamy swirl on the surface, two soft curls of steam above, a small metal spoon resting on the saucer, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 946, "name": "Kopi O kosong", "desc": "a kopitiam cup of strong Malaysian coffee, with o kosong (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 948, "name": "Milo kosong", "desc": "a glass of chocolate malt drink, with kosong (1 cup, no condensed milk)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 949, "name": "Limau ais kurang manis", "desc": "limau ais kurang manis: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with pale cloudy yellow-green lime juice, slightly see-through, five clear ice cubes, three green calamansi lime halves floating inside the glass, and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 950, "name": "Sirap bandung", "desc": "sirap bandung: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with opaque bright pastel pink rose milk drink, evenly pink like strawberry milk from top to bottom, five clear ice cubes and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 951, "name": "Milk tea, 0% sugar, no pearls", "desc": "a plastic cup of milk tea; with black tapioca pearls at the bottom and a thick straw, with 0% sugar, no", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 952, "name": "Jasmine green milk tea, less sugar", "desc": "a plastic cup of milk tea, with jasmine green , less sugar", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 953, "name": "Americano", "desc": "Americano", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 954, "name": "Latte", "desc": "Latte", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 955, "name": "Spanish latte", "desc": "Spanish latte", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 956, "name": "Fresh coconut water", "desc": "a fresh green coconut with a straw, with fresh (1 coconut)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 957, "name": "100 Plus", "desc": "one white and blue aluminium drink can, beside a glass of clear bubbly sparkling drink with ice cubes, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 958, "name": "Brown rice", "desc": "Brown rice (1 cup cooked)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 959, "name": "Fried chicken", "desc": "Fried chicken (1 piece (thigh))", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 960, "name": "Grilled chicken breast", "desc": "Grilled chicken breast", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 961, "name": "Tempeh, fried", "desc": "fried tempeh: thin rectangular slices of golden-brown fried tempeh showing the soybeans pressed together, piled on a plate with sliced red chilli, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 962, "name": "Tofu, braised", "desc": "braised tofu: thick rectangles of tofu with brown fried skin in glossy dark brown soy gravy with shiitake mushrooms and spring onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 963, "name": "Siew mai", "desc": "three open-topped siew mai pork and shrimp dumplings with orange roe in a bamboo steamer (3 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 964, "name": "Dhal curry", "desc": "small bowl of yellow lentil dhal curry, with curry, bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 965, "name": "Nasi lemak ayam goreng (Village Park)", "desc": "Malaysian spiced fried chicken piece, crispy and dark golden; Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg (1 plate with a fried chicken leg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 966, "name": "Lontong (Village Park)", "desc": "lontong: rice cakes in yellow coconut vegetable gravy with egg and sambal (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 967, "name": "Economy rice (Home Kitchen)", "desc": "Malaysian economy rice: white rice with a piece of meat and two heaps of stir-fried vegetables (Rice, 1 meat, 2 veg), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 968, "name": "Chicken chop (Steven's Corner)", "desc": "Malaysian chicken chop: a breaded fried chicken cutlet with brown or black pepper sauce, fries and coleslaw (Deep-fried, with fries, coleslaw and egg)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 969, "name": "Banana leaf rice (Kanna Curry House)", "desc": "South Indian banana leaf rice: white rice on a banana leaf with three small heaps of vegetable curries, a papadam and curry gravy, with rice (Rice, 3 vegetables, chicken curry), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 970, "name": "Roasted chicken rice (Choon Yien)", "desc": "sliced boneless roasted chicken with glossy golden-brown skin laid in strips over cucumber, a dome of rice beside it and a small dish of chilli sauce (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 971, "name": "Fried chicken with curry rice (Lim Fried Chicken)", "desc": "Malaysian Fried chicken with curry rice (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 972, "name": "Hainan chicken rice (Hometown Hainan Coffee)", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce, with hainan (1 set), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 973, "name": "Nasi lemak ayam goreng (Nasi Lemak Bumbung)", "desc": "Malaysian spiced fried chicken piece, crispy and dark golden; Malaysian nasi lemak: a dome of white coconut rice, a spoon of dark red sambal, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg (1 plate with a fried egg)", "neg": "whole roast chicken", "n": 1}, {"id": 974, "name": "Wantan mee (PJ Old Town)", "desc": "Malaysian wantan mee: thin springy egg noodles tossed in dark soy sauce with slices of red-edged char siu, choy sum and small boiled wontons (1 plate, dry)", "neg": "whole roast chicken, fried egg, sunny side up egg, triangle, pointed, samosa", "n": 1}, {"id": 975, "name": "Yong tau foo (PJ Old Town)", "desc": "Malaysian yong tau foo: stuffed tofu, fish balls, stuffed chillies and stuffed bitter gourd in clear soup (6 pieces in soup), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 976, "name": "Chee cheong fun (SS2 morning market)", "desc": "plain rolled white rice noodle rolls drizzled with sweet brown sauce and sesame seeds (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 977, "name": "Claypot chicken rice (Uncle Mani's)", "desc": "a black clay pot of rice topped with chicken pieces, Chinese sausage slices and spring onion (1 pot, chicken satti soru), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 978, "name": "Hainanese chicken chop (Yut Kee)", "desc": "Malaysian chicken chop: a breaded fried chicken cutlet with brown or black pepper sauce, fries and coleslaw, with hainanese (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 979, "name": "Hailam mee (Yut Kee)", "desc": "Hainanese fried noodles in dark sauce with pork and greens (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 980, "name": "Beef noodles (Soong Kee)", "desc": "braised beef noodle soup: dark brown broth, thick noodles, braised beef chunks and bok choy (1 bowl, dry noodles with soup), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 981, "name": "Fish porridge (Hon Kee)", "desc": "smooth Cantonese rice porridge, with fish (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 982, "name": "Banana leaf rice (Sri Nirwana Maju)", "desc": "South Indian banana leaf rice: white rice on a banana leaf with three small heaps of vegetable curries, a papadam and curry gravy, with rice (Rice, 3 vegetables, fried fish), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 983, "name": "Nasi bojari (Madam Kwan's)", "desc": "Malaysian Nasi bojari (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 984, "name": "Poached chicken rice (Chicken Rice Hub)", "desc": "sliced boneless poached chicken with smooth pale skin over cucumber slices with soy sauce dressing, a dome of rice and a small dish of chilli sauce, with poached (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 985, "name": "Karaage don (Hungry Don)", "desc": "Japanese Karaage don (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 986, "name": "Food court meal (Rock Cafe)", "desc": "Chinese Food court meal (Noodle soup, or economy rice with 2 veg), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 987, "name": "Burger (myBurgerLab)", "desc": "Burger (1 single burger, no fries)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 988, "name": "Japanese set (Hana Dining)", "desc": "Japanese set (Sashimi or grilled fish set)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 989, "name": "Spring rolls, fried", "desc": "golden crispy fried spring rolls, with s, fried (3 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 990, "name": "Chicken curry with rice", "desc": "Malaysian chicken curry with potatoes in orange gravy, with rice (1 plate), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 992, "name": "French toast", "desc": "french toast: two thick slices of bread dipped in egg and pan-fried to golden yellow with browned patches, stacked on a small plate, one square of butter on top and golden syrup drizzled over and running down the side onto the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 993, "name": "Club sandwich", "desc": "Club sandwich (1 sandwich)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 994, "name": "Tiramisu", "desc": "Tiramisu (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 995, "name": "Lasagna", "desc": "Lasagna (1 portion)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 996, "name": "Macaroni and cheese", "desc": "Macaroni and cheese (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 997, "name": "Samosa", "desc": "Indian Samosa (1 piece), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 998, "name": "Cupcake", "desc": "Cupcake (1 cupcake)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 999, "name": "Frozen yogurt", "desc": "Frozen yogurt", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1000, "name": "Chocolate sundae", "desc": "Chocolate sundae (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1001, "name": "Tacos", "desc": "Tacos (2 tacos)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1002, "name": "Macarons", "desc": "Macarons (3 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1003, "name": "Fried calamari", "desc": "Fried calamari (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1004, "name": "Eggs Benedict", "desc": "Eggs Benedict (2 eggs on a muffin)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1005, "name": "Masala thosai", "desc": "a thin crispy golden dosa folded on a plate with coconut chutney and sambar, with masala (1 piece with chutney), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1006, "name": "Yakult Ace", "desc": "one tiny chubby plastic cultured milk bottle, short and round, the size of a thumb, with a pinched narrow waist in the middle like an hourglass, a shiny red foil lid on top, the drink inside is creamy warm white, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1007, "name": "Yakult Ace Light", "desc": "one tiny chubby plastic cultured milk bottle, short and round, the size of a thumb, with a pinched narrow waist in the middle like an hourglass, a shiny light blue foil lid on top, the drink inside is creamy warm white, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1008, "name": "Vitagen", "desc": "one small plump plastic cultured milk bottle with a rounded body and a shiny orange foil lid, the drink inside is light creamy orange, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1009, "name": "Vitagen Less Sugar", "desc": "one small plump plastic cultured milk bottle with a rounded body and a shiny bright green foil lid, the drink inside is creamy warm white, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1010, "name": "Milo, packet drink", "desc": "one small bright green rectangular drink box with a bent straw poked into the top, beside a short glass of chocolate brown malt drink, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1011, "name": "Chocolate milk, packet", "desc": "one small brown rectangular drink box with a bent straw poked into the top, beside a short glass of chocolate milk, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1012, "name": "Whey protein shake, with water", "desc": "a glass of milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1013, "name": "Whey protein shake, with low-fat milk", "desc": "a glass of milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1014, "name": "Gold Standard Whey shake, with water", "desc": "a glass of milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1015, "name": "Herbalife Formula 1 shake", "desc": "a glass of milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1016, "name": "Ensure Gold", "desc": "one tall glass of creamy pale vanilla milk drink with a straw, nothing else, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1017, "name": "Anlene Gold 5X", "desc": "one tall glass of creamy pale milk drink with a striped straw, nothing else, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1018, "name": "Dutch Lady +Protein milk", "desc": "one blue and white plastic milk bottle with a blue cap, and a tall glass of creamy warm white milk, nothing else, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1019, "name": "Barebells Protein Milkshake", "desc": "one tall plastic bottle of light brown chocolate milkshake with a black cap, the brown drink visible, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1020, "name": "Soy milk, sweetened packet", "desc": "one small cream coloured rectangular drink box with a bent straw poked into the top, beside a short glass of pale beige soy milk and a few soybeans, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1021, "name": "Yogurt drink", "desc": "one small plump plastic bottle completely filled with bright pastel pink strawberry yogurt drink, the whole bottle looks pink because of the pink drink inside, with a pink cap, no logo, bright warm light, clean bright colours", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1022, "name": "Flavoured yogurt cup", "desc": "Flavoured yogurt cup", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1023, "name": "Iced lemon tea, packet", "desc": "one small bright yellow rectangular drink box with a bent straw poked into the top, beside a glass of amber iced lemon tea with ice cubes and a lemon slice, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 1024, "name": "Teh ais", "desc": "teh ais: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with opaque bright orange-brown iced milk tea, five clear ice cubes at the top, a paler creamy layer near the bottom, and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1025, "name": "Kopi ais", "desc": "kopi ais: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with opaque iced milk coffee, dark coffee brown at the top fading to creamy light brown at the bottom where the condensed milk sits, five clear ice cubes at the top and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1026, "name": "Cham ais", "desc": "cham ais: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with opaque warm reddish-brown iced coffee and tea mixed with milk, a shade between coffee brown and tea orange, five clear ice cubes, a thin froth line at the top and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1027, "name": "Neslo ais", "desc": "neslo ais: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with an opaque iced drink in two soft tones, chocolate-brown malt drink in the lower half and darker coffee brown in the upper half gently blending together, five clear ice cubes and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1028, "name": "Horlicks ais", "desc": "horlicks ais: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with opaque pale creamy beige-yellow iced malted milk drink, the colour of light custard, five clear ice cubes, a thin froth line at the top and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1029, "name": "Teh O limau ais", "desc": "teh o limau ais: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with clear see-through amber iced black tea with no milk, five clear ice cubes, four small green calamansi lime halves floating inside the glass showing their pale yellow cut faces, and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1030, "name": "Barley drink, sweetened", "desc": "barley drink, sweetened: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with cloudy pale off-white barley water, slightly see-through, with a thick layer of soft plump white pearl barley grains settled at the bottom of the glass, three clear ice cubes and a long spoon standing in the glass, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1031, "name": "Soya cincau", "desc": "soya cincau: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with creamy pale beige soy milk with many shiny black grass jelly strips and cubes clearly showing through the glass in the lower half, three clear ice cubes and a wide paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1032, "name": "Air mata kucing", "desc": "Air mata kucing (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1033, "name": "Sugarcane juice", "desc": "a cup of pale green sugarcane juice, with juice (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1034, "name": "Watermelon juice, no sugar", "desc": "watermelon juice, no sugar: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with opaque bright pink-red watermelon juice with a thin pale pink foam on top, a small triangle of watermelon with green rind and black seeds fixed on the rim of the glass, and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1035, "name": "Masala tea", "desc": "Masala tea (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1036, "name": "Brown sugar pearl fresh milk", "desc": "Brown sugar pearl fresh milk", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1037, "name": "Taro milk tea with pearls", "desc": "a plastic cup of milk tea; with black tapioca pearls at the bottom and a thick straw, with taro", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1038, "name": "Fruit tea, less sugar", "desc": "Fruit tea, less sugar", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1039, "name": "Yakult green tea", "desc": "a small bottle of cultured milk drink", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1040, "name": "Jasmine green milk tea, large, normal sugar", "desc": "a plastic cup of milk tea, with jasmine green , large, normal sugar", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1041, "name": "Da Hong Pao milk tea, less sugar", "desc": "a plastic cup of milk tea, with da hong pao , less sugar", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1042, "name": "Peach oolong milk tea, less sugar", "desc": "a plastic cup of milk tea, with peach oolong , less sugar", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1043, "name": "Matcha latte, iced", "desc": "Matcha latte, iced", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1044, "name": "Iced chocolate", "desc": "Iced chocolate", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1045, "name": "Caramel frappé", "desc": "Caramel frappé (1 tall)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1046, "name": "Vanilla milkshake", "desc": "Vanilla milkshake", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1047, "name": "All Berry Bang smoothie", "desc": "All Berry Bang smoothie", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1048, "name": "Gym Junkie protein smoothie", "desc": "Gym Junkie protein smoothie", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1049, "name": "llaollao, small tub with 1 topping", "desc": "small tub with 1 topping", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1050, "name": "llaollao, medium tub with 3 toppings", "desc": "medium tub with 3 toppings", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1051, "name": "llaollao Sanum", "desc": "Sanum (1 cup with fruit, crunch and sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1052, "name": "Dubai chocolate strawberry cup", "desc": "Dubai chocolate strawberry cup (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1053, "name": "Inside Scoop, single scoop", "desc": "single scoop (1 scoop in a cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1054, "name": "Ice cream waffle", "desc": "Ice cream waffle (1 waffle, 1 scoop and sauce)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1055, "name": "Soufflé pancakes", "desc": "soufflé pancakes: a stack of three very thick tall fluffy round pancakes on a plate, pale golden tops and soft pale yellow wobbly sides, a cloud of whipped cream on top, a dusting of white icing sugar, two red strawberries and a little syrup on the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1056, "name": "Mango bingsu", "desc": "Korean Mango bingsu", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1057, "name": "Chocolate bingsu", "desc": "Korean Chocolate bingsu", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1058, "name": "Mango sago pomelo", "desc": "mango sago pomelo: a bowl of bright orange-yellow mango cream with small translucent sago pearls, cubes of fresh mango and pale pomelo pulp sacs on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1059, "name": "Red bean soup", "desc": "Chinese Red bean soup (1 bowl), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1060, "name": "Bubur cha cha", "desc": "bubur cha cha: coconut milk dessert with sweet potato and taro cubes (1 bowl), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1061, "name": "Sago gula melaka", "desc": "white sago pudding with dark palm sugar syrup and coconut milk (1 bowl), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1062, "name": "Mango sticky rice", "desc": "Thai Mango sticky rice (1 plate)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1063, "name": "Muah chee", "desc": "muah chee: chewy glutinous rice dough pieces coated in crushed peanuts and sugar, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1064, "name": "Mochi", "desc": "Japanese Mochi (2 pieces)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1065, "name": "Kaya puff", "desc": "kaya puff: two half-moon shaped baked pastry puffs shaped like curry puffs, golden-brown flaky layered pastry with a crimped pleated rope edge along the curved side, one puff broken open with thick kaya coconut jam in a warm yellowish olive-green colour like pandan custard, more yellow than green and not bright green oozing out of the middle, on a small plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1066, "name": "Polo bun with butter", "desc": "Hong Kong pineapple bun with crackly golden top and a slab of butter, with butter (1 bun), bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1067, "name": "Chicken floss bun", "desc": "chicken floss bun: a soft round bun coated all over with fluffy golden-brown chicken floss, with a line of mayonnaise showing, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1068, "name": "Brownie", "desc": "Brownie (1 square)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1069, "name": "Churros with chocolate dip", "desc": "churros with chocolate dip: five long ridged golden-brown fried dough sticks dusted with cinnamon sugar, stacked on a small plate, with one small round white dipping bowl of thick dark melted chocolate on the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1070, "name": "Crêpe with banana and chocolate", "desc": "Crêpe with banana and chocolate (1 crêpe)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1071, "name": "Molten lava cake with ice cream", "desc": "Molten lava cake with ice cream (1 cake, 1 scoop)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1072, "name": "McFlurry with Oreo", "desc": "chocolate sandwich cookies with white cream", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1073, "name": "Ais krim potong", "desc": "ais krim potong: two rectangular ice cream bars on wooden sticks, one dark red with red beans and one pale yellow durian, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1074, "name": "Chocolate-coated ice cream stick", "desc": "Chocolate-coated ice cream stick (1 stick)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1075, "name": "Ice cream cone, packaged", "desc": "Ice cream cone, packaged (1 cone)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1076, "name": "Corn snack pack", "desc": "crunchy corn snacks in a bowl", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1077, "name": "Seaweed snack", "desc": "crispy seaweed snack sheets", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1078, "name": "Granola bar", "desc": "Granola bar (1 bar)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1079, "name": "Popcorn, salted", "desc": "Popcorn, salted", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1080, "name": "Siew mai", "desc": "siew mai: four open-topped dumplings in a bamboo steamer, each a short cylinder with a thin yellow wrapper, pink pork and prawn filling showing at the open top, and a small orange dot of roe in the centre, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1081, "name": "Steamed fish balls", "desc": "white round fish balls, with steamed s (3 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1082, "name": "Steamed pork ribs with black bean", "desc": "dim sum steamed pork ribs: small bite-sized pieces of pale steamed pork spare ribs with bone, glossy, with black fermented beans, chopped red chilli and garlic, in a small round dish inside a bamboo steamer, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1083, "name": "Chicken feet in black bean sauce", "desc": "dim sum phoenix claws: four braised chicken feet, puffy wrinkled skin in glossy dark red-brown sauce with black beans and red chilli slices, in a small dish inside a bamboo steamer, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1084, "name": "Beancurd skin roll, fried", "desc": "fried beancurd skin rolls: three crispy golden-brown deep-fried rolls with a blistered wrinkled tofu skin surface, one cut to show prawn filling, with a small dish of mayonnaise, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1085, "name": "Prawn cheong fun", "desc": "rolled smooth white rice noodle rolls drizzled with sweet soy sauce and sesame, with prawn (1 plate, 3 rolls), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1086, "name": "Char siu cheong fun", "desc": "rolled smooth white rice noodle rolls drizzled with sweet soy sauce and sesame; slices of red-edged glossy char siu barbecued pork (1 plate, 3 rolls), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1087, "name": "Lo bak go, pan-fried radish cake", "desc": "lo bak go: four thick rectangular slices of white radish cake, pan-fried until the flat faces are dark golden-brown and crispy, with small bits of red Chinese sausage and dried shrimp showing in the white cake, chopped spring onion on top, a small dish of chilli sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1088, "name": "Yam puff (wu kok)", "desc": "golden lacy crispy deep-fried yam puffs (2 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1089, "name": "Ham sui kok", "desc": "ham sui kok: three golden-brown deep-fried glutinous rice dumplings shaped like small footballs with a smooth blistered crisp skin, one cut in half to show the chewy white dough wall and a savoury filling of minced pork, mushroom and spring onion, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1090, "name": "Fried prawn dumplings with mayo", "desc": "fried prawn dumplings: five golden-brown crispy deep-fried dumplings folded like half-moons with pleated edges, on a plate with a small dish of white mayonnaise for dipping, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1092, "name": "Sui kow in soup", "desc": "large sui kow prawn dumplings in clear soup, with in soup (4 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1093, "name": "Sheng jian bao", "desc": "pan-fried pork buns with crispy golden bottoms and sesame (3 pieces, pan-fried), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1094, "name": "Salted egg custard bun (lau sar bao)", "desc": "lau sar bao: three round smooth white steamed buns in a bamboo steamer, one torn open with bright golden-yellow molten salted egg custard flowing out, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1095, "name": "Egg custard bun (nai wong bao)", "desc": "nai wong bao: three round smooth white steamed buns in a bamboo steamer, one torn open showing thick pale-yellow firm custard filling, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1096, "name": "Lotus paste bun", "desc": "lotus paste buns: three round smooth white steamed buns in a bamboo steamer, one torn open showing smooth light golden-brown lotus seed paste, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1097, "name": "Chicken bao", "desc": "fluffy white steamed bun, with chicken (1 bun), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1098, "name": "Big bao (dai bao)", "desc": "dai bao: one very large round white steamed bun with a pleated twisted top, cut open to show chicken pieces, a wedge of boiled egg and mushroom in brown gravy, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1099, "name": "Fried mantou with condensed milk", "desc": "fried mantou: four golden-brown deep-fried mantou buns, square pillow shaped with a smooth crisp skin, on a plate with a small white dish filled to the brim with thick pale creamy-yellow condensed milk for dipping, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1100, "name": "Ma lai gao", "desc": "a slice of fluffy brown steamed sponge cake (1 slice), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1101, "name": "Sesame balls (jin deui)", "desc": "golden fried sesame balls covered in white sesame seeds (2 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1102, "name": "Char siu sou", "desc": "char siu sou: three golden-brown flaky baked puff pastries, rectangular and triangular, with a shiny egg glaze and white sesame seeds on top, one broken open to show red-brown diced barbecued pork filling, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1103, "name": "Century egg and pork porridge", "desc": "smooth Cantonese rice porridge, with century egg and pork (1 bowl), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1104, "name": "Stuffed tofu and brinjal", "desc": "yong tau foo: pieces of white tofu, purple brinjal slices, green okra and red chilli, each stuffed with pale fish paste, in a shallow plate with brown bean sauce, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1105, "name": "Bak chang", "desc": "a pyramid-shaped glutinous rice dumpling wrapped in bamboo leaves, opened to show savoury filling (1 rice dumpling), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1106, "name": "Pain au chocolat", "desc": "Pain au chocolat (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1107, "name": "Crookie", "desc": "Crookie (1 croissant with cookie dough)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1108, "name": "Chicken ham and cheese croissant", "desc": "Chicken ham and cheese croissant (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1109, "name": "Fruit Danish", "desc": "Fruit Danish (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1110, "name": "Cinnamon roll", "desc": "Cinnamon roll (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1111, "name": "Blueberry muffin", "desc": "Blueberry muffin (1 muffin)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1112, "name": "Chocolate muffin", "desc": "Chocolate muffin (1 muffin)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1113, "name": "Banana bread", "desc": "Banana bread (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1114, "name": "Cream puff", "desc": "Cream puff (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1115, "name": "Chocolate éclair", "desc": "Chocolate éclair (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1116, "name": "Swiss roll", "desc": "Swiss roll (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1117, "name": "Pandan chiffon cake", "desc": "a tall wedge slice of light fluffy bright green pandan chiffon cake with a thin brown crust on the outside, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1118, "name": "Red velvet cake", "desc": "Red velvet cake (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1119, "name": "Carrot cake with cream cheese", "desc": "Carrot cake with cream cheese (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1120, "name": "Mille crêpe", "desc": "Mille crêpe (1 slice)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1121, "name": "Portuguese egg tart", "desc": "Portuguese egg tart with caramelised burnt-spotted custard top (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1122, "name": "Mooncake, lotus paste", "desc": "a mooncake cut in quarters showing lotus paste filling, with lotus paste (1 quarter), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1123, "name": "Taiyaki", "desc": "taiyaki: one fish-shaped golden-brown waffle cake with moulded scales and a tail fin pressed into it, beside another one broken in half to show thick dark red-brown sweet red bean paste filling, on a small plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1124, "name": "Chicken sausage roll", "desc": "Chicken sausage roll (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1125, "name": "Tuna puff", "desc": "Tuna puff (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1126, "name": "Sausage bun", "desc": "Sausage bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1127, "name": "Mexican coffee bun", "desc": "Mexican coffee bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1128, "name": "Tuna mayo bun", "desc": "Tuna mayo bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1129, "name": "Korean garlic cream cheese bun", "desc": "Korean garlic cream cheese bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1130, "name": "Curry chicken bun", "desc": "Malaysian chicken curry with potatoes in orange gravy (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1131, "name": "Cheese bun", "desc": "Cheese bun (1 bun)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1132, "name": "Pretzel, cinnamon sugar", "desc": "Pretzel, cinnamon sugar (1 pretzel)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1133, "name": "White bread", "desc": "White bread (2 slices)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1134, "name": "Wholemeal bread", "desc": "Wholemeal bread (2 slices)", "neg": "fried egg, sunny side up egg", "n": 1}, {"id": 1135, "name": "Green bean soup", "desc": "green bean sweet soup: a bowl of thin pale olive-green sweet soup full of small split green mung beans, with a few translucent sago pearls, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1136, "name": "Black sesame paste", "desc": "black sesame paste: a white bowl of thick smooth glossy jet-black sweet soup with a white swirl of coconut milk on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1137, "name": "Peanut paste", "desc": "peanut paste: a white bowl of thick smooth light-brown creamy sweet soup with a few crushed peanuts on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1138, "name": "Walnut paste", "desc": "walnut paste: a white bowl of thick smooth beige-brown creamy sweet soup with one walnut half on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1139, "name": "Almond paste", "desc": "almond paste: a white bowl of thick smooth pure white creamy sweet soup with a few almond slices on top, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1140, "name": "Sweet potato ginger soup", "desc": "sweet potato ginger soup: a bowl of clear light golden-brown ginger syrup with big chunks of orange and yellow sweet potato and a slice of ginger, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1141, "name": "Tang yuan in ginger syrup", "desc": "glutinous rice balls in ginger syrup, with in ginger syrup (4 pieces), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1142, "name": "Snow fungus with longan", "desc": "snow fungus sweet soup: a bowl of clear pale golden syrup with frilly translucent white snow fungus, round longans, red dates and red goji berries, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1143, "name": "Barley, fu chuk and gingko", "desc": "barley fu chuk ginkgo sweet soup: a bowl of milky white sweet soup with many small white pearl barley grains, soft pale-yellow beancurd skin ribbons and round yellow ginkgo nuts, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1144, "name": "Bubur pulut hitam", "desc": "black glutinous rice porridge with a swirl of coconut milk (1 bowl with coconut milk), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1145, "name": "Honeydew sago", "desc": "honeydew sago: a bowl of pale green creamy coconut milk dessert with small translucent sago pearls and light green honeydew melon balls, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1146, "name": "Leng chee kang", "desc": "leng chee kang: a white bowl of clear light-brown sweet soup filled with round white lotus seeds, translucent longan fruits, frilly white snow fungus, dark red dates, and tiny black basil seeds, with ice cubes, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1147, "name": "Gui ling gao herbal jelly", "desc": "gui ling gao: a small white bowl of shiny jet-black wobbly herbal jelly, smooth like grass jelly, cut into a few large cubes, with clear golden honey syrup poured over, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1148, "name": "Taro sago", "desc": "taro sago: a bowl of pale purple creamy coconut milk dessert with small translucent sago pearls and cubes of light purple taro, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1149, "name": "Salt bread (shio pan)", "desc": "Korean Salt bread (1 roll)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1150, "name": "Kunafa", "desc": "kunafa: a round slice of golden-orange crispy shredded pastry with stretchy white cheese inside, topped with crushed green pistachios and shiny syrup, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1151, "name": "Cromboloni", "desc": "Cromboloni (1 filled round croissant)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1152, "name": "Mochi cheesecake", "desc": "Mochi cheesecake (1 piece)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1153, "name": "Tanghulu", "desc": "tanghulu: two wooden skewers of whole red strawberries and green grapes coated in a hard shiny clear sugar glaze, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1154, "name": "Jagung cheese", "desc": "jagung cheese: a clear plastic cup filled with bright yellow sweet corn kernels, topped with a big heap of shredded cheddar cheese and a drizzle of white condensed milk, with a small plastic spoon, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 1155, "name": "Fruit sando", "desc": "Japanese Fruit sando (1 sandwich)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1156, "name": "Musang King durian", "desc": "Musang King durian", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1157, "name": "Durian cendol", "desc": "Malaysian cendol: shaved ice with green rice-flour jelly strands, red beans, coconut milk and dark palm sugar syrup, with durian (1 bowl)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1158, "name": "Pandan gula melaka cake", "desc": "green pandan, with gula melaka cake (1 slice), bright warm light, clean bright colours", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1159, "name": "Garrett popcorn, Chicago mix", "desc": "Chicago mix", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1160, "name": "Mixue lemonade", "desc": "lemonade", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1161, "name": "Brown sugar boba milk (Tiger Sugar)", "desc": "with black tapioca pearls at the bottom and a thick straw, with brown sugar milk (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 1162, "name": "Strawberry matcha latte", "desc": "Strawberry matcha latte", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1163, "name": "Pistachio latte", "desc": "Pistachio latte", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1164, "name": "Einspänner cream coffee", "desc": "Einspänner cream coffee (1 cup)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1165, "name": "Coconut shake", "desc": "Malaysian shake (1 cup with ice cream)", "neg": "whole roast chicken, fried egg, sunny side up egg, food, plate", "n": 1}, {"id": 1166, "name": "Soda limau asam boi", "desc": "soda limau asam boi: one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with clear fizzy very pale yellow lime soda with many tiny bubbles rising, four clear ice cubes, two green lime halves floating, and two wrinkled dark red-brown dried sour plums sitting at the bottom of the glass, a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 1167, "name": "Air katira", "desc": "Malaysian Air katira (1 glass)", "neg": "whole roast chicken, fried egg, sunny side up egg", "n": 1}, {"id": 2001, "name": "Pizza, margherita", "desc": "a whole round margherita pizza cut into eight slices. A puffy golden-brown baked crust with darker toasted spots, red tomato sauce, many round pools of melted white mozzarella with browned patches, and plenty of fresh green basil leaves", "neg": "", "n": 1}, {"id": 2002, "name": "Pizza, pepperoni", "desc": "a whole round pepperoni pizza cut into eight slices. A puffy golden-brown baked crust with darker toasted spots, melted bubbly cheese with browned patches, fully covered edge to edge with many round red pepperoni slices that are slightly curled and crisp at the edges", "neg": "", "n": 1}, {"id": 2003, "name": "Pizza, Hawaiian chicken", "desc": "a whole round Hawaiian pizza cut into eight slices. A puffy golden-brown baked crust with darker toasted spots, melted bubbly cheese with browned patches, generously covered with many yellow pineapple chunks and pink pieces of chicken ham", "neg": "", "n": 1}, {"id": 2004, "name": "Pizza, meat lovers", "desc": "a whole round meat lovers pizza cut into eight slices. A puffy golden-brown baked crust, melted bubbly cheese, heaped with pepperoni slices, brown minced beef crumbles, sliced sausage and pieces of chicken", "neg": "", "n": 1}, {"id": 2005, "name": "Pizza, veggie", "desc": "a whole round vegetable pizza cut into eight slices. A puffy golden-brown baked crust, melted bubbly cheese, generously covered with green capsicum rings, sliced mushrooms, black olives, red onion rings, sweetcorn and tomato slices", "neg": "", "n": 1}, {"id": 2006, "name": "Pizza, BBQ chicken", "desc": "a whole round BBQ chicken pizza cut into eight slices. A puffy golden-brown baked crust, melted bubbly cheese, generously covered with chunks of grilled chicken, red onion rings and green capsicum, with dark brown barbecue sauce drizzled in zigzags", "neg": "", "n": 1}, {"id": 2007, "name": "Curry puff, mini", "desc": "two small bite-size Malaysian curry puffs on a small plate: half-moon shaped flaky golden pastry with a crimped twisted spiral edge along the curved side, one broken open showing yellow curry potato filling", "neg": "", "n": 1}, {"id": 3001, "name": "Pork noodle soup", "desc": "a bowl of Malaysian pork noodle soup. Thick round white rice noodles in cloudy light brown pork broth, with pinkish crumbly minced pork, a few slices of dark brown pork liver, pieces of pork intestine, lean pork slices and bright green choy sum and dark green sweet potato leaves", "neg": "", "n": 1}, {"id": 3002, "name": "Pork noodle soup, dry", "desc": "a plate of dry pork noodles: noodles soaked in thick glossy very dark brown soy sauce so the noodles are dark brown like Hokkien mee, sitting in a pool of dark sauce at the bottom of the plate, topped with minced pork, pork slices, liver slices, crispy pork lard bits and a little green vegetable, with a small separate bowl of clear pork soup beside it", "neg": "", "n": 1}, {"id": 3003, "name": "Wantan mee, soup", "desc": "a bowl of Malaysian wantan mee soup. Thin springy yellow egg noodles in clear light broth, with slices of glossy red-edged char siu roast pork on top, four prawn dumplings with thin translucent wrappers showing a pink whole prawn inside, and green choy sum", "neg": "", "n": 1}, {"id": 3004, "name": "Beef ball noodle soup", "desc": "a bowl of beef ball noodle soup. Flat white rice noodles in clear beef broth with six round brown beef balls, thin slices of beef, bean sprouts, chopped spring onion and coriander", "neg": "", "n": 1}, {"id": 3005, "name": "Beef ball noodles, dry", "desc": "a bowl of dry beef noodles: noodles coated in dark sauce topped with brown minced beef and chopped spring onion, with a small separate bowl of clear soup holding round beef balls", "neg": "", "n": 1}, {"id": 3006, "name": "Beef noodle soup", "desc": "a big bowl of beef noodle soup. Wheat noodles in rich dark brown beef broth with large chunks of braised beef, bok choy and chopped spring onion", "neg": "", "n": 1}, {"id": 3007, "name": "Fish ball noodle soup", "desc": "a bowl of fish ball noodle soup. Thin yellow noodles in clear broth with round white fish balls, slices of fish cake with a pink edge, lettuce and fried shallots", "neg": "", "n": 1}, {"id": 3008, "name": "Fish head noodle, milky soup", "desc": "a bowl of fish head noodle soup. Thick white rice noodles in creamy milky white fish broth with golden fried fish pieces, a few soft white tofu slices, tomato wedges, salted mustard greens and chopped spring onion", "neg": "", "n": 1}, {"id": 3009, "name": "Fish slice noodle soup", "desc": "a bowl of sliced fish noodle soup. White rice noodles in clear broth with soft white slices of poached fish, tomato wedges, ginger strips and green vegetables", "neg": "", "n": 1}, {"id": 3010, "name": "Kway teow soup", "desc": "a bowl of kway teow soup. Soft bright white flat wide rice noodles in clear golden-yellow broth with white fish balls, pink pork slices, fresh bean sprouts, bright green chopped spring onion and golden fried garlic, bright fresh colours with no grey", "neg": "", "n": 1}, {"id": 3011, "name": "Duck noodle soup", "desc": "a bowl of duck noodle soup. Yellow noodles in dark herbal broth with slices of braised duck with brown skin, green vegetables and coriander", "neg": "", "n": 1}, {"id": 3012, "name": "Duck noodles, dry", "desc": "a plate of dry duck noodles: yellow noodles in dark glossy sauce topped with slices of braised duck with brown skin and cucumber, with a small bowl of soup beside it", "neg": "", "n": 1}, {"id": 3013, "name": "Chicken noodle soup", "desc": "a bowl of chicken noodle soup. Thin yellow noodles in clear golden chicken broth with shredded white chicken, green choy sum and fried shallots", "neg": "", "n": 1}, {"id": 3014, "name": "Chicken noodles, dry", "desc": "a plate of dry chicken noodles: noodles tossed in light soy and sesame oil, topped with neat slices of pale poached chicken and choy sum, with a small bowl of soup beside it", "neg": "", "n": 1}, {"id": 3015, "name": "Mee sua soup with chicken", "desc": "a bowl of Foochow mee sua: very fine thin white wheat vermicelli in reddish red wine chicken soup with chicken pieces, a hard boiled egg and shredded black fungus", "neg": "", "n": 1}, {"id": 3016, "name": "Sang har mee", "desc": "a plate of sang har mee: a nest of golden crispy fried egg noodles under glossy pale yellow egg gravy, topped with ONE single huge freshwater prawn, the size of a hand, cut in half lengthwise with its long blue-orange claws and orange roe showing, with green vegetables. No small prawns", "neg": "", "n": 1}, {"id": 3017, "name": "Cantonese fried noodles (yee mee)", "desc": "a plate of Cantonese fried yee mee: a nest of golden crispy fried egg noodles sitting on top of a pool of glossy pale yellow egg gravy at the bottom of the plate, the gravy underneath the noodles not poured over them, with sliced pork, prawns, fish cake and green choy sum in the gravy", "neg": "", "n": 1}, {"id": 3018, "name": "Curry mee", "desc": "a bowl of Malaysian curry mee. Yellow noodles and rice vermicelli in orange-red coconut curry broth with tofu puffs, chicken pieces, long beans, cockles, a dollop of red sambal and a slick of red chilli oil on top", "neg": "", "n": 1}, {"id": 3019, "name": "Mee soto", "desc": "a bowl of mee soto. Yellow noodles in clear yellowish spiced chicken soup with shredded chicken, bean sprouts, fried shallots, chopped celery and a small dish of sambal kicap", "neg": "", "n": 1}, {"id": 3020, "name": "Mee hoon kueh soup", "desc": "a bowl of mee hoon kueh soup full of thick chewy hand-torn dough pieces: irregular pieces of pale cream flour dough the size of a coin, thick and soft, curled and folded with wavy edges like torn pasta dumplings, in clear golden anchovy broth with minced pork, dark green sweet potato leaves, sliced mushrooms and golden crispy fried anchovies on top, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 3021, "name": "Mee jawa", "desc": "a plate of mee jawa: yellow noodles under thick orange sweet potato and prawn gravy, topped with fried tofu, half a boiled egg, bean sprouts, prawn fritter pieces and a lime", "neg": "", "n": 1}, {"id": 3022, "name": "Claypot loh shu fun", "desc": "a black claypot of loh shu fun: short fat white rice noodles in dark soy sauce with minced pork, a raw egg yolk on top, chopped spring onion", "neg": "", "n": 1}, {"id": 3023, "name": "Kolo mee / Kon lo mee", "desc": "a bowl of kolo mee: curly springy thin egg noodles tossed dry, topped with a big generous heap of brown minced pork covering half the noodles, a few slices of red-edged char siu roast pork and chopped spring onion, bright warm light, clean bright colours", "neg": "", "n": 1}, {"id": 3024, "name": "Kampua mee", "desc": "a plate of Sibu kampua mee: pale straight thin springy noodles tossed in shallot oil, topped with a few slices of red-edged char siu roast pork and chopped spring onion, plain and simple", "neg": "", "n": 1}, {"id": 3025, "name": "Hakka mee", "desc": "a bowl of Hakka mee: pale yellow flat egg noodles topped with a big heap of brown minced pork sauce and chopped spring onion, with a little green lettuce on the side", "neg": "", "n": 1}, {"id": 3026, "name": "Bak chor mee", "desc": "a bowl of bak chor mee: flat yellow egg noodles tossed with dark vinegar chilli sauce, topped with minced pork, thin pork slices, pork liver slices, braised shiitake mushroom slices, white fish balls and chopped spring onion", "neg": "", "n": 1}, {"id": 3027, "name": "Mee kicap", "desc": "a plate of Malaysian mee kicap: every noodle is dark brown, the thick noodles completely soaked and coloured dark brown by glossy sweet soy sauce like the colour of dark soy, no yellow noodles showing, with pieces of dark braised chicken, bright green choy sum and sliced red chillies, bright warm light, clean bright colours", "neg": "", "n": 1}, {"id": 3028, "name": "Kway teow goreng", "desc": "a plate of kway teow goreng: very wide flat ribbon rice noodles, each noodle a broad flat strip like a ribbon, fried with dark soy sauce and chilli paste, with egg, bean sprouts, chives, prawns and fish cake slices, bright warm light, clean bright colours with no grey shading", "neg": "", "n": 1}, {"id": 3029, "name": "Bee hoon goreng", "desc": "a plate of bee hoon goreng: very very thin rice vermicelli, thin as thread, fried light golden brown with soy sauce, tossed with egg, cabbage, carrot strips, bean sprouts, small prawns and chopped spring onion, bright warm colours", "neg": "", "n": 1}, {"id": 3030, "name": "Bee hoon soup", "desc": "a bowl of bee hoon soup: very thin rice vermicelli, thin as thread, in clear golden broth with white fish balls, pink fish cake slices, green leafy vegetables and golden fried shallots, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 3031, "name": "Zha jiang mian", "desc": "a bowl of jjajangmyeon: thick white wheat noodles covered with a thick glossy very very dark brown, almost black, black bean sauce with diced pork, onion and zucchini, with a few fresh julienned cucumber strips on top", "neg": "", "n": 1}, {"id": 3032, "name": "Dan dan noodles", "desc": "a bowl of dan dan noodles: wheat noodles in a pool of red chilli oil and sesame sauce, topped with crispy minced pork, crushed peanuts and bok choy", "neg": "", "n": 1}, {"id": 3033, "name": "Mala noodles", "desc": "a bowl of mala noodles: noodles in bright red spicy oily broth with dried red chillies and Sichuan peppercorns, sliced beef, tofu skin and green vegetables", "neg": "", "n": 1}, {"id": 3034, "name": "Glass noodle stir-fry", "desc": "a plate of stir-fried glass noodles: clear glossy mung bean noodles stir-fried with egg, cabbage, carrot strips and spring onion", "neg": "", "n": 1}, {"id": 3035, "name": "Tomato egg noodle soup", "desc": "a bowl of tomato egg noodle soup: noodles in bright red-orange tomato broth with soft fluffy yellow scrambled egg curds in big soft folds, chunks of red tomato and chopped green spring onion", "neg": "", "n": 1}, {"id": 4001, "name": "White rice (economy rice)", "desc": "a small white bowl of fluffy loose white steamed rice, every grain visible, the rice is warm creamy white and bright, lit by warm light, clean bright colours with no grey shading", "neg": "", "n": 1}, {"id": 4002, "name": "Brown rice (economy rice)", "desc": "a small white bowl of loose brown rice, every grain visible, warm tan and light brown grains, bright warm light, clean bright colours with no grey shading", "neg": "", "n": 1}, {"id": 4003, "name": "Oil rice (chicken rice)", "desc": "a shallow small white bowl of Hainanese chicken rice, loose fragrant oily rice with every single grain clearly visible, pale golden-yellow and glossy, a few green pandan bits, loosely piled, not a smooth dome", "neg": "", "n": 1}, {"id": 4004, "name": "Nasi lemak rice", "desc": "a small white bowl of loose glossy nasi lemak coconut rice, every grain visible, warm creamy white with a pandan leaf tucked in the side, bright warm light, clean bright colours with no grey shading", "neg": "", "n": 1}, {"id": 4005, "name": "Basmati rice, plain", "desc": "a small white bowl of basmati rice with extra long thin straight grains, each long grain clearly visible, warm creamy white, bright warm light, clean bright colours with no grey shading", "neg": "", "n": 1}, {"id": 4006, "name": "Briyani rice", "desc": "a small mound of long-grain briyani rice with yellow and orange grains, fried onions and a few raisins and cashews on a small white plate", "neg": "", "n": 1}, {"id": 4007, "name": "Turmeric rice (nasi kunyit)", "desc": "a shallow small white bowl of nasi kunyit, loose glossy bright yellow turmeric rice, every single grain clearly visible and separate, loosely piled, not a smooth dome", "neg": "", "n": 1}, {"id": 4008, "name": "Plain fried rice", "desc": "a small mound of plain egg fried rice with bits of scrambled egg and spring onion on a small white plate", "neg": "", "n": 1}, {"id": 4009, "name": "Sweet and sour pork", "desc": "a small dish of sweet and sour pork: crispy battered pork pieces in glossy bright red-orange sauce with pineapple chunks, green capsicum and onion", "neg": "", "n": 1}, {"id": 4010, "name": "Sweet and sour chicken", "desc": "a small dish of sweet and sour chicken: crispy battered chicken pieces in glossy bright red-orange sauce with pineapple chunks, green capsicum and onion", "neg": "", "n": 1}, {"id": 4011, "name": "Marmite chicken", "desc": "a small dish of marmite chicken: fried chicken pieces coated in a sticky dark brown glossy marmite glaze, sprinkled with white sesame seeds", "neg": "", "n": 1}, {"id": 4012, "name": "Kung pao chicken", "desc": "a small dish of kung pao chicken: diced chicken in glossy dark brown sauce with dried red chillies, roasted peanuts and spring onion", "neg": "", "n": 1}, {"id": 4013, "name": "Black pepper chicken", "desc": "a small dish of black pepper chicken: pieces of chicken in thick glossy brown black pepper sauce with sliced onion and green capsicum", "neg": "", "n": 1}, {"id": 4014, "name": "Lemon chicken", "desc": "a small dish of lemon chicken: golden crispy battered chicken slices topped with glossy yellow lemon sauce and a lemon slice", "neg": "", "n": 1}, {"id": 4015, "name": "Curry chicken (economy rice)", "desc": "a small bowl of curry chicken: a piece of chicken and a potato chunk in orange-red coconut curry gravy", "neg": "", "n": 1}, {"id": 4016, "name": "Braised pork belly (tau yu bak)", "desc": "a small bowl of tau yu bak: slices of pork belly with fat layers and a halved hard-boiled egg braised in dark brown soy sauce", "neg": "", "n": 1}, {"id": 4017, "name": "Fried chicken wing (economy rice)", "desc": "a single crispy golden-brown fried chicken wing on a small white plate", "neg": "", "n": 1}, {"id": 4018, "name": "Fried fish fillet", "desc": "a breaded fried fish chop like a chicken chop: a flat wide fish fillet in crunchy golden-brown breadcrumb coating, cut in slices showing white flaky fish inside, on a small white plate", "neg": "", "n": 1}, {"id": 4019, "name": "Steamed fish (economy rice)", "desc": "a piece of steamed fish on a small white plate: a thick fish steak with light silvery-grey skin and white flaky flesh, clearly a fish with fish scales texture on the skin, in light soy sauce with ginger strips and spring onion. The skin is grey, not yellow", "neg": "", "n": 1}, {"id": 4020, "name": "Sambal fish", "desc": "a piece of fried fish covered in red sambal chilli paste with sliced onion on a small white plate", "neg": "", "n": 1}, {"id": 4021, "name": "Sambal prawns", "desc": "a small dish of prawns in red sambal chilli paste with sliced onion", "neg": "", "n": 1}, {"id": 4022, "name": "Luncheon meat slices", "desc": "two pan-fried slices of pink luncheon meat with golden browned edges on a small white plate", "neg": "", "n": 1}, {"id": 4023, "name": "Egg foo yong (omelette)", "desc": "a round golden fluffy Chinese omelette with bits of onion and carrot on a small white plate", "neg": "", "n": 1}, {"id": 4024, "name": "Sambal egg", "desc": "a hard-boiled egg cut in half, covered in red sambal chilli sauce, on a small white plate", "neg": "", "n": 1}, {"id": 4025, "name": "Braised tofu (economy rice)", "desc": "a small dish of golden fried tofu cubes braised in glossy brown oyster sauce with mushrooms", "neg": "", "n": 1}, {"id": 4026, "name": "Mapo tofu", "desc": "a small dish of mapo tofu: soft white tofu cubes in red spicy chilli bean sauce with minced pork and spring onion", "neg": "", "n": 1}, {"id": 4027, "name": "Stir-fried spinach (bayam)", "desc": "a small dish of stir-fried dark green spinach leaves with garlic", "neg": "", "n": 1}, {"id": 4028, "name": "Kangkung belacan", "desc": "a small dish of stir-fried kangkung water spinach with hollow stems and long leaves in reddish belacan chilli paste", "neg": "", "n": 1}, {"id": 4029, "name": "Stir-fried cabbage", "desc": "a small dish of stir-fried pale green cabbage with carrot strips", "neg": "", "n": 1}, {"id": 4030, "name": "Lettuce with oyster sauce", "desc": "a small dish of blanched bright green lettuce leaves drizzled with brown oyster sauce and fried garlic", "neg": "", "n": 1}, {"id": 4031, "name": "Long beans with egg", "desc": "a small dish of chopped green long beans stir-fried with bits of yellow scrambled egg", "neg": "", "n": 1}, {"id": 4032, "name": "Brinjal with sambal", "desc": "a small dish of soft purple brinjal (eggplant) slices topped with red sambal", "neg": "", "n": 1}, {"id": 4033, "name": "Bean sprouts stir-fry", "desc": "a small dish of stir-fried white bean sprouts with chives", "neg": "", "n": 1}, {"id": 4034, "name": "Curry vegetables (sayur lodeh)", "desc": "a small bowl of sayur lodeh: cabbage, long beans, carrot and tofu in bright orange coconut curry", "neg": "", "n": 1}, {"id": 4035, "name": "Stir-fried broccoli", "desc": "a small dish of stir-fried green broccoli florets with carrot slices", "neg": "", "n": 1}, {"id": 4036, "name": "Bitter gourd with egg", "desc": "a small dish of sliced green bitter gourd stir-fried with yellow scrambled egg", "neg": "", "n": 1}, {"id": 4037, "name": "Stir-fried choy sum", "desc": "a small dish of stir-fried green choy sum with garlic", "neg": "", "n": 1}, {"id": 4038, "name": "Curry gravy, add-on", "desc": "a small bowl of curry gravy: smooth rich orange-red curry with a little red chilli oil on top, no rice, no meat", "neg": "", "n": 1}, {"id": 5001, "name": "Chobani Greek yogurt, strawberry", "desc": "a small round white plastic yogurt tub with the lid peeled back, showing thick white Greek yogurt with a layer of red strawberry fruit at the bottom, a plain tub with no logo and no writing, a strawberry beside it", "neg": "", "n": 1}, {"id": 5002, "name": "Chobani Greek yogurt, mango", "desc": "a small round white plastic yogurt tub with the lid peeled back, showing thick white Greek yogurt with orange mango fruit, a plain tub with no logo and no writing, a mango cube beside it", "neg": "", "n": 1}, {"id": 5003, "name": "Chobani Greek yogurt, blueberry", "desc": "a small round white plastic yogurt tub with the lid peeled back, showing thick white Greek yogurt with purple blueberry fruit, a plain tub with no logo and no writing, a few blueberries beside it", "neg": "", "n": 1}, {"id": 5004, "name": "Chobani Greek yogurt, passion fruit", "desc": "a small round white plastic yogurt tub with the lid peeled back, showing thick white Greek yogurt with golden passion fruit pulp with black seeds, a plain tub with no logo and no writing, half a passion fruit beside it", "neg": "", "n": 1}, {"id": 5005, "name": "Chobani Greek yogurt, natural", "desc": "a small round white plastic yogurt tub with the lid peeled back, showing plain thick smooth white Greek yogurt, a plain tub with no logo and no writing, a small spoon beside it", "neg": "", "n": 1}, {"id": 5006, "name": "Chobani Greek yogurt, natural light", "desc": "a small round white plastic yogurt tub with a light blue band, lid peeled back, showing plain thick white Greek yogurt, a plain tub with no logo and no writing", "neg": "", "n": 1}, {"id": 5007, "name": "Chobani Fit, strawberry", "desc": "a small round white plastic yogurt tub with a pink band, lid peeled back, showing pale pink strawberry Greek yogurt, a plain tub with no logo and no writing", "neg": "", "n": 1}, {"id": 5008, "name": "FamilyMart onigiri, tuna mayo", "desc": "a Japanese triangle rice ball broken open into two halves placed side by side, the broken faces turned to the front, each half showing white rice around a big lump of creamy pale yellow tuna mayonnaise filling in the middle, the outside of each half wrapped in dark nori seaweed at the bottom, no face, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5009, "name": "FamilyMart onigiri, salmon", "desc": "a Japanese triangle rice ball broken open into two halves placed side by side, the broken faces turned to the front, each half showing white rice around a big lump of flaked bright pink-orange salmon filling in the middle, the outside of each half wrapped in dark nori seaweed at the bottom, no face, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5010, "name": "FamilyMart onigiri, chicken teriyaki", "desc": "a Japanese triangle rice ball broken open into two halves placed side by side, the broken faces turned to the front, each half showing white rice around a big lump of glossy brown teriyaki chicken filling in the middle, the outside of each half wrapped in dark nori seaweed at the bottom, no face, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5011, "name": "FamilyMart onigiri, spicy tuna", "desc": "a Japanese triangle rice ball broken open into two halves placed side by side, the broken faces turned to the front, each half showing white rice around a big lump of orange-red spicy tuna mayonnaise filling in the middle, the outside of each half wrapped in dark nori seaweed at the bottom, no face, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5012, "name": "FamilyMart oden, fish cake", "desc": "a wooden skewer holding a flat square golden-brown Japanese fish cake, glistening with light broth", "neg": "", "n": 1}, {"id": 5013, "name": "FamilyMart oden, fish ball stick", "desc": "a wooden skewer of three round white fish balls glistening with light golden broth, warm creamy white, bright warm light, clean bright colours with no grey shading", "neg": "", "n": 1}, {"id": 5014, "name": "FamilyMart oden, radish", "desc": "two thick chunks of simmered Japanese daikon radish in a small paper bowl of golden broth: big round thick white cylinders, soft and translucent ivory white with a slightly yellow tint from the broth, with a small dab of yellow mustard on the side, clearly a radish and not a fruit, bright warm light, clean bright colours", "neg": "", "n": 1}, {"id": 5015, "name": "FamilyMart oden, egg", "desc": "a peeled hard-boiled egg stained light brown from simmering in broth, cut in half to show the white and the yellow yolk inside, in a small paper bowl with a little golden broth, bright warm light, clean bright colours with no grey shading", "neg": "", "n": 1}, {"id": 5016, "name": "FamilyMart oden, tofu puff", "desc": "a square golden fried tofu puff soaked in light broth on a wooden skewer", "neg": "", "n": 1}, {"id": 5017, "name": "FamilyMart oden, crab stick", "desc": "a red and white crab stick on a wooden skewer glistening with broth", "neg": "", "n": 1}, {"id": 5018, "name": "FamilyMart oden, chikuwa", "desc": "a Japanese chikuwa fish cake on a wooden skewer: a long white tube with a hollow centre, the middle part toasted golden-brown with darker grilled spots while both ends stay white, bright warm light, clean bright colours with no grey shading", "neg": "", "n": 1}, {"id": 5019, "name": "FamilyMart oden, udon noodles", "desc": "a small paper bowl of thick white udon noodles in clear golden oden broth with a little spring onion, bright warm light, clean bright colours with no grey shading", "neg": "", "n": 1}, {"id": 5020, "name": "FamilyMart Sofuto, cone", "desc": "a FamilyMart Sofuto soft serve cone: a very long slim pointed waffle cone shaped like a tall narrow triangle with a sharp point at the bottom, the cone is twice as long as a normal ice cream cone, with a white paper sleeve wrapped around the lower part of the cone, topped with a tall neat swirl of creamy milk soft serve, held upright, no hand, bright warm light, clean bright colours", "neg": "", "n": 1}, {"id": 5021, "name": "FamilyMart Sofuto, cup", "desc": "a Japanese convenience store soft serve in a cup: a tall slim pointed swirl of creamy milk soft serve inside a clear plastic cup covered by a clear transparent dome lid, with a small plastic spoon, bright warm light, clean bright colours", "neg": "", "n": 1}, {"id": 5022, "name": "FamilyMart Sofuto parfait", "desc": "a clear plastic parfait cup layered with cornflakes, strawberries and fruit, topped with a tall swirl of creamy warm white soft serve, strawberry sauce and a wafer, bright warm light, clean bright colours with no grey shading", "neg": "", "n": 1}, {"id": 5023, "name": "FamilyMart egg mayo sando", "desc": "a Japanese sandwich cut into two triangles showing soft white bread with a thick creamy yellow egg mayonnaise filling, in a clear plastic triangle pack with no writing", "neg": "", "n": 1}, {"id": 5024, "name": "FamilyMart tuna mayo sando", "desc": "a Japanese sandwich cut into two triangles showing soft white bread with creamy tuna mayonnaise and lettuce, in a clear plastic triangle pack with no writing", "neg": "", "n": 1}, {"id": 5025, "name": "FamilyMart chicken katsu sando", "desc": "a thick Japanese sandwich cut in half showing soft white bread with a golden breaded chicken cutlet, brown katsu sauce and shredded cabbage", "neg": "", "n": 1}, {"id": 5026, "name": "FamilyMart fruit sando", "desc": "a Japanese fruit sandwich cut in half showing soft white bread with thick whipped cream and whole strawberries, kiwi and mandarin slices arranged inside", "neg": "", "n": 1}, {"id": 5027, "name": "FamilyMart chicken katsu bento", "desc": "a black rectangular takeaway bento box with white rice, golden sliced chicken katsu with brown sauce, shredded cabbage, a small piece of tamagoyaki and pickles", "neg": "", "n": 1}, {"id": 5028, "name": "FamilyMart chicken teriyaki don", "desc": "a round takeaway bowl of white rice topped with glossy brown teriyaki chicken slices, sesame seeds, spring onion and a few vegetables", "neg": "", "n": 1}, {"id": 5029, "name": "FamilyMart nasi lemak", "desc": "a rectangular takeaway box of nasi lemak: white coconut rice, red sambal, fried chicken, half a boiled egg, cucumber slices, fried anchovies and peanuts", "neg": "", "n": 1}, {"id": 5030, "name": "FamilyMart spaghetti bolognese", "desc": "a rectangular takeaway box of spaghetti with red bolognese meat sauce and grated cheese", "neg": "", "n": 1}, {"id": 5031, "name": "FamilyMart spaghetti carbonara", "desc": "a rectangular takeaway box of spaghetti in creamy white carbonara sauce with bits of beef bacon and black pepper", "neg": "", "n": 1}, {"id": 5032, "name": "FamilyMart mac and cheese", "desc": "a rectangular takeaway box of macaroni in thick golden melted cheese sauce, lightly browned on top", "neg": "", "n": 1}, {"id": 5033, "name": "FamilyMart fried chicken", "desc": "a single piece of crispy golden-brown fried chicken in a small paper holder", "neg": "", "n": 1}, {"id": 5034, "name": "FamilyMart chicken karaage", "desc": "a small paper cup of five pieces of Japanese karaage fried chicken, golden and crispy, with a toothpick", "neg": "", "n": 1}, {"id": 5035, "name": "FamilyMart daifuku mochi", "desc": "a round soft white Japanese daifuku mochi dusted with powder, cut open to show red bean paste and a strawberry inside", "neg": "", "n": 1}, {"id": 5036, "name": "FamilyMart roll cake", "desc": "a slice of Japanese roll cake: soft light sponge rolled around a thick spiral of whipped cream", "neg": "", "n": 1}, {"id": 5037, "name": "FamilyMart iced matcha latte", "desc": "a clear plastic cup of iced matcha latte with green matcha on top of white milk and ice cubes, with a straw, a plain cup with no logo", "neg": "", "n": 1}, {"id": 5038, "name": "FamilyMart iced latte", "desc": "a clear plastic cup of iced latte with coffee swirling into white milk and ice cubes, with a straw, a plain cup with no logo", "neg": "", "n": 1}, {"id": 5039, "name": "7-Eleven onigiri, tuna mayo", "desc": "a Japanese triangle rice ball broken open into two halves placed side by side, the broken faces turned to the front, each half showing white rice around a big lump of creamy pale yellow tuna mayonnaise filling in the middle, the outside of each half wrapped in dark nori seaweed at the bottom, no face, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5040, "name": "7-Eleven onigiri, chicken", "desc": "a Japanese triangle rice ball broken open into two halves placed side by side, the broken faces turned to the front, each half showing white rice around a big lump of brown seasoned chicken filling in the middle, the outside of each half wrapped in dark nori seaweed at the bottom, no face, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5041, "name": "7-Eleven egg mayo sandwich", "desc": "two triangle sandwich halves of white bread with creamy yellow egg mayonnaise filling, in a clear plastic triangle pack with no writing", "neg": "", "n": 1}, {"id": 5042, "name": "7-Eleven tuna sandwich", "desc": "two triangle sandwich halves of white bread with tuna mayonnaise and lettuce, in a clear plastic triangle pack with no writing", "neg": "", "n": 1}, {"id": 5043, "name": "7-Eleven chicken slice sandwich", "desc": "two triangle sandwich halves of white bread with pink chicken ham slices, cheese and lettuce, in a clear plastic triangle pack with no writing", "neg": "", "n": 1}, {"id": 5044, "name": "7-Eleven nasi lemak", "desc": "a nasi lemak packed in a plastic box: white coconut rice, red sambal, half a boiled egg, cucumber, fried anchovies and peanuts", "neg": "", "n": 1}, {"id": 5045, "name": "7-Eleven fried rice", "desc": "a takeaway box of fried rice with egg, small pieces of chicken, peas and carrot", "neg": "", "n": 1}, {"id": 5046, "name": "7-Eleven spaghetti bolognese", "desc": "a microwave takeaway box of spaghetti with red bolognese meat sauce", "neg": "", "n": 1}, {"id": 5047, "name": "7-Eleven hot dog", "desc": "a hot dog: a long grilled sausage in a soft split bun with a zigzag of yellow mustard and red ketchup", "neg": "", "n": 1}, {"id": 5048, "name": "7-Eleven mashed potato with gravy", "desc": "a small paper cup of smooth fluffy light buttery yellow mashed potato in a soft heap, clearly pale yellow like butter, with rich brown gravy poured over the top, bright warm light, clean bright colours", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5049, "name": "7-Eleven chicken pau", "desc": "a round fluffy white steamed bun, one half torn open showing savoury chicken filling", "neg": "", "n": 1}, {"id": 5050, "name": "7-Eleven Slurpee, regular", "desc": "a medium clear plastic cup of bright red and blue icy slushie drink with a dome lid and a wide straw, a plain cup with no logo", "neg": "", "n": 1}, {"id": 5051, "name": "7-Eleven Slurpee, large", "desc": "a big tall clear plastic cup of bright red and blue icy slushie drink with a dome lid and a wide straw, a plain cup with no logo", "neg": "", "n": 1}, {"id": 5052, "name": "myNEWS Maru onigiri, tuna mayo", "desc": "a Japanese triangle rice ball broken open into two halves placed side by side, the broken faces turned to the front, each half showing white rice around a big lump of creamy pale yellow tuna mayonnaise filling in the middle, the outside of each half wrapped in dark nori seaweed at the bottom, no face, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5053, "name": "myNEWS Maru sandwich, egg mayo", "desc": "a Japanese style egg mayo sandwich cut into two triangles, showing soft white bread with a thick creamy yellow egg mayonnaise filling, in a clear plastic triangle pack with no writing", "neg": "", "n": 1}, {"id": 5054, "name": "myNEWS Maru bento, chicken", "desc": "a rectangular takeaway bento box with white rice, glazed chicken pieces, a fried egg and vegetables", "neg": "", "n": 1}, {"id": 5055, "name": "KK Mart nasi lemak bungkus", "desc": "a small nasi lemak bungkus wrapped in brown paper and green banana leaf, opened to show white coconut rice, red sambal, a boiled egg slice, cucumber and fried anchovies", "neg": "", "n": 1}, {"id": 5056, "name": "Gardenia Original Classic bread", "desc": "one slice of soft white bread with a golden brown crust standing in front, and behind it a whole sliced white bread loaf inside a clear plastic bread bag tied at the top, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5057, "name": "Gardenia wholemeal bread", "desc": "one slice of brown wholemeal bread with lots of small wheat bits and seed dots and a darker brown crust standing in front, behind it a whole sliced brown wholemeal loaf inside a clear plastic bread bag tied at the top, and a golden stalk of wheat beside it, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5058, "name": "Gardenia Twiggies", "desc": "two Gardenia Twiggies Choc-A-Lot chocolate sponge fingers lying on their own with no plate: each a soft chocolate brown sponge cake log, rounded rectangle shape like a small fat finger, each inside its own clear transparent plastic wrapper packet with crimped sealed ends, one packet torn open with the sponge peeking out bitten to show chocolate brown sponge inside and a thick dark chocolate cream filling, no white cream, no plate, no writing, bright warm light, clean bright colours", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5059, "name": "Gardenia Delicia butterscotch roll", "desc": "a soft golden brown bread roll split along the top and filled with a thick line of light caramel coloured butterscotch cream, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5060, "name": "Massimo sandwich loaf", "desc": "two slices of soft white sandwich bread with light golden crusts, beside a plain sliced loaf in a clear bag with no logo", "neg": "", "n": 1}, {"id": 5061, "name": "Farm Fresh fresh milk", "desc": "one small plastic bottle of fresh milk with a blue cap, the creamy warm white milk visible, beside a short glass of milk, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5062, "name": "Farm Fresh chocolate milk", "desc": "one small plastic bottle of chocolate milk with a brown cap, the brown milk visible, beside a short glass of chocolate milk, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5063, "name": "Farm Fresh yogurt drink", "desc": "one small plastic bottle completely filled with bright pastel pink strawberry yogurt drink, the whole bottle looks pink because of the pink drink inside, with a pink cap, no logo, bright warm light, clean bright colours", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5064, "name": "Oatside oat milk, barista blend", "desc": "one small beige drink carton with a straw, beside a glass of creamy light beige oat milk and a few oats, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5065, "name": "Oatside oat milk, chocolate", "desc": "one small brown drink carton with a straw, beside a glass of chocolate oat milk and a few oats, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5066, "name": "Soft drink, zero sugar", "desc": "one shiny black aluminium soda can, beside a glass of dark brown fizzy cola with ice cubes and bubbles, no logo, bright warm light, clean bright colours with no grey shading", "neg": "grey, gray, dull, stone, rock, face, eyes, smile, writing, logo", "n": 1}, {"id": 5067, "name": "Kaya bun", "desc": "a soft round golden baked bun torn open to show green-brown kaya coconut jam filling inside, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 5068, "name": "Red bean bun", "desc": "a soft round golden baked bun with black sesame seeds on top, torn open to show dark red-brown sweet red bean paste filling inside, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 5069, "name": "Nasi lemak kerang", "desc": "nasi lemak with cockles: a dome of white coconut rice, a big serving of small round orange-brown cockle meats in dark red sambal sauce, crispy fried anchovies, roasted peanuts, cucumber slices and half a hard-boiled egg, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6001, "name": "Grapes, purple seedless", "desc": "a small bunch of round dark purple seedless grapes on a green stem, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6002, "name": "Grapes, green seedless", "desc": "a small bunch of oval pale green seedless grapes on a green stem, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6003, "name": "Grapes, red", "desc": "a small bunch of round red grapes on a green stem, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6004, "name": "Strawberries", "desc": "five fresh red strawberries with green leafy tops, one cut in half, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6005, "name": "Blueberries", "desc": "a small white bowl of round dark blue blueberries, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6006, "name": "Raspberries", "desc": "a small white bowl of red raspberries, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6007, "name": "Mixed berries", "desc": "a small white bowl of mixed strawberries, blueberries and raspberries, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6008, "name": "Cherries", "desc": "six dark red cherries with long green stems, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6009, "name": "Mango", "desc": "a ripe yellow mango with one cheek cut off and scored into cubes showing bright orange-yellow flesh, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6010, "name": "Pineapple", "desc": "a few golden-yellow pineapple chunks and one pineapple ring on a small plate, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6011, "name": "Honeydew", "desc": "two wedges of pale green honeydew melon with pale green rind, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6012, "name": "Rock melon", "desc": "two wedges of orange rock melon cantaloupe with netted beige rind, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6013, "name": "Dragon fruit, red", "desc": "half a dragon fruit with bright pink skin and green scales, showing deep magenta flesh with tiny black seeds, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6014, "name": "Dragon fruit, white", "desc": "half a dragon fruit with bright pink skin and green scales, showing white flesh with tiny black seeds, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6015, "name": "Kiwi", "desc": "one whole brown fuzzy kiwi and one half showing bright green flesh with a white centre and black seeds, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6016, "name": "Pear", "desc": "one green-yellow Western pear with a stem, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6017, "name": "Chinese pear", "desc": "one round pale yellow Chinese pear with a stem, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6018, "name": "Green apple", "desc": "one shiny green apple with a stem, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6019, "name": "Peach", "desc": "one whole round ripe peach, bright warm yellow-orange with a strong rosy pink-red blush across one side, soft fuzzy skin, a short brown stem and one green leaf, with a wedge slice of peach beside it showing juicy golden-yellow flesh, vivid colours, not grey, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6020, "name": "Plum", "desc": "two dark purple-red plums, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6021, "name": "Persimmon", "desc": "one orange persimmon with a green leafy cap, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6022, "name": "Pomegranate", "desc": "half a red pomegranate showing many shiny ruby-red seeds, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6023, "name": "Mandarin orange", "desc": "one mandarin orange with a green leaf and a few peeled segments beside it, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6024, "name": "Pomelo", "desc": "three large peeled pomelo segments with pale pink juicy pulp and a piece of thick green-yellow rind, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6025, "name": "Grapefruit", "desc": "half a grapefruit showing pink-red segmented flesh with yellow rind, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6026, "name": "Rambutan", "desc": "four red rambutans with soft green-tipped hairy spikes, one opened to show the white translucent fruit, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6027, "name": "Longan", "desc": "a small cluster of round light-brown longans on a twig, one peeled to show the translucent white fruit, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6028, "name": "Lychee", "desc": "four lychees with bumpy pink-red skin, one peeled to show the white translucent fruit, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6029, "name": "Mangosteen", "desc": "two dark purple mangosteens with green caps, one opened to show white segments inside, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6030, "name": "Duku langsat", "desc": "a small cluster of round pale yellow-beige langsat fruits, one peeled to show translucent white segments, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6031, "name": "Jackfruit (nangka)", "desc": "four bright yellow jackfruit pods on a small plate, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6032, "name": "Cempedak", "desc": "cempedak: half of a long oval cempedak fruit with bumpy yellow-green skin, split open to show a tight row of plump golden-orange oval pods clinging to the core, with three loose golden-orange pods beside it on a plate, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6033, "name": "Durian", "desc": "three creamy pale-yellow durian flesh pods inside a piece of green spiky durian husk, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6034, "name": "Ciku (sapodilla)", "desc": "one round brown ciku sapodilla and one half showing soft brown flesh with black seeds, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6035, "name": "Starfruit", "desc": "one yellow starfruit and three star-shaped slices, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6036, "name": "Jambu air (rose apple)", "desc": "three bell-shaped glossy pink-red jambu air rose apples, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6037, "name": "Soursop", "desc": "a wedge of soursop with green spiky skin and white fibrous pulp with black seeds, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6038, "name": "Avocado", "desc": "half an avocado with dark green skin, pale green-yellow flesh and a round brown seed, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6039, "name": "Pisang emas", "desc": "a small hand of four short golden-yellow pisang emas bananas, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6040, "name": "Fig, fresh", "desc": "two purple figs, one cut in half showing pink-red seedy flesh, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6041, "name": "Cherry tomatoes", "desc": "a small pile of round red cherry tomatoes with green stems, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6042, "name": "Dates", "desc": "four glossy dark brown dates on a small plate, fresh fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6043, "name": "Raisins", "desc": "raisins: a small heap of many tiny wrinkled dark brown-purple raisins spilling out of a small white bowl onto the plate, each raisin small and shrivelled, no box, dried fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6044, "name": "Prunes", "desc": "prunes: five plump moist dried plums, glossy and sticky deep purple-black with soft wrinkles and a wet shine, one cut open to show soft amber-brown flesh, on a small plate, dried fruit, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 6045, "name": "Dried mango", "desc": "dried mango: six thin flat leathery strips of dried mango, deep dark orange with a matte chewy surface and slightly curled dry edges, dusted lightly with sugar, overlapping on a small plate, clearly dried and not fresh, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 5070, "name": "Fish porridge", "desc": "fish porridge: a bowl filled to the top with thick creamy white rice congee with soft broken rice grains visible in the porridge, topped with a few thin slices of white fish fillet with a grey skin edge, ginger strips, chopped spring onion and a drizzle of sesame oil, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 5072, "name": "Tanghulu, strawberry", "desc": "strawberry tanghulu: one wooden skewer of five large whole red strawberries with green tops, coated in a thick hard shiny clear sugar glaze with bright highlights, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 5073, "name": "Tanghulu, mini strawberries", "desc": "mini strawberry tanghulu: one wooden skewer of eight small red strawberries packed closely, coated in a hard shiny clear sugar glaze, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 5074, "name": "Tanghulu, grapes", "desc": "grape tanghulu: one long thin wooden skewer held upright with eight separate round grapes threaded one by one in a single straight row, alternating green and purple, each grape a separate ball touching the next, not a bunch, coated in a hard shiny clear sugar glaze with bright highlights, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5075, "name": "Tanghulu, hawthorn (classic)", "desc": "classic hawthorn tanghulu: one wooden skewer of eight round bright red hawthorn berries, coated in a hard shiny clear amber sugar glaze with a thin sugar sheet at the tip, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 5076, "name": "Tanghulu, mandarin", "desc": "mandarin tanghulu: one long thin wooden skewer with five small whole peeled mandarin oranges threaded in a single row, each one bright orange and round with clear segment lines, fine white pith threads and a slightly wrinkled bumpy surface, coated in a hard shiny clear sugar glaze, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7001, "name": "Greek yogurt (ingredient)", "desc": "one small clear plastic yogurt cup with straight sides and a rolled rim, seen from a three-quarter angle with a soft blue-grey shadow so the cup stands out, filled with thick plain white Greek yogurt piled high in a creamy swirl with a soft peak, the yogurt pure white with pale cream shading, no fruit and no strawberry, a small silver spoon resting beside the cup, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7002, "name": "Plain low-fat yogurt (ingredient)", "desc": "one small clear plastic yogurt cup with straight sides and a rolled rim, seen from a three-quarter angle with a soft blue-grey shadow so the cup stands out, filled with smooth plain white low-fat yogurt with a gentle flat swirl on top, thinner and glossier than Greek yogurt, pure white with pale blue-grey shading, no fruit and no strawberry, a peeled-back silver foil lid on the rim, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7003, "name": "Flavoured yogurt (ingredient)", "desc": "smooth pale pink strawberry yogurt in a small clear cup, with a few tiny red fruit specks, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7004, "name": "Soy yogurt, unsweetened (ingredient)", "desc": "smooth off-white beige soy yogurt in a small white cup, with two yellow soybeans beside it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7005, "name": "Instant oats, dry (ingredient)", "desc": "a small heap of dry flat pale beige rolled oat flakes, with a wooden scoop half buried in them, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7006, "name": "Low-fat milk (ingredient)", "desc": "low-fat milk: one tall clear drinking glass with a thick dark outline, three-quarters full of white milk with a clear flat milk line and a pale blue shadow inside the glass so the white milk is easy to see, a light blue striped paper straw in the glass, empty space above the milk, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7007, "name": "Full cream milk (ingredient)", "desc": "full cream milk: one short wide clear drinking glass with a thick dark outline, filled nearly to the top with rich creamy white milk tinted very slightly warm cream, a thick creamy surface line and soft cream-yellow shading, a small splash crown on the surface, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7008, "name": "Soy milk, unsweetened (ingredient)", "desc": "unsweetened soy milk: one tall clear drinking glass with a thick dark outline, filled with pale beige-cream soy milk, clearly more beige than cow's milk, with a clear surface line, and three small round yellow soybeans lying beside the glass, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7009, "name": "Oat milk (ingredient)", "desc": "a short clear glass of pale beige oat milk, with a few oat flakes scattered beside it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7010, "name": "Wholemeal (ingredient)", "desc": "wholemeal sub roll: one long sandwich roll, medium brown wholemeal crust, the whole top surface speckled with many tiny dark brown bran flecks, oat flakes and seeds, a cut end showing a tan-brown crumb with brown specks, clearly rougher and darker than white bread, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7011, "name": "Italian white (ingredient)", "desc": "one long sub roll of soft pale golden white bread with a smooth crust, cut open lengthways and empty, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7012, "name": "Honey oat (ingredient)", "desc": "one long golden-brown sub roll topped with rolled oats and a honey glaze, cut open lengthways and empty, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7013, "name": "Tortilla wrap (ingredient)", "desc": "one thin round pale flour tortilla with light brown toasted spots, lying flat with one edge folded over, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7014, "name": "Wholemeal wrap (ingredient)", "desc": "one thin round light brown wholemeal tortilla with darker flecks, lying flat with one edge folded over, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7015, "name": "Pita bread (ingredient)", "desc": "pita bread: two round flat thin pocket breads stacked, pale beige with light golden-brown toasted patches and a floury matte surface, the top one cut in half to show the hollow pocket opening inside, flat like a disc and not a puffy bun, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7016, "name": "Burger bun (ingredient)", "desc": "burger bun, plain with no filling: the bottom half of a soft golden bun lying flat showing its pale cut surface, and the domed top half with white sesame seeds leaning sideways against it at an angle, two separate bread halves, completely empty, no patty, no cheese, no lettuce, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7017, "name": "Baguette (ingredient)", "desc": "baguette: one single very long thin French baguette lying diagonally across the picture from corner to corner, golden-brown crisp crust with five diagonal slashes along the top and pointed ends, only one loaf, nothing else beside it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7018, "name": "Sliced bread, 2 slices (ingredient)", "desc": "two square slices of soft white sandwich bread with a thin brown crust, one leaning on the other, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7019, "name": "Flatbread (ingredient)", "desc": "one oval soft flatbread with light brown grill marks and a dusting of flour, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7020, "name": "Roasted chicken (ingredient)", "desc": "roasted chicken: a neat pile of sliced roasted chicken on a small plate, each piece with a strip of well-browned deep golden-brown roasted skin on top and white juicy meat underneath, all pieces inside the plate, nothing lying outside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7021, "name": "Chicken teriyaki (ingredient)", "desc": "a small pile of bite-sized chicken pieces coated in shiny dark brown sticky teriyaki glaze with white sesame seeds, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7022, "name": "Tuna mayo (ingredient)", "desc": "tuna mayo: a scoop of tuna mayonnaise on a small plate, clearly visible pink-beige flaked tuna in shreds and flakes mixed with creamy pale yellow mayonnaise, small bits of green celery and sweetcorn kernels showing, rough flaky texture, not smooth, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7023, "name": "Egg mayo (ingredient)", "desc": "egg mayo: a scoop of egg mayonnaise on a small plate, big chunky pieces of chopped boiled egg, clear white egg-white cubes and crumbly bright yellow yolk pieces, bound in a little pale yellow mayonnaise, chunky and lumpy, a sprinkle of black pepper, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7024, "name": "Egg, sliced (ingredient)", "desc": "five round slices of hard-boiled egg, white rings with bright yellow yolk centres, overlapping in a row, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7025, "name": "Turkey slices (ingredient)", "desc": "three thin round slices of pale pink-beige turkey ham folded loosely in a small stack, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7026, "name": "Roast beef (ingredient)", "desc": "roast beef: four thin slices of roast beef folded and overlapping on a small plate, each slice mostly brown-grey cooked beef with a darker brown roasted edge and only a faint hint of pink in the very centre, all slices together on the plate, nothing lying outside, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7027, "name": "Meatballs in tomato sauce (ingredient)", "desc": "four round brown meatballs coated in thick bright red tomato sauce, with a sprinkle of green herbs, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7028, "name": "Grilled chicken breast (ingredient)", "desc": "grilled chicken breast: one whole flat oval chicken breast fillet, pale golden-white meat with five dark brown parallel grill lines across the top, sliced into six thick slices fanned slightly to show white juicy meat inside, lean meat, not bread, not a pastry, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7029, "name": "Fried chicken cutlet (ingredient)", "desc": "fried chicken cutlet: one large flat oval breaded chicken cutlet, golden-brown crunchy breadcrumb coating, cut into five strips showing white chicken meat inside, all the strips together in one piece on a small plate, nothing else on the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7030, "name": "Black pepper chicken (ingredient)", "desc": "black pepper chicken: irregular bite-sized chunks of chicken thigh meat, uneven angular pieces and not round, glazed in dark glossy brown black pepper sauce and covered with many coarse cracked black peppercorn specks, with a few pieces of onion, on a small plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7031, "name": "Beef teriyaki (ingredient)", "desc": "a small pile of thin beef slices coated in shiny dark brown teriyaki glaze with white sesame seeds, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7032, "name": "Grilled beef slices (ingredient)", "desc": "five thin slices of grilled beef, brown with darker charred edges, fanned out, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7033, "name": "Beef patty (ingredient)", "desc": "one thick round grilled beef burger patty, dark brown with a charred criss-cross pattern and juicy sides, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7034, "name": "Grilled lamb slices (ingredient)", "desc": "five thin slices of grilled lamb, brown with a pink centre and charred edges, with a rosemary sprig, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7035, "name": "Pork teriyaki (ingredient)", "desc": "a small pile of thin pork slices coated in shiny brown teriyaki glaze with white sesame seeds, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7036, "name": "Char siu (ingredient)", "desc": "six slices of char siu barbecued pork, with a bright red glazed edge and pale pink-brown meat, overlapping in a row, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7037, "name": "Pulled pork (ingredient)", "desc": "a small heap of shredded pulled pork in sticky reddish-brown barbecue sauce, with long meat strands, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7038, "name": "Chicken ham (ingredient)", "desc": "three thin round slices of pale pink chicken ham with a smooth surface, overlapping, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7039, "name": "Bacon, 2 rashers (ingredient)", "desc": "two wavy strips of crispy fried bacon, streaked dark red and golden-white, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7040, "name": "Luncheon meat, 2 slices (ingredient)", "desc": "two rectangular slices of pink luncheon meat, pan-fried golden-brown on the edges, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7041, "name": "Grilled fish fillet (ingredient)", "desc": "one grilled white fish fillet with light golden grill marks and a lemon wedge, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7042, "name": "Fried fish fillet (ingredient)", "desc": "one golden-brown crispy breaded fried fish fillet, broken open to show flaky white fish inside, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7043, "name": "Salmon teriyaki (ingredient)", "desc": "salmon teriyaki: one thick rectangular cooked salmon fillet, opaque light pink-orange flaky cooked flesh with white fat lines, a strip of silvery light grey skin along one side, the top seared brown and glazed, with dark glossy brown teriyaki sauce drizzled over the top and pooling underneath, white sesame seeds, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7044, "name": "Smoked salmon (ingredient)", "desc": "smoked salmon: five very thin, almost see-through delicate slices of smoked salmon, each folded and ruffled like ribbon, bright orange-pink with fine pale white fat lines, glossy and silky, laid loosely on a small plate with a tiny sprig of dill, paper-thin, not thick cuts, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7045, "name": "Tuna, plain (ingredient)", "desc": "plain tuna: a small mound of flaked cooked tuna on a small plate, soft pale pink-beige flakes and shreds pulled apart with visible fibres, warm pink-beige and cream colours only, clearly flaked fish and not cubes, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7046, "name": "Tofu, pan-fried (ingredient)", "desc": "four rectangular slabs of tofu pan-fried golden-brown on the outside and white inside, stacked, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7047, "name": "Tempeh (ingredient)", "desc": "four thin rectangular slices of fried tempeh, golden-brown with whole soybeans visible pressed together, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7048, "name": "Falafel, 3 pieces (ingredient)", "desc": "three round deep-fried falafel balls with a rough dark golden-brown crust, one broken open to show green inside, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7049, "name": "Veggie patty (ingredient)", "desc": "one round vegetable burger patty, golden-brown with bits of green peas, orange carrot and yellow corn showing, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7050, "name": "Cheese slice (ingredient)", "desc": "cheese slice: three thin flat square slices of yellow-orange cheddar cheese fanned and overlapping, each a perfect thin square with smooth glossy surface and soft bendy corners, one corner folded over, flat sandwich cheese slices, not a wedge, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7051, "name": "Cream cheese (ingredient)", "desc": "cream cheese: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick smooth white cream cheese spread with a soft swirled peak and a few knife marks, very pale cream-white with light cream shading, a small butter knife resting on the bowl, no bagel and no bread, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7052, "name": "Feta (ingredient)", "desc": "a small pile of crumbly white feta cheese cubes, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7053, "name": "Mozzarella (ingredient)", "desc": "mozzarella: one ball of fresh white mozzarella cut into five thick round slices fanned out in a row, each slice a smooth milky white disc with a soft wet shine and a slightly stringy edge, two green basil leaves beside them, sliced cheese, not buns, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7054, "name": "Lettuce (ingredient)", "desc": "three crisp ruffled light green lettuce leaves, layered, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7055, "name": "Tomato (ingredient)", "desc": "four round slices of ripe red tomato showing the seeds and juicy centre, overlapping, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7056, "name": "Cucumber (ingredient)", "desc": "six thin round slices of cucumber with dark green skin and pale green centres, overlapping in a row, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7057, "name": "Onion (ingredient)", "desc": "a small pile of thin raw purple-red onion rings, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7058, "name": "Capsicum (ingredient)", "desc": "a small pile of thin strips of green, red and yellow bell pepper, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7059, "name": "Carrot, shredded (ingredient)", "desc": "a small fluffy heap of thin bright orange shredded carrot, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7060, "name": "Sweetcorn (ingredient)", "desc": "a small heap of bright yellow sweetcorn kernels, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7061, "name": "Cabbage, shredded (ingredient)", "desc": "a small fluffy heap of finely shredded pale green and white cabbage, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7062, "name": "Purple cabbage (ingredient)", "desc": "a small fluffy heap of finely shredded bright purple cabbage, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7063, "name": "Spinach (ingredient)", "desc": "a small bunch of dark green spinach leaves with thin stems, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7064, "name": "Avocado (ingredient)", "desc": "half an avocado cut into a fan of thin slices, bright green at the edge fading to pale yellow, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7065, "name": "Mushrooms, sautéed (ingredient)", "desc": "a small pile of sliced brown button mushrooms, sauteed glossy and golden-brown, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7066, "name": "Bean sprouts (ingredient)", "desc": "a small heap of crisp white bean sprouts with tiny yellow heads, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7067, "name": "Coleslaw (ingredient)", "desc": "a scoop of creamy coleslaw, shredded white cabbage and orange carrot in pale mayonnaise, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7068, "name": "Coriander (ingredient)", "desc": "a small bunch of fresh bright green coriander leaves on thin stems, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7069, "name": "Olives (ingredient)", "desc": "a small pile of black olive rings and three whole green olives, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7070, "name": "Pickles (ingredient)", "desc": "six round ridged slices of dark green pickled gherkin, overlapping, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7071, "name": "Jalapeños (ingredient)", "desc": "a small pile of round sliced green jalapeno peppers with pale seeds, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7072, "name": "Kimchi (ingredient)", "desc": "a small heap of red spicy kimchi, napa cabbage pieces coated in red chilli paste, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7073, "name": "Sweet onion (ingredient)", "desc": "sweet onion sauce: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with smooth glossy pale golden-yellow sweet onion sauce with tiny translucent onion bits and a few dark poppy seed specks, a shiny highlight on the surface, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7074, "name": "Honey mustard (ingredient)", "desc": "honey mustard sauce: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with smooth creamy golden-yellow honey mustard sauce with a few dark mustard seed specks and a glossy swirl on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7075, "name": "BBQ (ingredient)", "desc": "BBQ sauce: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick dark reddish-brown barbecue sauce, glossy and sticky with a bright highlight and a thick swirl on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7076, "name": "Mayonnaise (ingredient)", "desc": "mayonnaise: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick creamy mayonnaise in a soft swirled peak, clearly cream-yellow in colour like pale custard, glossy, with warm yellow shading so it stands out from the white bowl, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7077, "name": "Light mayonnaise (ingredient)", "desc": "light mayonnaise: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with smooth light mayonnaise in a low soft swirl, pale cream-yellow, a little paler and thinner than regular mayonnaise but still clearly yellowish against the white bowl, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7078, "name": "Ranch (ingredient)", "desc": "ranch dressing: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with creamy off-white ranch dressing with a visible swirl, dotted all over with many small green flakes of parsley, chives and dill and a few black pepper specks, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7079, "name": "Chilli sauce (ingredient)", "desc": "chilli sauce: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with smooth bright red chilli sauce, glossy with a shiny highlight and a few tiny pale seeds, a thick swirl on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7080, "name": "Sweet chilli sauce (ingredient)", "desc": "sweet chilli sauce: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with clear glossy orange-red sweet chilli sauce, translucent and syrupy, with many red chilli flakes and pale seeds floating in it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7081, "name": "Teriyaki sauce (ingredient)", "desc": "teriyaki sauce: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with dark glossy brown teriyaki sauce, thin and shiny like syrup with a bright highlight, a few white sesame seeds floating on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7082, "name": "Tomato ketchup (ingredient)", "desc": "tomato ketchup: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick smooth bright tomato-red ketchup with a glossy swirled peak on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7083, "name": "Mustard (ingredient)", "desc": "yellow mustard: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with smooth bright yellow mustard with a glossy swirled peak on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7084, "name": "Sriracha (ingredient)", "desc": "a small dish of smooth orange-red sriracha chilli sauce with a zigzag on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7085, "name": "Thousand island (ingredient)", "desc": "a small dish of creamy salmon-pink thousand island dressing with tiny green and red bits, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7086, "name": "Garlic sauce (ingredient)", "desc": "a small dish of fluffy bright white garlic sauce with one garlic clove beside it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7087, "name": "Black pepper sauce (ingredient)", "desc": "black pepper sauce: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick dark brown gravy-like black pepper sauce covered with many coarse cracked black peppercorn specks, glossy, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7088, "name": "Sambal (ingredient)", "desc": "sambal: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick chunky dark red sambal, a coarse rough paste of pounded red chillies with visible chilli skin flakes, seeds and bits of shallot, a ring of red chilli oil around the edge, thick and textured, not smooth, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7089, "name": "Hummus (ingredient)", "desc": "a small dish of smooth beige hummus with a swirl, a drop of golden olive oil and red paprika dust, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7090, "name": "Guacamole (ingredient)", "desc": "a small dish of chunky bright green guacamole with tiny red tomato and white onion bits, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7091, "name": "Olive oil (ingredient)", "desc": "a small dish of clear golden-green olive oil with one green olive beside it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7092, "name": "Wholemeal bread (ingredient)", "desc": "wholemeal bread: two slices of square wholemeal sandwich bread leaning on each other, medium brown crumb densely speckled all over with many tiny dark brown bran flecks, grains and seeds, a darker brown crust, clearly rough and grainy, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7093, "name": "White bread (ingredient)", "desc": "two square slices of soft white bread with a thin light brown crust, one leaning on the other, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7094, "name": "Sourdough (ingredient)", "desc": "sourdough: one whole rustic oval sourdough loaf, thick dark golden-brown crackly crust dusted with white flour, one long deep score down the top that has burst open into a raised crisp ear showing the paler golden inside, with one thick slice cut from the end showing an open airy crumb with big holes, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7095, "name": "Hainan thick toast (ingredient)", "desc": "one very thick square slice of white Hainan bread, toasted golden-brown on both faces, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7096, "name": "Butter (ingredient)", "desc": "one thick rectangular slab of pale yellow butter with one curl shaved off the top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7097, "name": "Margarine (ingredient)", "desc": "margarine: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with soft smooth pale yellow margarine with a curled scoop mark and a swirled peak, glossy and creamy, lighter yellow than butter, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7098, "name": "Kaya (ingredient)", "desc": "kaya: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick smooth kaya coconut jam, yellowish green in colour, a warm olive-yellow green like pandan custard and not bright green, glossy with a soft swirl on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7099, "name": "Peanut butter (ingredient)", "desc": "peanut butter: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick smooth light brown peanut butter with a swirled peak and a glossy sheen, two whole peanuts beside the bowl, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7100, "name": "Jam (ingredient)", "desc": "strawberry jam: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with glossy translucent red strawberry jam with small soft strawberry pieces and tiny seeds, shiny highlights, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7101, "name": "Chocolate hazelnut spread (ingredient)", "desc": "chocolate hazelnut spread: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick smooth dark brown chocolate hazelnut spread with a glossy swirled peak, one whole hazelnut beside the bowl, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7102, "name": "Half-boiled egg (ingredient)", "desc": "half-boiled eggs: two soft half-boiled eggs cracked into a small shallow saucer, the egg white barely set, watery, wobbly and half see-through, two runny bright orange-yellow yolks floating in it, a dash of dark soy sauce and a sprinkle of white pepper on top, very runny and liquid, kopitiam style, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7103, "name": "Fried egg (ingredient)", "desc": "fried egg: one sunny-side-up fried egg, a round bright vivid yellow yolk in the centre with a white shine, pure white egg white with a crisp lacy golden-brown edge, bright warm colours, the yolk strongly yellow, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7104, "name": "Scrambled eggs (ingredient)", "desc": "a small fluffy mound of soft pale yellow scrambled eggs with glossy curds, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7105, "name": "Chicken slice (ingredient)", "desc": "chicken slices: four very thin round slices of cooked chicken breast sandwich meat fanned and overlapping, pale creamy beige-white with a very faint pink tint and a thin light golden edge, smooth and thin like deli slices, thinner than ham, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7106, "name": "Tuna, in water (ingredient)", "desc": "tuna in water: an open round silver tin can seen from above at an angle, filled with a solid disc of pale pink-beige tuna chunks that flake apart in layers with visible fibres, a little clear water around it, nothing outside the can, warm pink-beige colours, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7107, "name": "Banana (ingredient)", "desc": "one ripe yellow banana, half peeled, with five round banana slices beside it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7108, "name": "White rice (ingredient)", "desc": "a neat dome of fluffy steamed white rice with separate grains visible, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7109, "name": "Brown rice (ingredient)", "desc": "brown rice: a small bowl heaped with cooked brown rice, every separate small oval grain clearly drawn, light tan-brown grains with darker brown bran tips, loose and fluffy with a few grains fallen beside the bowl, clearly many separate grains, not one smooth ball, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7110, "name": "Chicken rice (oily rice) (ingredient)", "desc": "chicken rice: a rounded mound of oily chicken rice on a small plate, every separate grain clearly drawn, pale golden-yellow glossy grains shining with chicken fat, loose grains at the edge, with a slice of cucumber, clearly many separate rice grains, not one smooth ball, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7111, "name": "Roast potatoes (ingredient)", "desc": "five chunky roast potatoes, golden-brown and crispy outside with a sprig of rosemary, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7112, "name": "Noodles, plain (ingredient)", "desc": "plain noodles: a nest of cooked plain yellow wheat noodles in a small bowl, many separate long thin round strands clearly drawn, curling and looping over each other, a few strands hanging over the rim of the bowl, clearly noodle strands, not one solid lump, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7113, "name": "Bread (ingredient)", "desc": "two slices of plain white bread and one small round bread roll, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7114, "name": "Baked chicken drumstick (ingredient)", "desc": "one baked chicken drumstick with golden-brown roasted skin and the bone end showing, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7115, "name": "Baked chicken thigh (ingredient)", "desc": "baked chicken thigh: one whole bone-in chicken thigh, a wide plump rounded triangular piece that is short and broad and not a drumstick, covered in smooth crisp well-browned golden-brown baked skin with darker roasted patches and herb specks, no breadcrumb coating, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7116, "name": "Steamed chicken, no skin (ingredient)", "desc": "six slices of pale white skinless steamed chicken breast, moist, fanned out with a sprig of coriander, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7117, "name": "Roasted chicken, with skin (ingredient)", "desc": "six slices of roasted chicken with glossy dark golden-brown skin on top and white meat below, fanned out, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7118, "name": "Fried chicken (ingredient)", "desc": "two pieces of deep-fried chicken with a thick craggy golden-brown crunchy coating, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7119, "name": "Curry chicken, with gravy (ingredient)", "desc": "curry chicken: two bone-in chicken pieces, a drumstick and a thigh, and one chunk of potato, sitting in a pool of thick orange-yellow curry gravy with red chilli oil floating on top, the chicken coated in the orange curry, a curry leaf and a piece of star anise on top, in a small bowl, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7120, "name": "Steamed fish (ingredient)", "desc": "one thick slice of steamed white fish with grey skin, topped with ginger strips and spring onion in light soy sauce, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7121, "name": "Fried fish (ingredient)", "desc": "fried fish: two thick slices of fried fish steak cut across the body, each slice a horseshoe shape with a centre bone, crisp golden-brown fried outside with a strip of darker grey-brown crispy skin around the edge and white flaky flesh showing, sliced fish, not a whole fish, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7122, "name": "Boiled egg (ingredient)", "desc": "one hard-boiled egg cut in half showing a firm white and a pale yellow yolk, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7123, "name": "Braised tofu (ingredient)", "desc": "three thick rectangles of tofu with brown fried skin in glossy dark brown soy gravy, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7124, "name": "Fried tempeh (ingredient)", "desc": "fried tempeh: six flat rectangular slices of fried tempeh piled on a small plate, each slice a thin rectangle packed with visible whole soybeans pressed together like a honeycomb, deep golden-brown and crispy with darker brown edges, clearly rectangular slabs with a bumpy bean texture, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7125, "name": "Beef stir-fry (ingredient)", "desc": "a small pile of thin stir-fried beef slices with onion and green capsicum strips in glossy brown sauce, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7126, "name": "Stir-fried greens (ingredient)", "desc": "a small heap of stir-fried green choy sum with garlic bits, glossy with oil, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7127, "name": "Steamed or blanched vegetables (ingredient)", "desc": "a small heap of blanched vegetables: green broccoli florets, orange carrot slices and white cauliflower, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7128, "name": "Mixed vegetables (ingredient)", "desc": "a small heap of mixed stir-fried vegetables: cabbage, carrot slices, baby corn, snow peas and black fungus, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7129, "name": "Long beans (ingredient)", "desc": "a small pile of stir-fried green long beans cut into finger lengths, with bits of garlic, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7130, "name": "Salad leaves (ingredient)", "desc": "a small fluffy pile of mixed salad leaves, light green lettuce, dark green rocket and purple radicchio, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7131, "name": "Curry gravy (ingredient)", "desc": "a small bowl of thick orange-red curry gravy with a sheen of red oil and one curry leaf, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7132, "name": "Soy sauce gravy (ingredient)", "desc": "a small bowl of thin glossy dark brown soy gravy, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7133, "name": "Chilli or tomato sauce (ingredient)", "desc": "a small dish split in two halves, bright red chilli sauce on one side and darker red tomato sauce on the other, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7134, "name": "Extra oil or butter (ingredient)", "desc": "a small spoon of clear golden cooking oil beside a small cube of yellow butter, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7135, "name": "Clear soup (ingredient)", "desc": "clear soup: a small bowl of clear light golden broth, see-through and warm pale yellow, with three slices of orange carrot, a piece of white radish and chopped green spring onion floating, tiny drops of oil shining on the surface, bright and clean, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7136, "name": "Mixed leaves (ingredient)", "desc": "a fluffy pile of mixed green salad leaves with a few purple leaves, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7137, "name": "Baby spinach (ingredient)", "desc": "a small fluffy pile of small round dark green baby spinach leaves, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7138, "name": "Grilled chicken (ingredient)", "desc": "six slices of grilled chicken breast with dark grill lines, fanned out, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7139, "name": "Salmon (ingredient)", "desc": "one thick piece of cooked orange-pink salmon fillet, flaking apart, with a lemon wedge, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7140, "name": "Chickpeas (ingredient)", "desc": "a small heap of round beige chickpeas, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7141, "name": "Cherry tomatoes (ingredient)", "desc": "five small round red cherry tomatoes, two cut in half, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7142, "name": "Carrot, grated (ingredient)", "desc": "a small fluffy heap of finely grated bright orange carrot, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7143, "name": "Sweet potato cubes (ingredient)", "desc": "a small heap of roasted orange sweet potato cubes with browned edges, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7144, "name": "Quinoa, cooked (ingredient)", "desc": "cooked quinoa: a small bowl heaped with cooked quinoa, many tiny round pale golden-beige grains each with a little white curled tail, fluffy and loose, with a few red and black quinoa grains mixed in, warm beige colours, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7145, "name": "Croutons (ingredient)", "desc": "a small heap of golden-brown toasted bread cubes, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7146, "name": "Parmesan (ingredient)", "desc": "a small heap of thin pale yellow shaved parmesan cheese curls, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7147, "name": "Mixed nuts (ingredient)", "desc": "mixed nuts: a small heap of mixed nuts on a plate, brown almonds with ridged skin, pale curved cashews, wrinkled light brown walnut halves and round hazelnuts, warm brown, tan and cream colours, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7148, "name": "Caesar (ingredient)", "desc": "a small dish of creamy pale yellow-white Caesar dressing with black pepper specks, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7149, "name": "Roasted sesame (ingredient)", "desc": "a small dish of creamy light brown roasted sesame dressing with sesame seeds on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7150, "name": "Olive oil and lemon (ingredient)", "desc": "a small dish of golden olive oil with a bright yellow lemon wedge beside it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7151, "name": "Balsamic vinegar (ingredient)", "desc": "a small dish of dark brown-black glossy balsamic vinegar, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7152, "name": "Teh (tea) (ingredient)", "desc": "a clear glass mug of dark amber-brown plain black tea with a tea bag string hanging out, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7153, "name": "Kopi (coffee) (ingredient)", "desc": "a white kopitiam cup with a green flower pattern, filled with black coffee, on a saucer, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7154, "name": "Milo (ingredient)", "desc": "Milo: a heaped spoonful of Milo powder spilling onto a small plate, a mound of rich chocolate-brown malt powder with tiny crunchy lumps and granules, clearly warm chocolate brown all over, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7155, "name": "Matcha (ingredient)", "desc": "a small bowl of bright green frothy matcha tea with a bamboo whisk beside it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7156, "name": "Hot chocolate powder (ingredient)", "desc": "a heaped spoon of dark brown cocoa powder with a little spilled beside it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7157, "name": "Lime juice (ingredient)", "desc": "a small glass of pale cloudy green lime juice with a halved green lime beside it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7158, "name": "Condensed milk (ingredient)", "desc": "a spoon with thick pale creamy-yellow condensed milk slowly dripping from it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7159, "name": "Evaporated milk (ingredient)", "desc": "evaporated milk: one small metal can with a thick dark outline, a light blue label band with a white milk drop on it, the top of the can open with thick creamy ivory evaporated milk pouring from it in a stream into a spoon, the cream-ivory milk clearly visible against the blue can, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7160, "name": "Fresh milk (ingredient)", "desc": "fresh milk: one tall rectangular milk carton with a folded gable roof top, white carton with a wide light blue band and a simple milk splash drawing on the front and a blue screw cap, standing next to a small clear glass of white milk with a thick dark outline, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7161, "name": "Sugar (ingredient)", "desc": "a heaped teaspoon of white sugar crystals with two white sugar cubes beside it, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7162, "name": "Brown sugar syrup (ingredient)", "desc": "a small dish of thick dark brown glossy brown sugar syrup with a drip on the rim, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7163, "name": "Honey (ingredient)", "desc": "a wooden honey dipper dripping thick clear golden honey into a small dish, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7164, "name": "Pearls (ingredient)", "desc": "a small heap of round shiny black tapioca boba pearls in a little brown syrup, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7165, "name": "Grass jelly (ingredient)", "desc": "a small heap of shiny wobbly black grass jelly cubes, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 7166, "name": "Whipped cream (ingredient)", "desc": "one tall swirl of fluffy white whipped cream with soft peaks, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey", "n": 1}, {"id": 5077, "name": "Hainan toast with kaya, and eggs", "desc": "Hainan kaya butter toast set: two thick slices of golden charcoal-toasted bread sandwiched with a thick slab of cold yellow butter and a generous layer of kaya coconut jam in a warm yellowish olive-green colour like pandan custard, more yellow than green and not bright green, the kaya showing at the cut edge, with two half-boiled eggs in a small saucer with dark soy sauce, on a plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5078, "name": "Steamed Hainan bread with kaya", "desc": "steamed Hainan bread with kaya: two thick soft fluffy white steamed bread slices, pillowy and untoasted, sandwiched with a slab of yellow butter and a thick layer of kaya coconut jam in a warm yellowish olive-green colour like pandan custard, more yellow than green and not bright green squeezing out at the edges, on a small plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5079, "name": "Kaya toast set", "desc": "kopitiam kaya toast set: thin crisp brown toasted bread cut into two rectangles, sandwiched with a cold slab of yellow butter and kaya coconut jam in a warm yellowish olive-green colour like pandan custard, more yellow than green and not bright green showing at the edge, with two half-boiled eggs in a saucer with dark soy sauce and white pepper, and a small cup of black kopi, no face on the cup, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5080, "name": "Kaya bun", "desc": "kaya bun: one soft round golden-brown baked bun with a shiny top, torn open in the middle to show a thick filling of kaya coconut jam in a warm yellowish olive-green colour like pandan custard, more yellow than green and not bright green inside the fluffy white crumb, on a small plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5081, "name": "Teh O ais kurang manis", "desc": "one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with clear see-through reddish-brown iced black tea with no milk at all, amber-brown and transparent, four clear ice cubes floating at the top, one thin round lime slice on the rim and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5082, "name": "Milo kosong", "desc": "one white ceramic mug with a handle, seen from a three-quarter angle, filled with smooth medium chocolate-brown hot malt drink, a thin ring of lighter brown froth around the edge, two soft curls of steam above, a small spoon resting inside the mug, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5083, "name": "Kopi O kosong ais", "desc": "one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with very dark brown almost black iced coffee with no milk and no cream, dark amber-brown and see-through at the edges, four clear ice cubes at the top and a paper straw, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5084, "name": "Fresh coconut water", "desc": "fresh coconut water: one whole young green coconut with the top sliced off flat to show a round opening, a rim of white coconut flesh and clear water inside, a paper straw standing in the opening, the husk bright fresh green, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5085, "name": "Lassi kosong", "desc": "one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with thick creamy white plain yogurt drink with a frothy top of small bubbles, a pale blue shadow inside the glass so the white drink is easy to see, no fruit and no colour added, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5086, "name": "Sparkling water with lime", "desc": "one tall clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with clear sparkling water with many tiny bubbles rising, a very pale blue tint, three clear ice cubes, two thin round green lime slices inside the glass and one on the rim, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5087, "name": "Cincau with less syrup", "desc": "grass jelly dessert: one small clear glass bowl of shiny black grass jelly cut into neat cubes, sitting in a little clear pale brown syrup with two ice cubes, the jelly black with bright glossy highlights, a small spoon inside the bowl, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5088, "name": "Chia pudding with mango", "desc": "chia pudding with mango: one small clear glass cup with a thick dark outline, the lower two-thirds pale cream pudding speckled all over with many tiny black chia seeds, topped with a thick layer of bright orange-yellow mango cubes and one small green mint leaf, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5089, "name": "Agar-agar with fruit, less sugar", "desc": "agar-agar jelly with fruit: four diamond-shaped pieces of clear see-through pale yellow jelly on a small plate, each piece with small cubes of orange peach and red strawberry set inside, glossy and wobbly, all pieces inside the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5090, "name": "Papaya with a squeeze of lime", "desc": "papaya with lime: one long wedge of ripe papaya cut lengthways on a small plate, bright orange flesh with a thin yellow-green skin, a few round black seeds left at one end, one small green lime wedge resting on top of the papaya, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5091, "name": "Guava with asam boi", "desc": "guava with sour plum powder: six crisp guava wedges fanned on a small plate, pale green-white flesh with thin light green skin and small pale seeds in the centre, dusted with fine reddish-brown sour plum powder, a small pile of the powder on the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5092, "name": "Apple slices with peanut butter", "desc": "apple slices with peanut butter: six apple wedges with bright red skin and cream-white flesh fanned in a half circle on a small plate, around one small round white ceramic dipping bowl filled with smooth light brown peanut butter, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5093, "name": "Dark chocolate, 70% or more", "desc": "dark chocolate: three squares of very dark brown chocolate broken from a bar, stacked loosely on a small plate, deep dark brown with a glossy highlight, clean snapped edges and moulded lines on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5094, "name": "Homemade hot cocoa", "desc": "one white ceramic mug with a handle, seen from a three-quarter angle, standing on a small saucer, filled with rich very dark brown hot cocoa, a light dusting of cocoa powder on the surface, two soft curls of steam above, one small square of dark chocolate on the saucer, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5095, "name": "Oat and banana cookies", "desc": "oat and banana cookies: three round rustic cookies overlapping on a small plate, pale golden-brown with a rough lumpy surface of visible whole rolled oats and small dark brown banana flecks, one cookie broken in half to show a soft inside, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5096, "name": "Bubur kacang hijau, less sugar", "desc": "mung bean porridge: one small bowl of thick sweet green bean porridge, soft olive-green split mung beans in a pale creamy coconut milk broth, a white swirl of coconut milk on top and one knotted green pandan leaf at the edge, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5097, "name": "One onde-onde, eaten slowly", "desc": "onde-onde: two small round bright green pandan glutinous rice balls coated all over in fine white grated coconut, on a small square of banana leaf on a small plate, one ball whole and one cut in half to show dark brown melted palm sugar inside, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5098, "name": "Thin kaya on wholemeal toast", "desc": "kaya on wholemeal toast: one slice of brown wholemeal toast with visible bran flecks, cut diagonally into two triangles on a small plate, spread with a thin even layer of smooth yellowish-green kaya, no butter and no egg, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5099, "name": "Frozen banana, blended", "desc": "blended frozen banana: one small bowl holding a tall creamy pale yellow soft-serve style swirl with a soft peak, three round banana slices tucked into the side inside the bowl, a small spoon beside the swirl in the bowl, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5105, "name": "Teh tarik", "desc": "teh tarik: one clear glass mug with a handle and a thick dark outline, seen from a three-quarter angle, filled with opaque warm orange-brown milk tea and topped with a very thick tall layer of cream-white foam bubbles rising above the rim, a little foam running down the side of the mug, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5116, "name": "Fresh orange juice, no sugar", "desc": "fresh orange juice, no sugar: one short wide clear drinking glass with a thick dark outline, seen from a three-quarter angle, filled with bright opaque orange juice with fine pulp and a thin pale foam line at the top, and half a cut orange showing its juicy segments standing right beside the glass, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5124, "name": "Hainanese chicken chop", "desc": "hainanese chicken chop: one fried boneless chicken thigh with a golden-brown crisp coating on a plate, covered in glossy orange-brown tomato and onion gravy with green peas, onion slices and red tomato wedges, a pile of thick-cut fries and a small heap of peas and diced carrot beside it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5125, "name": "Century egg", "desc": "century egg: one century egg cut into four wedges arranged like a flower on a small plate, each wedge with see-through dark amber-brown jelly on the outside and a creamy dark green-black yolk in the centre, a small pile of thin pink pickled ginger slices beside them, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5129, "name": "Roti tisu", "desc": "roti tisu: one very tall thin crisp cone of paper-thin roti standing upright on a plate like a pointed hat, golden-brown and shiny with a sugar glaze, thin drizzled lines of white condensed milk running down it, the crisp edges almost see-through, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5130, "name": "Roti planta", "desc": "roti planta: one square folded roti canai on a plate, flaky golden layers with browned spots, extra glossy and rich with melted yellow margarine pooling on top and a sprinkle of white sugar crystals, a small bowl of orange dhal curry beside it on the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5131, "name": "Roti pisang", "desc": "roti pisang: one square folded roti canai cut into six pieces on a plate, golden flaky pastry, the cut edges showing soft pale yellow banana slices inside, three banana slices on top and a thin drizzle of white condensed milk, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5132, "name": "Roti sardin", "desc": "roti sardin: one rectangular folded roti canai cut into four pieces on a plate, golden flaky pastry, the cut edges showing a filling of red-orange sardine in tomato sauce with onion and egg, a small bowl of orange curry beside it on the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5133, "name": "Thosai telur", "desc": "thosai telur: one large thin golden-brown crisp thosai folded in half on a round metal tray, a cooked egg spread across the inside showing yellow and white patches at the open edge, with three small metal cups of white coconut chutney, orange dhal and red chutney on the tray, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5134, "name": "Braised chicken with potatoes", "desc": "braised chicken with potatoes: a shallow dish of braised chicken pieces on the bone and large golden potato chunks in thick glossy dark soy brown gravy, the chicken with brown glazed skin, slices of ginger and green spring onion lengths on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5135, "name": "Steamed minced pork patty", "desc": "steamed minced pork patty: one round flat steamed minced pork patty filling a shallow round metal dish, pale pink-beige meat with a slightly bumpy surface, sitting in a pool of clear light brown soy juices, topped with chopped green spring onion, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5136, "name": "Steamed minced pork with salted egg", "desc": "steamed minced pork with salted egg: one round flat steamed minced pork patty filling a shallow round metal dish, pale pink-beige meat, with one round deep orange salted egg yolk pressed into the centre and pieces of white salted egg white set into the surface, clear light brown juices around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5137, "name": "Steamed egg custard", "desc": "steamed egg custard: a shallow round dish filled with silky smooth pale yellow steamed egg custard, a perfectly flat glossy surface that looks wobbly, a thin drizzle of brown soy sauce and chopped green spring onion on top, one spoonful scooped out at the edge, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5138, "name": "Three-colour steamed egg", "desc": "three-colour steamed egg: a square slab of steamed egg cut into six neat cubes on a plate, pale yellow egg custard studded with pieces of dark amber-black century egg and bright orange salted egg yolk, three colours showing on every cut side, a little soy sauce on the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5139, "name": "Tomato egg stir-fry", "desc": "tomato egg stir-fry: a plate of soft scrambled egg in large fluffy golden-yellow curds mixed with juicy red tomato wedges, a light glossy orange-red tomato sauce around them, chopped green spring onion on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5140, "name": "Sour fish soup (suan cai fish)", "desc": "sour fish soup: a wide bowl of golden-yellow sour broth with many thin boneless white fish slices floating on top, strips of yellow-green pickled mustard greens, a few dried red chillies and green coriander leaves, the fish in neat slices, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5141, "name": "Herbal chicken soup", "desc": "herbal chicken soup: a small brown clay pot of clear dark amber-brown herbal broth with two pieces of chicken on the bone, red wolfberries, two red dates and thin slices of pale herbal root floating on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5142, "name": "Chai boey (mustard greens stew)", "desc": "chai boey: a bowl of mustard greens stew, big soft olive-green mustard leaf pieces in a thin orange-red sour and spicy broth, with chunks of roast pork with brown skin, dried red chillies and brown tamarind slices, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5143, "name": "Vinegar pork trotter", "desc": "vinegar pork trotter: a small brown clay pot of pork trotter pieces in very dark glossy black-brown sweet vinegar sauce, each piece with shiny brown skin, thick slices of old yellow ginger and one brown braised egg among them, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5144, "name": "Soy sauce chicken (economy rice)", "desc": "soy sauce chicken: sliced soy sauce chicken laid in a neat row on an oval plate, each slice with glossy dark amber-brown skin on top and pale cream meat underneath, a little thin brown soy sauce on the plate, cucumber slices at the side and green coriander on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5145, "name": "Ginger and spring onion chicken", "desc": "ginger and spring onion chicken: bite-sized boneless chicken pieces stir-fried in a light glossy brown sauce on a plate, with many thin slices of pale yellow ginger and long cut lengths of bright green spring onion mixed through, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5146, "name": "Black pepper pork", "desc": "black pepper pork: thin slices of pork stir-fried in dark glossy brown black pepper sauce on a plate, covered with coarse cracked black pepper specks, with white onion wedges and pieces of green and red capsicum, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5147, "name": "Braised pork with yam", "desc": "braised pork with yam: alternating thick slices of pork belly and pale purple yam standing in a row on a plate like a fan, the pork with dark brown skin, a layer of pale fat and brown meat, the yam lilac with fine purple specks, glossy brown sauce poured over the top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5148, "name": "Hot plate tofu", "desc": "hot plate tofu: a black oval cast-iron hot plate on a wooden board, a layer of cooked yellow egg on the base, topped with round golden-brown fried egg tofu slices in glossy brown sauce with minced meat, green peas and diced orange carrot, light steam rising, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5149, "name": "Stir-fried long beans", "desc": "stir-fried long beans: a plate of bright green long beans cut into short finger lengths, stir-fried and glossy, with golden minced garlic, small orange dried shrimps and a few red chilli slices, no egg, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5150, "name": "Stir-fried kai lan with garlic", "desc": "stir-fried kai lan with garlic: a plate of stir-fried kai lan cut in lengths, thick pale green stems and broad dark green leaves, glossy from the wok, topped with golden-brown fried garlic bits and a little clear sauce on the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5151, "name": "Stir-fried sweet potato leaves", "desc": "stir-fried sweet potato leaves: a plate of soft wilted sweet potato leaves in a low mound, heart-shaped dark green leaves on thin tender pale green stems, glossy, with red chilli slices and pale garlic slices mixed through, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5152, "name": "Lady's fingers with sambal", "desc": "lady's fingers with sambal: eight whole green lady's fingers lying side by side on an oval plate, bright green with ridged sides and pointed tips, a thick line of dark red sambal chilli paste spooned across the middle, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5153, "name": "Cereal prawns", "desc": "cereal prawns: six large fried prawns in orange-red shells on a plate, half buried under a big fluffy heap of golden toasted cereal flakes, with dark green fried curry leaves and small red chilli slices scattered on the cereal, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5154, "name": "Curry squid", "desc": "curry squid: a shallow dish of white squid rings and curled cross-scored squid pieces in thick orange-red curry gravy with a sheen of red oil on top, a few dark green curry leaves and two green okra pieces, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7201, "name": "Tomato broth (ingredient)", "desc": "tomato broth: one small round white ceramic soup bowl seen from a three-quarter angle, filled with bright orange-red tomato broth with a light sheen, two soft red tomato wedges and a few green spring onion rings floating on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7202, "name": "Mushroom broth (ingredient)", "desc": "mushroom broth: one small round white ceramic soup bowl seen from a three-quarter angle, filled with clear light golden-brown mushroom broth with three brown shiitake mushroom slices and a few red wolfberries floating on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7203, "name": "Chicken or herbal broth (ingredient)", "desc": "chicken or herbal broth: one small round white ceramic soup bowl seen from a three-quarter angle, filled with clear golden-yellow chicken broth with two red dates, a few red wolfberries and one thin slice of pale herbal root floating on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7204, "name": "Pork bone broth (ingredient)", "desc": "pork bone broth: one small round white ceramic soup bowl seen from a three-quarter angle, filled with opaque creamy milky-white pork bone broth with a silky surface, one piece of sweetcorn and green spring onion rings floating on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7205, "name": "Tom yum broth (ingredient)", "desc": "tom yum broth: one small round white ceramic soup bowl seen from a three-quarter angle, filled with clear orange-red tom yum broth with a ring of red chilli oil, one stalk of lemongrass, two lime leaves and a red chilli floating on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7206, "name": "Mala (spicy) broth (ingredient)", "desc": "mala: one small round white ceramic soup bowl seen from a three-quarter angle, filled with deep red mala broth covered with a glossy layer of red chilli oil, many dried red chillies and small brown Sichuan peppercorns floating on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7207, "name": "Beef slices (ingredient)", "desc": "beef slices: eight paper-thin raw beef slices rolled into neat curls and lined up in two rows, bright red meat with thin white fat marbling, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7208, "name": "Lamb slices (ingredient)", "desc": "lamb slices: eight paper-thin raw lamb slices rolled into tight curls in two rows, pinkish-red meat with wide white fat stripes, clearly more white fat than beef, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7209, "name": "Pork slices (ingredient)", "desc": "pork slices: eight thin raw pork slices rolled into curls in two rows, pale rosy-pink meat with a thin white fat edge, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7210, "name": "Chicken slices (ingredient)", "desc": "chicken slices: eight thin raw chicken breast slices fanned in a row, very pale pink-cream meat, smooth and lean with no fat, a small parsley sprig beside them, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7211, "name": "Pork belly slices (ingredient)", "desc": "pork belly slices: eight long thin raw pork belly slices laid flat and overlapping, each with clear stripes of white fat and pink meat like bacon, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7212, "name": "Wagyu slices (ingredient)", "desc": "wagyu slices: six thin raw wagyu beef slices laid flat and overlapping, rosy red meat covered in a fine dense web of white fat marbling, very pale and lacy, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7213, "name": "Beef tripe (ingredient)", "desc": "beef tripe: a neat pile of raw beef tripe strips, cream-white with a dark honeycomb and frilly leaf texture on one side, cut into finger-length strips, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7214, "name": "Luncheon meat (ingredient)", "desc": "luncheon meat: five thick rectangular slices of pink luncheon meat fanned in a row, smooth even salmon-pink with rounded corners, cut from a block, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7215, "name": "Sausage or Taiwan sausage (ingredient)", "desc": "sausage or taiwan sausage: six thick diagonal slices of Taiwan sausage and one whole short plump sausage, glossy reddish-pink with small white fat specks, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7216, "name": "Prawns (ingredient)", "desc": "prawns: five whole raw prawns lined up in a row, translucent blue-green and pale shells with dark tail tips, heads and long feelers on, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7217, "name": "Fish slices (ingredient)", "desc": "fish slices: eight thin raw boneless white fish slices fanned in a row, translucent pearly white with a faint pink line down the middle of each slice, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7218, "name": "Fish balls (ingredient)", "desc": "fish balls: seven smooth round white fish balls piled in a small mound, bouncy and glossy with soft cream shading, all the same size, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7219, "name": "Fish tofu (ingredient)", "desc": "fish tofu: six golden-brown fish tofu cubes with rounded corners piled loosely, one cut open to show a smooth white inside, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7220, "name": "Beef balls (ingredient)", "desc": "beef balls: seven round beef balls piled in a small mound, warm brown with a slightly rough bouncy surface, one cut in half showing a brown inside, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7221, "name": "Squid (ingredient)", "desc": "squid: one raw white squid tube cut into six rings laid in a row, with a small bunch of curly purple-tipped tentacles beside them, glossy and smooth, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7222, "name": "Scallops (ingredient)", "desc": "scallops: five thick round raw scallops, creamy ivory-white discs with a soft sheen, each with a small crescent of orange roe attached, set in a circle, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7223, "name": "Clams or mussels (ingredient)", "desc": "clams or mussels: six clams with pale brown ridged shells and three black mussels, each shell half open to show the plump orange and cream meat inside, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7224, "name": "Cheese tofu (ingredient)", "desc": "cheese tofu: six pale yellow cheese tofu cubes piled loosely, one cut in half with bright yellow melted cheese flowing out of the centre, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7225, "name": "Crab sticks (ingredient)", "desc": "crab sticks: five crab sticks stacked like logs, each a white cylinder with a bright orange-red stripe along one side, one pulled apart into thin white strands, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7226, "name": "Shrimp paste (ingredient)", "desc": "shrimp paste: one split length of green bamboo filled with smooth pale pink minced shrimp paste, a small flat scraper resting on it, and two shaped pink paste balls beside it, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7227, "name": "Dumplings (ingredient)", "desc": "dumplings: six raw crescent dumplings with neatly pleated edges in a row, smooth pale cream wrappers lightly dusted with flour, a faint green filling showing through, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7228, "name": "Tofu (ingredient)", "desc": "tofu: one white tofu block cut into six neat thick cubes set close together, smooth and wobbly with clean cut sides, pure white with pale cream shading, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7229, "name": "Fried tofu puffs (ingredient)", "desc": "fried tofu puffs: seven golden-brown fried tofu puffs piled in a mound, light airy cubes with a wrinkled skin, one torn open showing a spongy cream inside, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7230, "name": "Beancurd skin (ingredient)", "desc": "beancurd skin: a stack of dried beancurd skin sheets and sticks, pale golden-yellow, thin and wrinkled with a soft sheen, two sheets folded loosely on top, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7231, "name": "Fried beancurd skin rolls (ingredient)", "desc": "fried beancurd skin rolls: six crisp fried beancurd skin rolls stacked in a pyramid, golden-yellow tight spirals with many thin flaky layers showing at the ends, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7232, "name": "Egg tofu (ingredient)", "desc": "egg tofu: one tube of egg tofu cut into seven thick round slices fanned in a row, smooth bright pale yellow discs with a soft custard shine, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7233, "name": "Quail eggs (ingredient)", "desc": "quail eggs: eight small peeled boiled quail eggs piled together, smooth glossy white ovals, with two unpeeled ones showing cream shells with dark brown speckles, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7234, "name": "Napa cabbage (ingredient)", "desc": "napa cabbage: five large napa cabbage leaves stacked and fanned, wide crisp white ribs with frilly crinkled pale yellow-green edges, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7235, "name": "Enoki mushrooms (ingredient)", "desc": "enoki mushrooms: one bundle of enoki mushrooms, many long thin cream-white stems with tiny round caps, joined together at the pale brown base, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7236, "name": "Shiitake mushrooms (ingredient)", "desc": "shiitake mushrooms: five fresh shiitake mushrooms, round brown caps each with a star-shaped cross cut showing white flesh, short cream stems, one turned over to show pale gills, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7237, "name": "Corn (ingredient)", "desc": "corn: four thick rounds of sweetcorn on the cob standing and lying together, bright yellow plump kernels in neat rows around a pale core, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7238, "name": "Lotus root (ingredient)", "desc": "lotus root: six round slices of lotus root fanned in a row, pale cream discs each with a ring of round holes like a flower pattern, thin light brown skin on the edge, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7239, "name": "Bok choy (ingredient)", "desc": "bok choy: four small whole bok choy, each with thick spoon-shaped pale green-white stems and smooth dark green leaves, lying side by side, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7240, "name": "Tang oh (garland chrysanthemum) (ingredient)", "desc": "tang oh: a bunch of tang oh greens, slender pale green stems with soft feathery deeply-cut bright green leaves like chrysanthemum leaves, lying loosely, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7241, "name": "Yam (taro) (ingredient)", "desc": "yam: five thick slices of raw yam stacked and fanned, pale lilac-white flesh with fine purple specks and a thin brown skin edge, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7242, "name": "King oyster mushroom (ingredient)", "desc": "king oyster mushroom: two whole king oyster mushrooms with thick fat cream-white stems and small tan caps, and four thick round stem slices beside them, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7243, "name": "Black fungus (ingredient)", "desc": "black fungus: a small pile of black fungus, glossy dark brown-black wavy ear-shaped pieces with a lighter brown underside, thin and frilly with shiny highlights, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7244, "name": "Glass noodles (ingredient)", "desc": "glass noodles: one neat bundle of soaked glass noodles tied in a loose knot, clear see-through thin threads with a pale blue-white shine, every strand drawn, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7245, "name": "Udon (ingredient)", "desc": "udon: one neat coil of thick white udon noodles, each fat round strand clearly drawn, smooth and glossy with soft cream shading, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7246, "name": "Instant noodles (ingredient)", "desc": "instant noodles: one square block of dry instant noodles, pale yellow tightly wavy noodle strands pressed into a brick, with a few broken curly bits on the plate beside it, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7247, "name": "Fried dough stick (ingredient)", "desc": "fried dough stick: one golden-brown fried dough stick cut into six short chunks, each a puffy double stick with a crisp blistered surface and an airy holey inside, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7248, "name": "Sesame paste (ingredient)", "desc": "sesame paste: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick smooth light tan-brown sesame paste with a glossy swirl and a few white sesame seeds on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7249, "name": "Sha cha sauce (ingredient)", "desc": "sha cha sauce: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with grainy dark brown sha cha sauce with a layer of red-brown oil and visible tiny bits of dried shrimp and garlic, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7250, "name": "Garlic chilli oil (ingredient)", "desc": "garlic chilli oil: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with clear bright red chilli oil full of red chilli flakes and pale golden minced garlic bits, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5201, "name": "Old cucumber soup with pork ribs", "desc": "old cucumber soup with pork ribs: a bowl of clear light golden-brown soup with three thick chunks of old cucumber, each with rough brown netted skin and soft pale cream flesh, two pork rib pieces, two red dates and a few red wolfberries, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5202, "name": "Winter melon soup with dried scallops", "desc": "winter melon soup with dried scallops: a bowl of clear pale golden soup with large see-through pale jade-green winter melon cubes, four small round golden dried scallops, a few red wolfberries and thin ginger slices, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5203, "name": "Corn and carrot soup with chicken", "desc": "corn and carrot soup with chicken: a bowl of clear light golden soup with three bright yellow corn cob rounds, big orange carrot chunks and two pieces of chicken on the bone, two red dates floating, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5204, "name": "White radish soup with pork ribs", "desc": "white radish soup with pork ribs: a bowl of clear pale soup with big half-moon chunks of see-through white radish, three pork rib pieces, green coriander leaves on top and a few white peppercorns, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5205, "name": "Bitter gourd soup with lean pork", "desc": "bitter gourd soup with lean pork: a bowl of clear light soup with many bright green bitter gourd half-rings with bumpy ridged edges, thin slices of pale lean pork and a few yellow soybeans, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5206, "name": "Tomato and egg soup", "desc": "tomato and egg soup: a bowl of light orange-red clear soup with soft red tomato wedges and wispy ribbons of yellow and white egg floating through it, chopped green spring onion on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5208, "name": "Steamed minced beef patty", "desc": "steamed minced beef patty: one round flat steamed minced beef patty filling a shallow round dish, darker warm brown than pork with a coarse texture, sitting in clear brown juices, topped with bright green coriander leaves, thin strips of orange dried tangerine peel and small white water chestnut bits, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5209, "name": "Pan-fried beef pancake", "desc": "pan-fried beef pancake: two round flat pan-fried stuffed beef pancakes on a plate, golden-brown crisp blistered pastry discs, one cut in half to show a juicy brown minced beef and green spring onion filling between thin pastry layers, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5222, "name": "Dorayaki", "desc": "dorayaki: two round golden-brown pancake sandwiches on a small plate, smooth even brown tops, one cut in half to show a thick layer of dark red-brown red bean paste between two soft sponge pancakes, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5236, "name": "Korean fried chicken, yangnyeom", "desc": "korean fried chicken, yangnyeom: six pieces of fried chicken wings and drumettes coated in thick sticky glossy red-orange sweet chilli glaze, sprinkled with white sesame seeds and crushed peanuts, piled on a small plate with a few white pickled radish cubes, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5240, "name": "Kimchi fried rice with egg", "desc": "kimchi fried rice with egg: a dome of orange-red fried rice flecked with red kimchi pieces on a plate, topped with one sunny-side-up fried egg with a bright yolk, thin strips of dark seaweed and white sesame seeds, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5243, "name": "Sticky rice", "desc": "sticky rice: a small round woven bamboo basket with its woven lid leaning against it, filled with a mound of shiny white glutinous rice, the grains clumped together and slightly see-through, plain rice with nothing on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5244, "name": "Satay daging", "desc": "satay daging: three wooden skewers of beef satay lying side by side on a small plate, small dark brown grilled beef pieces with char marks and a glossy sheen, clearly darker than chicken satay, with cucumber chunks and red onion pieces on the plate, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5245, "name": "Peanut sauce for satay", "desc": "peanut sauce for satay: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick chunky orange-brown peanut satay sauce with many visible crushed peanut bits and a spoonful of dark red chilli oil floating on top, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5246, "name": "Nasi impit", "desc": "nasi impit: eight neat white cubes of compressed rice piled on a square of green banana leaf on a small plate, each cube smooth and solid with clean cut sides, pure white with pale cream shading, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5247, "name": "Grilled chicken chop with salad", "desc": "grilled chicken chop with salad: one flat boneless grilled chicken thigh with golden-brown skin and dark criss-cross grill marks on a plate, with a fresh salad of green lettuce, red cherry tomato halves and cucumber slices beside it, no fries and no gravy, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 5251, "name": "Steak with vegetables", "desc": "steak with vegetables: one grilled sirloin steak with dark criss-cross grill marks on a plate, cut into slices at one end to show a warm pink centre, a round pat of herb butter on top, with green broccoli, orange carrot sticks and three roasted potato halves, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7301, "name": "Pork belly (samgyeopsal) (ingredient)", "desc": "pork belly: six thick strips of grilled pork belly cut into bite-sized pieces, golden-brown crisp edges with clear layers of white fat and pale meat, glistening, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7302, "name": "Pork neck (moksal) (ingredient)", "desc": "pork neck: four thick slices of grilled pork neck, pale pink-beige meat with golden-brown sear marks and fine white fat marbling, leaner than pork belly, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7303, "name": "Marinated pork ribs (dwaeji galbi) (ingredient)", "desc": "marinated pork ribs: a long butterflied strip of marinated pork rib meat attached to a short bone, glossy dark red-brown glaze with charred edges, cut into pieces, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7304, "name": "Beef short rib (galbi) (ingredient)", "desc": "beef short rib: four pieces of beef short rib, each a strip of dark glossy brown marinated meat with diamond score marks attached to a flat white bone, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7305, "name": "Beef bulgogi (ingredient)", "desc": "beef bulgogi: a small heap of thin beef bulgogi slices, glossy caramel-brown with onion slivers, green spring onion and white sesame seeds on top, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7306, "name": "Beef brisket slices (chadolbagi) (ingredient)", "desc": "beef brisket slices: eight very thin curled slices of beef brisket, lightly browned with a wide white fat edge on each, small and delicate, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7307, "name": "Marinated chicken (dak galbi) (ingredient)", "desc": "marinated chicken: six chunks of boneless chicken coated in a sticky bright red-orange chilli marinade with charred spots and white sesame seeds, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7308, "name": "Chicken wing (ingredient)", "desc": "chicken wing: three whole chicken wings with deep golden-brown crisp skin and dark char spots, glossy, lying side by side, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7309, "name": "Chicken thigh, boneless (ingredient)", "desc": "chicken thigh, boneless: one flat boneless chicken thigh with golden-brown skin and dark criss-cross grill lines, cut into five strips showing juicy pale meat, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7310, "name": "Beef steak slices (ingredient)", "desc": "beef steak slices: six thick slices of beef steak fanned in a row, dark brown seared edges and a warm pink centre, a pinch of coarse salt on top, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7311, "name": "Ox tongue slices (ingredient)", "desc": "ox tongue slices: seven thin round slices of ox tongue fanned in a row, pale pink-beige with browned edges and a fine grain, a lemon wedge and chopped spring onion on top, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7312, "name": "Lamb chop (ingredient)", "desc": "lamb chop: two lamb chops with long clean curved rib bones, browned seared meat with dark grill lines and a thin rim of golden fat, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7313, "name": "Sausage (ingredient)", "desc": "sausage: three plump sausages with glossy browned skin and dark diagonal score cuts that have opened up, lying side by side, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7314, "name": "Grilled saba (mackerel) (ingredient)", "desc": "grilled saba: one mackerel fillet with shiny blue-silver striped skin blistered golden-brown, cut into four boneless slices, a lemon wedge beside it, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7315, "name": "Grilled salmon (ingredient)", "desc": "grilled salmon: one thick salmon fillet cut into four boneless slices, opaque pink-orange flaky flesh with dark grill lines and a crisp skin edge, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7316, "name": "Grilled prawns (ingredient)", "desc": "grilled prawns: four whole prawns on two wooden skewers, bright orange-red shells with dark char marks, tails and heads on, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7317, "name": "Grilled squid (ingredient)", "desc": "grilled squid: one whole squid body with diamond score cuts, cut into rings but kept in shape, white flesh with golden-brown grill marks and curled tentacles at the end, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7318, "name": "Lettuce wrap leaf (ingredient)", "desc": "lettuce wrap leaf: five fresh lettuce leaves stacked like cups, bright light green with frilly curly edges and pale crisp ribs, a few water drops, on a small white plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7319, "name": "Perilla leaf (ingredient)", "desc": "perilla leaf: six perilla leaves fanned like a hand of cards, broad heart-shaped deep green leaves with saw-tooth edges and clear veins, on a small white plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7320, "name": "Grilled garlic (ingredient)", "desc": "grilled garlic: one small round foil cup holding eight peeled garlic cloves in golden sesame oil, the cloves soft and golden-brown, on a small white plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7321, "name": "Corn on the cob (ingredient)", "desc": "corn on the cob: one whole corn on the cob on a wooden stick, bright yellow kernels with dark charred patches and a glossy butter shine, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7322, "name": "Onion slices (ingredient)", "desc": "onion slices: four thick round onion slices showing white rings, edges browned and charred with dark grill lines, slightly see-through, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7323, "name": "Eggplant (ingredient)", "desc": "eggplant: five long diagonal slices of eggplant with deep purple skin edges and soft cream flesh marked with dark grill lines, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7324, "name": "Sweet potato (ingredient)", "desc": "sweet potato: four thick round slices of sweet potato with reddish-purple skin and bright golden-orange flesh marked with dark grill lines, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7325, "name": "Enoki in butter foil (ingredient)", "desc": "enoki in butter foil: one open square foil packet holding a bundle of soft cooked enoki mushrooms in melted golden butter, a square of butter melting on top, on a small white plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7326, "name": "Pineapple (ingredient)", "desc": "pineapple: three thick round pineapple rings with a hole in the centre, bright golden-yellow with dark caramelised grill lines, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7327, "name": "Green chilli (ingredient)", "desc": "green chilli: four long whole green chillies with stems, glossy bright green skin with blistered brown charred patches, grilled, on a small white plate, everything inside the plate, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7328, "name": "Seasoned bean sprouts (ingredient)", "desc": "seasoned bean sprouts: a small white side dish piled with pale cream bean sprouts with yellow heads, glossy with sesame oil, sprinkled with red chilli flakes, green spring onion and sesame seeds, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7329, "name": "Pickled onion or radish (ingredient)", "desc": "pickled onion or radish: a small white side dish of see-through sliced onion in clear brown soy pickle and a pile of thin round white pickled radish slices, glossy and wet, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7330, "name": "Steamed egg (gyeran jjim) (ingredient)", "desc": "steamed egg: a small black stone pot with a tall fluffy pale yellow steamed egg puffed high above the rim like a souffle, green spring onion on top, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7331, "name": "Corn cheese (ingredient)", "desc": "corn cheese: a small black round cast-iron pan of bright yellow sweetcorn kernels covered with melted golden-white cheese with browned bubbly spots and a stretchy cheese pull, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7332, "name": "Fish cake strips (ingredient)", "desc": "fish cake strips: a small white side dish of thin flat tan-brown fish cake strips stir-fried glossy with soy, with a few green chilli slices and white sesame seeds, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7333, "name": "Soybean paste stew (doenjang jjigae) (ingredient)", "desc": "soybean paste stew: a small black stone pot of cloudy golden-brown soybean paste stew with white tofu cubes, green zucchini half-moons, a mushroom and a green chilli slice, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7334, "name": "Cold noodles (naengmyeon) (ingredient)", "desc": "cold noodles: a metal bowl of thin brown buckwheat noodles coiled in clear icy broth with ice shards, topped with half a boiled egg, cucumber strips and a slice of pale pear, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7335, "name": "Ssamjang (ingredient)", "desc": "ssamjang: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick coarse reddish-brown ssamjang paste with sesame seeds and tiny green onion bits, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7336, "name": "Sesame oil with salt (ingredient)", "desc": "sesame oil with salt: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with clear golden sesame oil with a small mound of white salt and black pepper specks in the middle, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7337, "name": "Gochujang (ingredient)", "desc": "gochujang: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick smooth glossy deep red chilli paste in a swirled peak, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7338, "name": "Yakiniku sauce (ingredient)", "desc": "yakiniku sauce: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thin dark brown soy-based sauce with white sesame seeds and a little grated pale onion floating, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7339, "name": "Garlic butter (ingredient)", "desc": "garlic butter: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with melted golden-yellow butter with pale minced garlic and green parsley flecks, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7340, "name": "Sambal (ingredient)", "desc": "sambal: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with thick coarse dark red sambal with visible chilli seeds and a sheen of red oil, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}, {"id": 7341, "name": "Sweet soy onion sauce (ingredient)", "desc": "sweet soy onion sauce: one small round white ceramic dipping sauce bowl with tall sides, seen from a three-quarter angle, filled to the brim with glossy dark brown sweet soy sauce with soft see-through onion slivers, one single ingredient portion on its own, nothing else around it, bright warm light, clean bright colours with no grey patches, plain food with no face and no eyes", "neg": "face, eyes, mouth, grey, dull, extra pieces outside the plate", "n": 1}]
STYLES = {
 "Sticker": "a cute hand-drawn food sticker illustration of {d}. Chubby rounded soft shapes, pastel colours with soft pink accents, smooth clean outlines, simple soft shading, tiny white shine highlights, like a sticker from a Japanese stationery set, just the food on its dish, three-quarter view, centered, plain white background"
}

JOB = [p for p in PROMPTS if p["id"] in set(ONLY)]


In [ ]:
import json, os, subprocess, sys
os.makedirs("/kaggle/working", exist_ok=True)
job = JOB
json.dump({"style": STYLES[STYLE_KEY], "size": SIZE, "steps": STEPS, "items": job}, open("/kaggle/working/job.json", "w"))
open("/kaggle/working/worker.py", "w").write(r"""
import sys, json, time, torch, numpy as np
from PIL import Image
gpu, n = int(sys.argv[1]), int(sys.argv[2])
import os
J = json.load(open("/kaggle/working/job.json")); items = [p for p in J["items"][gpu::n] if not os.path.exists(f"/kaggle/working/c{p['id']}.webp")]
from diffusers import ZImagePipeline, ZImageTransformer2DModel, BitsAndBytesConfig as DB
from transformers import BitsAndBytesConfig as TB, AutoModel
from rembg import remove, new_session
R = "Tongyi-MAI/Z-Image-Turbo"; dev = f"cuda:{gpu}"
MODE = ""
def load(tdt, vae32, small_te=True):
    # tdt: number format for the painting part. vae32: run the final picture decoder in full precision.
    tr = ZImageTransformer2DModel.from_pretrained(R, subfolder="transformer", torch_dtype=tdt,
        quantization_config=DB(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=tdt), device_map={"": dev})
    kw = {}
    if small_te:
        try:
            kw["text_encoder"] = AutoModel.from_pretrained(R, subfolder="text_encoder", torch_dtype=tdt,
                quantization_config=TB(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=tdt), device_map={"": dev})
        except Exception as e: print("4-bit text reader not available:", str(e)[:200], flush=True)
    p = ZImagePipeline.from_pretrained(R, transformer=tr, torch_dtype=tdt, **kw)
    if "text_encoder" in kw: p.vae.to(dev)
    else: p.enable_model_cpu_offload(gpu_id=gpu)
    if vae32: p.vae.to(dtype=torch.float32)
    return p
def gen(prompt, seed):
    g = torch.Generator("cpu").manual_seed(seed)
    return pipe(prompt=prompt, height=J["size"], width=J["size"], num_inference_steps=J["steps"], guidance_scale=0.0, generator=g).images[0]
def good(im):
    a = np.asarray(im).astype("float32"); return a.mean() > 8 and a.std() > 6      # a blank or single-colour picture means the numbers overflowed
LONG = J["style"].format(d=max((p["desc"] for p in J["items"]), key=len))   # the longest real prompt, so a memory problem shows up now
# Try the fast settings first and keep the first one that draws a real picture. The last one is slow but known to work.
LADDER = [("full precision painting", torch.float32, True, True),      # measured: about 51 s a picture, works
          ("bfloat16 (slow, known to work)", torch.bfloat16, False, True),
          ("bfloat16, swapping parts in and out (slowest)", torch.bfloat16, False, False)]
import gc
pipe = None
for name, tdt, vae32, small in LADDER:
    try:
        t1 = time.time(); pipe = load(tdt, vae32, small); tl = time.time() - t1
        t1 = time.time(); test = gen(LONG, 1); tg = time.time() - t1
        ok = good(test); print(f"gpu{gpu}: tried {name}: loaded in {tl:.0f} s, test picture {tg:.0f} s, {'OK' if ok else 'blank picture'}", flush=True)
        if ok: MODE = name; break
    except Exception as e:
        print(f"gpu{gpu}: tried {name}: failed, {type(e).__name__}: {str(e)[:200]}", flush=True)
    pipe = None; gc.collect(); torch.cuda.empty_cache()
if pipe is None: print(f"gpu{gpu}: no setting worked", flush=True); sys.exit(1)
print(f"gpu{gpu}: using {MODE}; memory in use {torch.cuda.memory_allocated(gpu)/2**30:.1f} GB, peak {torch.cuda.max_memory_allocated(gpu)/2**30:.1f} GB", flush=True)
open(f"/kaggle/working/ready{gpu}", "w").write("1")
CUT = new_session("isnet-general-use"); t0 = time.time(); done = 0
for p in items:
    try:
        im = gen(J["style"].format(d=p["desc"]), 17)
        o = remove(im, session=CUT).convert("RGBA"); bb = o.getbbox()
        if bb: o = o.crop(bb)
        o.thumbnail((256, 256), Image.LANCZOS); o.save(f"/kaggle/working/c{p['id']}.webp", "WEBP", quality=80, method=6); done += 1
    except Exception as e: print("failed", p["id"], str(e)[:160], flush=True); torch.cuda.empty_cache()
    if done and done % 10 == 0: print(f"gpu{gpu}: {done}/{len(items)} done, {(time.time()-t0)/max(done,1):.1f} s each", flush=True)
print(f"gpu{gpu} finished {done} of {len(items)} in {time.time()-t0:.0f} s, {(time.time()-t0)/max(done,1):.1f} s each, {MODE}", flush=True)
""")
import time, torch, glob
n = max(1, torch.cuda.device_count()); print("GPUs found:", n, flush=True)
def start(i, k):
    log = open(f"/kaggle/working/log{i}.txt", "w")
    return subprocess.Popen([sys.executable, "/kaggle/working/worker.py", str(i), str(k)], stdout=log, stderr=subprocess.STDOUT)
# start the GPUs one after another: loading two copies of the model at once runs the machine out of memory
procs = []
for i in range(n):
    pr = start(i, n); procs.append(pr); t = time.time()
    while pr.poll() is None and not os.path.exists(f"/kaggle/working/ready{i}") and time.time() - t < 1500: time.sleep(5)
    print(f"gpu{i}: {'ready' if os.path.exists(f'/kaggle/working/ready{i}') else 'did not start'} after {time.time()-t:.0f} s", flush=True)
for i, pr in enumerate(procs):
    code = pr.wait(); print(f"gpu{i} ended with code {code}", flush=True)
    print("".join([l for l in open(f"/kaggle/working/log{i}.txt", errors="replace").read().replace("\r", "\n").splitlines(True) if "it/s]" not in l and "s/it]" not in l][-25:]), flush=True)
left = [p for p in JOB if not os.path.exists(f"/kaggle/working/c{p['id']}.webp")]
if left:
    print("finishing", len(left), "missed pictures on the first GPU", flush=True)
    pr = start(0, 1); print("retry ended with code", pr.wait(), flush=True)
    print("".join(open("/kaggle/working/log0.txt", errors="replace").read().replace("\r", "\n").splitlines(True)[-15:]), flush=True)
for f in glob.glob("/kaggle/working/ready*") + glob.glob("/kaggle/working/log*.txt"): os.remove(f)
print("all workers finished")


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import glob
try: f = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 13)
except Exception: f = ImageFont.load_default()
made = [(p, f"/kaggle/working/c{p['id']}.webp") for p in JOB if os.path.exists(f"/kaggle/working/c{p['id']}.webp")]
print("pictures made:", len(made), "of", len(JOB))
C = 200; W = 8; PER = 48
for s in range(0, len(made), PER):
    part = made[s:s + PER]; H = (len(part) + W - 1) // W
    sheet = Image.new("RGB", (W * C, H * (C + 34)), "#fff0f4"); dr = ImageDraw.Draw(sheet)
    for k, (p, fn) in enumerate(part):
        im = Image.open(fn).convert("RGBA"); x, y = (k % W) * C, (k // W) * (C + 34); im.thumbnail((C - 16, C - 16))
        sheet.paste(im, (x + (C - im.width) // 2, y + 6 + (C - 16 - im.height) // 2), im)
        dr.text((x + 6, y + C - 8), f"{p['id']} {p['name'][:26]}", fill="#7a2e4d", font=f)
    sheet.save(f"/kaggle/working/contact_dish_{s:04d}.jpg", quality=82)
os.remove("/kaggle/working/job.json"); os.remove("/kaggle/working/worker.py")
print("sheets done")